In [2]:

from pathlib import Path

drive = Path("E:/")
project_name = "BDHS_ANC_Q1_FINAL_REBUILD_2026"
master_name = "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"

if not drive.is_dir():
    raise FileNotFoundError("E: drive is not accessible.")

bases = [drive / "Drive E", drive]
candidates = []

for base in bases:
    if not base.is_dir():
        continue

    candidates.extend([base, base / project_name])

    for depth in range(4):
        pattern = "*/" * depth + f"01_processed_data/{master_name}"
        for file in base.glob(pattern):
            candidates.append(file.parent.parent)

required = {
    "Primary master":
        "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv",

    "Outer CV splits":
        "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv",

    "Inner CV splits":
        "04_cv_splits/STEP13_FROZEN_inner_4fold_row_assignments_long.csv",

    "Candidate registry":
        "03_protocol/STEP16_FINAL_hyperparameter_candidate_registry.csv",

    "Selected models":
        "05_models/STEP19C_PRIMARY_joint_winners_by_outer_stage.csv",

    "Calibration":
        "07_calibration/STEP19C_PRIMARY_calibration_thresholds_by_outer_stage.csv",

    "Outer OOF predictions":
        "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv",
}

def project_score(path):
    return sum((path / relative).is_file() for relative in required.values())

candidates = list(dict.fromkeys(p for p in candidates if p.is_dir()))

if not candidates:
    raise FileNotFoundError("No accessible project directory found on E: drive.")

root = max(candidates, key=project_score)

if project_score(root) == 0:
    raise FileNotFoundError(
        "ANC project not identified. Check the folder location on E: drive."
    )

print("PROJECT ROOT:", root)
print("=" * 70)

found = 0

for name, relative in required.items():
    path = root / relative
    exists = path.is_file()
    found += int(exists)
    print(f"{name:25} {'FOUND' if exists else 'MISSING'}")

print("=" * 70)
print(f"REQUIRED FILES: {found}/{len(required)}")

extensions = {".joblib", ".pkl", ".pickle", ".sav"}

model_files = sorted(
    p for p in root.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)

print("\nSERIALIZED MODEL FILES:", len(model_files))

for path in model_files[:40]:
    print(" ", path.relative_to(root))

xai_dir = root / "10_explainability"

xai_files = (
    sorted(p for p in xai_dir.rglob("*") if p.is_file())
    if xai_dir.is_dir() else []
)

print("\nEXISTING XAI FILES:", len(xai_files))

for path in xai_files[:40]:
    print(" ", path.relative_to(root))

print("\nPROJECT SUBDIRECTORIES:")

for path in sorted(root.iterdir()):
    if path.is_dir():
        n = sum(1 for p in path.iterdir() if p.is_file())
        print(f"{path.name:35} {n} direct files")

print("\nAUDIT COMPLETE")

PROJECT ROOT: E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026
Primary master            FOUND
Outer CV splits           FOUND
Inner CV splits           FOUND
Candidate registry        FOUND
Selected models           FOUND
Calibration               FOUND
Outer OOF predictions     FOUND
REQUIRED FILES: 7/7

SERIALIZED MODEL FILES: 0

EXISTING XAI FILES: 0

PROJECT SUBDIRECTORIES:
00_raw_reference                    7 direct files
01_processed_data                   22 direct files
02_quality_control                  201 direct files
03_protocol                         127 direct files
04_cv_splits                        4 direct files
05_models                           41 direct files
06_predictions                      22 direct files
07_calibration                      23 direct files
08_bootstrap                        13 direct files
09_geographic_validation            451 direct files
10_explainability                   0 direct files
11_sensitivity                      284 direct files


In [4]:

from pathlib import Path
import pandas as pd

root = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
folder = root / "10_explainability" / "XAI_SELECTED_REFITS"

manifest = folder / "reconstruction_manifest.json"
qc_file = folder / "reconstruction_qc.csv"
models = sorted(folder.glob("outer_*_stage_*.joblib"))

print("Reconstruction manifest:", manifest.is_file())
print("Reconstruction QC:", qc_file.is_file())
print("Saved selected models:", len(models), "/ 10")

if manifest.is_file() and qc_file.is_file() and len(models) == 10:
    qc = pd.read_csv(qc_file)
    print("\nReconstruction QC:")
    print(qc.to_string(index=False))
    print("\nREADY FOR XAI STEP 2")
else:
    print("\nRECONSTRUCTION INCOMPLETE")
    print("Run XAI Step 1 before permutation importance.")

Reconstruction manifest: False
Reconstruction QC: False
Saved selected models: 0 / 10

RECONSTRUCTION INCOMPLETE
Run XAI Step 1 before permutation importance.


In [6]:

from pathlib import Path
import os
import platform

drive = Path("E:/")
root = drive / "Drive E" / "BDHS_ANC_Q1_FINAL_REBUILD_2026"
data_dir = root / "01_processed_data"
filename = "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
master = data_dir / filename

print("SYSTEM:", platform.system())
print("WORKING DIRECTORY:", Path.cwd())
print("=" * 70)

checks = {
    "E: drive": drive,
    "Drive E folder": drive / "Drive E",
    "ANC project": root,
    "Processed data folder": data_dir,
    "Primary master CSV": master,
}

for name, path in checks.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:28} {status}")
    print(" ", path)

print("\n" + "=" * 70)
print("PROJECT DIRECTORY CHECK")

for base in [drive, drive / "Drive E"]:
    if base.is_dir():
        print(f"\nDirectories under {base}:")
        dirs = sorted(
            [p.name for p in base.iterdir() if p.is_dir()]
        )
        for name in dirs[:40]:
            print(" ", name)

print("\n" + "=" * 70)
print("MASTER DATASET SEARCH")

matches = []

if root.is_dir():
    matches.extend(
        p for p in root.rglob("*.csv")
        if "master" in p.name.lower()
    )

if not matches:
    for base in [drive / "Drive E", drive]:
        if not base.is_dir():
            continue

        for depth in range(1, 5):
            pattern = (
                "*/" * depth
                + "01_processed_data/*master*.csv"
            )
            matches.extend(base.glob(pattern))

matches = sorted(set(matches))

if matches:
    for path in matches[:30]:
        print("FOUND:", path)
        print("SIZE:", round(path.stat().st_size / 1048576, 2), "MB")
else:
    print("No matching master CSV found within searched locations.")

print("\n" + "=" * 70)

if master.is_file():
    print("STATUS: ORIGINAL PROJECT PATH VERIFIED")
elif matches:
    print("STATUS: POSSIBLE ALTERNATIVE MASTER LOCATION FOUND")
else:
    print("STATUS: PROJECT PATH OR MASTER FILE NEEDS ATTENTION")

print("DIAGNOSTIC COMPLETE")

SYSTEM: Windows
WORKING DIRECTORY: C:\Users\M R Computer
E: drive                     MISSING
  E:\
Drive E folder               MISSING
  E:\Drive E
ANC project                  MISSING
  E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026
Processed data folder        MISSING
  E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data
Primary master CSV           MISSING
  E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv

PROJECT DIRECTORY CHECK

MASTER DATASET SEARCH
No matching master CSV found within searched locations.

STATUS: PROJECT PATH OR MASTER FILE NEEDS ATTENTION
DIAGNOSTIC COMPLETE


In [7]:

from pathlib import Path
import string

PROJECT = "BDHS_ANC_Q1_FINAL_REBUILD_2026"

drives = [
    Path(f"{letter}:/")
    for letter in string.ascii_uppercase
    if Path(f"{letter}:/").is_dir()
]

print("AVAILABLE DRIVES:")
for drive in drives:
    print(" ", drive)

locations = []

for drive in drives:
    locations.extend([
        drive / PROJECT,
        drive / "Drive E" / PROJECT,
    ])

home = Path.home()

for base in [
    home,
    home / "Desktop",
    home / "Documents",
    home / "Downloads",
]:
    locations.extend([
        base / PROJECT,
        base / "Drive E" / PROJECT,
    ])

required = [
    "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv",
    "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv",
    "05_models/STEP19C_PRIMARY_joint_winners_by_outer_stage.csv",
    "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv",
]

matches = []

for path in dict.fromkeys(locations):
    if path.is_dir():
        count = sum(
            (path / relative).is_file()
            for relative in required
        )
        matches.append((count, path))

matches.sort(key=lambda item: item[0], reverse=True)

print("\nPROJECT SEARCH RESULTS:")

for count, path in matches:
    print(f"{count}/{len(required)} files found: {path}")

if matches and matches[0][0] == len(required):
    ROOT = matches[0][1]

    print("\nPROJECT ROOT:", ROOT)
    print("STATUS: READY FOR MODEL RECONSTRUCTION")

else:
    ROOT = None
    print("\nSTATUS: PROJECT NOT FOUND IN CHECKED LOCATIONS")
    print("Check whether the external drive is connected.")

AVAILABLE DRIVES:
  C:\
  D:\

PROJECT SEARCH RESULTS:

STATUS: PROJECT NOT FOUND IN CHECKED LOCATIONS
Check whether the external drive is connected.


In [9]:

from pathlib import Path
import ast
import hashlib
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.special import expit
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

PROJECT = "BDHS_ANC_Q1_FINAL_REBUILD_2026"
PREFERRED_ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
ALTERNATIVE_ROOT = Path(r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026")
ROOT = next((p for p in (PREFERRED_ROOT, ALTERNATIVE_ROOT) if p.is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("ANC project unavailable. Reconnect the E: drive.")

SEED = 2026
TOL = 1e-10
relative = {
    "master": "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv",
    "outer": "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv",
    "inner": "04_cv_splits/STEP13_FROZEN_inner_4fold_row_assignments_long.csv",
    "registry": "03_protocol/STEP16_FINAL_hyperparameter_candidate_registry.csv",
    "winners": "05_models/STEP19C_PRIMARY_joint_winners_by_outer_stage.csv",
    "calibration": "07_calibration/STEP19C_PRIMARY_calibration_thresholds_by_outer_stage.csv",
    "oof": "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv",
}
paths = {key: ROOT / rel for key, rel in relative.items()}
missing = [key for key, p in paths.items() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete project at {ROOT}. Missing: {missing}")

EXPECTED_SHA256 = {
    "master": "0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a",
    "outer": "258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372",
    "inner": "2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633",
    "registry": "f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697",
    "oof": "6c4f4e21b09a54c286521682e9fb8ee13eda36ed94196d350fbb0622733a99bc",
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for key, reference in EXPECTED_SHA256.items():
    actual = sha256(paths[key])
    if actual != reference:
        raise RuntimeError(f"Frozen input SHA256 mismatch: {key}: {actual}")

print("PROJECT ROOT:", ROOT)
print("REQUIRED FILES: 7/7 | FROZEN HASHES: PASS")
print("scikit-learn:", sklearn.__version__)

master = pd.read_csv(paths["master"])
outer = pd.read_csv(paths["outer"])
registry = pd.read_csv(paths["registry"])
winners = pd.read_csv(paths["winners"])
calibration = pd.read_csv(paths["calibration"])
oof = pd.read_csv(paths["oof"])

numeric = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]
categorical = [
    "maternal_education", "wealth_quintile",
    "partner_education_primary_final", "division", "pregnancy_intention",
    "healthcare_decision_primary_final", "residence", "weekly_media_exposure",
    "working_status", "religion_final", "household_head_sex", "distance_barrier",
]

def canonical_key(series):
    v = pd.to_numeric(series, errors="coerce")
    if v.notna().all() and np.allclose(v.to_numpy(float), np.round(v.to_numpy(float))):
        return pd.Series(np.round(v.to_numpy(float)).astype(np.int64), index=series.index)
    return series.astype(str).str.strip()

for frame, col in ((master, "_ROW_ID"), (outer, "_ROW_ID"), (oof, "_ROW_KEY")):
    frame["__KEY"] = canonical_key(frame[col])
    if frame["__KEY"].duplicated().any():
        raise RuntimeError(f"Duplicate row IDs: {col}")

if len(master) != 5063 or len(winners) != 10 or len(calibration) != 10:
    raise RuntimeError("Unexpected analytic cohort or model-selection count")
if len(registry) != 120 or len(numeric + categorical) != 15:
    raise RuntimeError("Frozen candidate/predictor count mismatch")
if set(master["__KEY"]) != set(outer["__KEY"]) or set(master["__KEY"]) != set(oof["__KEY"]):
    raise RuntimeError("Master/outer/OOF row IDs differ")
if not set(numeric + categorical).issubset(master.columns):
    raise RuntimeError("Some frozen predictors are missing")
if winners.duplicated(["Outer_Fold", "Stage"]).any() or calibration.duplicated(["Outer_Fold", "Stage"]).any():
    raise RuntimeError("Duplicate winner or calibration rows")

outer["OUTER_FOLD"] = pd.to_numeric(outer["OUTER_FOLD"], errors="raise").astype(int)
master["__FOLD"] = master["__KEY"].map(outer.set_index("__KEY")["OUTER_FOLD"])
if master["__FOLD"].isna().any():
    raise RuntimeError("Missing outer-fold assignments")
master["__FOLD"] = master["__FOLD"].astype(int)
if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError("Unexpected outer-fold IDs")
saved = oof.set_index("__KEY", verify_integrity=True)

EXPECTED_WINNERS = {
    0: ("EN_005", "GB_002"),
    1: ("EN_004", "GB_003"),
    2: ("EN_009", "ADA_008"),
    3: ("EN_010", "ADA_003"),
    4: ("EN_007", "ADA_007"),
}

def parse_params(value):
    if isinstance(value, dict):
        return dict(value)
    if pd.isna(value) or not str(value).strip():
        return {}
    try:
        result = json.loads(str(value))
    except json.JSONDecodeError:
        result = ast.literal_eval(str(value))
    if not isinstance(result, dict):
        raise RuntimeError("Candidate parameters must be a dictionary")
    return result

def instantiate(candidate):
    name = str(candidate["Model"])
    params = {
        **parse_params(candidate["Fixed_Params_JSON"]),
        **parse_params(candidate["Tuned_Params_JSON"]),
    }
    if name == "ElasticNet logistic regression":
        return LogisticRegression(**params)
    if name == "GradientBoosting":
        return GradientBoostingClassifier(**params)
    if name == "AdaBoost":
        aliases = {"estimatormaxdepth", "baseestimatormaxdepth", "stumpmaxdepth", "decisiontreemaxdepth", "maxdepth"}
        matching = [k for k in params if str(k).lower().replace("_", "").replace("-", "").replace(" ", "") in aliases]
        if len(matching) != 1:
            raise RuntimeError(f"AdaBoost depth parameter ambiguous: {params}")
        depth = int(params.pop(matching[0]))
        if "estimator" in params and not hasattr(params["estimator"], "fit"):
            params.pop("estimator")
        tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
        return AdaBoostClassifier(estimator=tree, **params)
    raise RuntimeError(f"Unexpected selected model: {name}")

def model_inputs(train, test, representation):
    tr_num = train[numeric].astype(float).to_numpy()
    te_num = test[numeric].astype(float).to_numpy()
    encoder = OneHotEncoder(drop=None, handle_unknown="ignore", sparse_output=False, dtype=np.float64)
    tr_cat = encoder.fit_transform(train[categorical].astype(str))
    te_cat = encoder.transform(test[categorical].astype(str))
    scaler = None
    if representation == "Scaled OHE":
        scaler = StandardScaler()
        tr_num = scaler.fit_transform(tr_num)
        te_num = scaler.transform(te_num)
    elif representation != "Unscaled OHE":
        raise RuntimeError(f"Unexpected preprocessing: {representation}")
    x_train = np.hstack([tr_num, tr_cat])
    x_test = np.hstack([te_num, te_cat])
    if not np.isfinite(x_train).all() or not np.isfinite(x_test).all():
        raise RuntimeError("Non-finite model inputs")
    return x_train, x_test, encoder, scaler

models_to_save = []
qc_rows = []
for fold in range(5):
    train = master.loc[master["__FOLD"] != fold].reset_index(drop=True)
    test = master.loc[master["__FOLD"] == fold].reset_index(drop=True)
    reference = saved.loc[test["__KEY"].to_numpy()]
    if not np.array_equal(reference["Outer_Fold"].to_numpy(int), np.full(len(test), fold)):
        raise RuntimeError(f"Outer fold alignment failed: {fold}")
    probabilities = {}
    for stage_num in (1, 2):
        stage = f"Stage {stage_num}"
        winner = winners.loc[(winners["Outer_Fold"] == fold) & (winners["Stage"] == stage)]
        cal = calibration.loc[(calibration["Outer_Fold"] == fold) & (calibration["Stage"] == stage)]
        if len(winner) != 1 or len(cal) != 1:
            raise RuntimeError(f"Non-unique selected model/calibration: {fold}, {stage}")
        winner, cal = winner.iloc[0], cal.iloc[0]
        candidate_id = str(winner["Candidate_ID"])
        if candidate_id != EXPECTED_WINNERS[fold][stage_num - 1]:
            raise RuntimeError(f"Winner differs from executed notebook: {fold}, {stage}")
        match = registry.loc[registry["Candidate_ID"].astype(str) == candidate_id]
        if len(match) != 1:
            raise RuntimeError(f"Missing or duplicate candidate: {candidate_id}")
        candidate = match.iloc[0]
        family, prep = str(candidate["Model"]), str(candidate["Preprocessing"])
        if (family != str(winner["Model"]) or family != str(cal["Model"])
            or prep != str(winner["Preprocessing"]) or candidate_id != str(cal["Candidate_ID"])):
            raise RuntimeError(f"Candidate metadata mismatch: {fold}, {stage}")
        training_subset = train if stage_num == 1 else train.loc[train["ANC_CLASS"].isin([1, 2])].reset_index(drop=True)
        positive = 0 if stage_num == 1 else 2
        y_train = (training_subset["ANC_CLASS"].to_numpy(int) == positive).astype(int)
        w = training_subset["SURVEY_WEIGHT"].to_numpy(float)
        if not np.isfinite(w).all() or (w <= 0).any():
            raise RuntimeError("Invalid survey weights")
        w = w / w.mean()
        x_train, x_test, encoder, scaler = model_inputs(training_subset, test, prep)
        estimator = instantiate(candidate)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            estimator.fit(x_train, y_train, sample_weight=w)
        classes = np.asarray(estimator.classes_)
        if set(classes.tolist()) != {0, 1}:
            raise RuntimeError("Unexpected estimator binary classes")
        raw = estimator.predict_proba(x_test)[:, int(np.flatnonzero(classes == 1)[0])]
        alpha, beta = float(cal["Calibration_Alpha"]), float(cal["Calibration_Beta"])
        clipped = np.clip(raw, 1e-6, 1 - 1e-6)
        calibrated = expit(alpha + beta * np.log(clipped / (1 - clipped)))
        raw_col = "Stage1_Raw_Score" if stage_num == 1 else "Stage2_Raw_Score"
        cal_col = "q0_CALIBRATED" if stage_num == 1 else "q2_CALIBRATED_CONDITIONAL"
        raw_error = float(np.max(np.abs(raw - reference[raw_col].to_numpy(float))))
        cal_error = float(np.max(np.abs(calibrated - reference[cal_col].to_numpy(float))))
        print(f"Fold {fold} | {stage:7} | {family:31} | raw={raw_error:.3e} | cal={cal_error:.3e}")
        if max(raw_error, cal_error) > TOL:
            raise RuntimeError(f"Prediction replay mismatch in fold={fold}, {stage}. No models saved.")
        probabilities[stage_num] = calibrated
        models_to_save.append((fold, stage_num, {
            "estimator": estimator, "encoder": encoder, "scaler": scaler,
            "numeric_predictors": numeric, "categorical_predictors": categorical,
            "model_name": family, "preprocessing": prep, "candidate_id": candidate_id,
            "outer_fold": fold, "stage": stage,
            "calibration_alpha": alpha, "calibration_beta": beta,
            "optimized_threshold": float(cal["Optimized_Threshold"]),
        }))
        qc_rows.append({
            "Outer_Fold": fold, "Stage": stage, "Model": family,
            "Candidate_ID": candidate_id, "Rows": len(test),
            "Raw_MaxAbsError": raw_error, "Calibrated_MaxAbsError": cal_error,
        })
    q0, q2 = probabilities[1], probabilities[2]
    joint = {
        "P_NO_ANC": q0,
        "P_ANC_1_7": (1 - q0) * (1 - q2),
        "P_ANC_GE8": (1 - q0) * q2,
    }
    max_fusion_error = max(float(np.max(np.abs(p - reference[col].to_numpy(float)))) for col, p in joint.items())
    if max_fusion_error > TOL:
        raise RuntimeError(f"Three-class probability replay mismatch: fold={fold}")
    print(f"Fold {fold} | fused probabilities PASS ({max_fusion_error:.3e})")

if len(models_to_save) != 10 or len(qc_rows) != 10:
    raise RuntimeError("Reconstruction incomplete; nothing saved")

output = ROOT / "10_explainability" / "XAI_SELECTED_REFITS"
output.mkdir(parents=True, exist_ok=True)
for fold, stage_num, bundle in models_to_save:
    filename = f"outer_{fold}_stage_{stage_num}.joblib"
    temporary = output / (filename + ".part")
    joblib.dump(bundle, temporary, compress=3)
    temporary.replace(output / filename)
qc_temp = output / "reconstruction_qc.csv.part"
pd.DataFrame(qc_rows).to_csv(qc_temp, index=False)
qc_temp.replace(output / "reconstruction_qc.csv")
manifest = {
    "status": "PASS", "number_of_models": 10,
    "project_root": str(ROOT), "original_data_sha256": EXPECTED_SHA256,
    "software": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
manifest_tmp = output / "reconstruction_manifest.json.part"
manifest_tmp.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest_tmp.replace(output / "reconstruction_manifest.json")
print("\nRECONSTRUCTION PASS")
print("Verified models: 10/10")
print("Saved:", output)

PROJECT ROOT: E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026
REQUIRED FILES: 7/7 | FROZEN HASHES: PASS
scikit-learn: 1.9.0


C:\Users\M R Computer\AppData\Local\Temp\ipykernel_14520\3588097123.py:113: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  saved = oof.set_index("__KEY", verify_integrity=True)


Fold 0 | Stage 1 | ElasticNet logistic regression  | raw=1.665e-16 | cal=2.220e-16
Fold 0 | Stage 2 | GradientBoosting                | raw=9.888e-17 | cal=9.714e-17
Fold 0 | fused probabilities PASS (2.220e-16)
Fold 1 | Stage 1 | ElasticNet logistic regression  | raw=2.220e-16 | cal=2.220e-16
Fold 1 | Stage 2 | GradientBoosting                | raw=9.888e-17 | cal=9.714e-17
Fold 1 | fused probabilities PASS (2.220e-16)
Fold 2 | Stage 1 | ElasticNet logistic regression  | raw=1.388e-16 | cal=2.220e-16
Fold 2 | Stage 2 | AdaBoost                        | raw=8.327e-17 | cal=1.388e-16
Fold 2 | fused probabilities PASS (2.220e-16)
Fold 3 | Stage 1 | ElasticNet logistic regression  | raw=1.665e-16 | cal=1.665e-16
Fold 3 | Stage 2 | AdaBoost                        | raw=8.327e-17 | cal=9.801e-17
Fold 3 | fused probabilities PASS (2.220e-16)
Fold 4 | Stage 1 | ElasticNet logistic regression  | raw=2.776e-16 | cal=2.220e-16
Fold 4 | Stage 2 | AdaBoost                        | raw=8.327e-17 | 

In [10]:

from pathlib import Path
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import average_precision_score

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
REFITS = ROOT / "10_explainability" / "XAI_SELECTED_REFITS"
OUTPUT = ROOT / "10_explainability" / "XAI_PRIMARY_PERMUTATION"

N_REPEATS = 10
SEED = 2026
TOL = 1e-10

master_path = ROOT / "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
outer_path = ROOT / "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv"
oof_path = ROOT / "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
manifest_path = REFITS / "reconstruction_manifest.json"
qc_path = REFITS / "reconstruction_qc.csv"

required = [master_path, outer_path, oof_path, manifest_path, qc_path]
required += [
    REFITS / f"outer_{fold}_stage_{stage}.joblib"
    for fold in range(5)
    for stage in (1, 2)
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

if manifest.get("status") != "PASS" or int(manifest.get("number_of_models", 0)) != 10:
    raise RuntimeError("Reconstruction manifest has not passed QC")

qc = pd.read_csv(qc_path)

if (
    len(qc) != 10
    or qc[["Raw_MaxAbsError", "Calibrated_MaxAbsError"]].isna().any().any()
    or qc[["Raw_MaxAbsError", "Calibrated_MaxAbsError"]].max().max() > TOL
):
    raise RuntimeError("Saved reconstruction QC is incomplete")

def hash_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for label, path in (("master", master_path), ("outer", outer_path)):
    expected = manifest.get("original_data_sha256", {}).get(label)
    if expected and hash_file(path) != expected:
        raise RuntimeError(f"Original frozen file changed: {label}")

master = pd.read_csv(master_path)
outer = pd.read_csv(outer_path)
oof = pd.read_csv(oof_path)

def normalize_key(values):
    numeric = pd.to_numeric(values, errors="coerce")
    if numeric.notna().all() and np.allclose(numeric, np.round(numeric)):
        return pd.Series(
            np.round(numeric.to_numpy(float)).astype(np.int64),
            index=values.index
        )
    return values.astype(str).str.strip()

for table, column in (
    (master, "_ROW_ID"),
    (outer, "_ROW_ID"),
    (oof, "_ROW_KEY")
):
    table["__KEY"] = normalize_key(table[column])
    if table["__KEY"].duplicated().any():
        raise RuntimeError(f"Duplicate participant IDs in {column}")

if len(master) != 5063 or not (
    set(master.__KEY) == set(outer.__KEY) == set(oof.__KEY)
):
    raise RuntimeError("Master, CV splits and OOF participants disagree")

fold_lookup = outer.set_index("__KEY")["OUTER_FOLD"]
master["__FOLD"] = master["__KEY"].map(fold_lookup)

if master["__FOLD"].isna().any():
    raise RuntimeError("Missing frozen outer-fold assignment")

master["__FOLD"] = master["__FOLD"].astype(int)

if sorted(master["__FOLD"].unique().tolist()) != list(range(5)):
    raise RuntimeError("Expected five outer folds numbered 0-4")

oof_indexed = oof.set_index("__KEY")

if not oof_indexed.index.is_unique:
    raise RuntimeError("OOF participant IDs are not unique")

oof = oof_indexed.loc[master["__KEY"]].reset_index(drop=True)

y = master["ANC_CLASS"].to_numpy(int)
w = master["SURVEY_WEIGHT"].to_numpy(float)

if (
    not np.array_equal(y, oof["ANC_CLASS_TRUE"].to_numpy(int))
    or not np.array_equal(
        master["__FOLD"].to_numpy(),
        oof["Outer_Fold"].to_numpy(int)
    )
    or not np.allclose(
        w,
        oof["SURVEY_WEIGHT"].to_numpy(float),
        rtol=0,
        atol=1e-12
    )
):
    raise RuntimeError("OOF labels, weights or folds are misaligned")

if (
    set(np.unique(y)) != {0, 1, 2}
    or not np.isfinite(w).all()
    or (w <= 0).any()
):
    raise RuntimeError("Unexpected outcome classes or survey weights")

q0 = oof["q0_CALIBRATED"].to_numpy(float)
q2 = oof["q2_CALIBRATED_CONDITIONAL"].to_numpy(float)

expected_final = {
    "P_NO_ANC": q0,
    "P_ANC_1_7": (1 - q0) * (1 - q2),
    "P_ANC_GE8": (1 - q0) * q2,
}

for label, prob in expected_final.items():
    if not np.allclose(
        prob,
        oof[label].to_numpy(float),
        atol=TOL,
        rtol=0
    ):
        raise RuntimeError(f"Saved fused probabilities disagree: {label}")

models = {}
features = None

for fold in range(5):
    for stage in (1, 2):
        bundle = joblib.load(
            REFITS / f"outer_{fold}_stage_{stage}.joblib"
        )

        if (
            int(bundle["outer_fold"]) != fold
            or bundle["stage"] != f"Stage {stage}"
        ):
            raise RuntimeError("Saved model metadata mismatch")

        current = (
            list(bundle["numeric_predictors"])
            + list(bundle["categorical_predictors"])
        )

        if features is None:
            features = current

        if current != features:
            raise RuntimeError("Predictor definitions differ across models")

        models[fold, stage] = bundle

if len(features) != 15 or not set(features).issubset(master.columns):
    raise RuntimeError("Expected 15 original predictors")

def predict_calibrated(bundle, data):
    xn = data[bundle["numeric_predictors"]].astype(float).to_numpy()

    if bundle["preprocessing"] == "Scaled OHE":
        xn = bundle["scaler"].transform(xn)
    elif bundle["preprocessing"] != "Unscaled OHE":
        raise RuntimeError("Unexpected preprocessing representation")

    xc = bundle["encoder"].transform(
        data[bundle["categorical_predictors"]].astype(str)
    )

    x = np.hstack((xn, xc))
    model = bundle["estimator"]

    positive = np.flatnonzero(
        np.asarray(model.classes_) == 1
    )

    if len(positive) != 1:
        raise RuntimeError("Invalid positive-class mapping")

    raw = model.predict_proba(x)[:, int(positive[0])]
    clipped = np.clip(raw, 1e-6, 1 - 1e-6)

    return expit(
        bundle["calibration_alpha"]
        + bundle["calibration_beta"] * np.log(
            clipped / (1 - clipped)
        )
    )

fold_data = {}

print("FROZEN MODEL VERIFICATION")

for fold in range(5):
    positions = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    data = (
        master.iloc[positions][features]
        .copy()
        .reset_index(drop=True)
    )

    pred0 = predict_calibrated(models[fold, 1], data)
    pred2 = predict_calibrated(models[fold, 2], data)

    err0 = np.max(np.abs(pred0 - q0[positions]))
    err2 = np.max(np.abs(pred2 - q2[positions]))

    if max(err0, err2) > TOL:
        raise RuntimeError(
            f"Frozen OOF prediction mismatch in fold {fold}"
        )

    eligible_idx = np.flatnonzero(y[positions] != 0)

    fold_data[fold] = (positions, data, eligible_idx)

    print(
        f"Fold {fold}: Stage 1={err0:.2e}, Stage 2={err2:.2e}"
    )

all_mask = np.ones(len(master), dtype=bool)
users_mask = y != 0

def weighted_ap(mask, probability, positive):
    return float(
        average_precision_score(
            (y[mask] == positive).astype(int),
            probability[mask],
            sample_weight=w[mask]
        )
    )

def evaluate(q0_current, q2_current, q2_users_current):
    multiclass = [
        q0_current,
        (1 - q0_current) * (1 - q2_current),
        (1 - q0_current) * q2_current
    ]

    class_ap = [
        weighted_ap(all_mask, multiclass[i], i)
        for i in range(3)
    ]

    return {
        "stage1_no_anc_ap": weighted_ap(
            all_mask, q0_current, 0
        ),
        "stage2_ge8_ap": weighted_ap(
            users_mask, q2_users_current, 2
        ),
        "final_macro_ap": float(np.mean(class_ap)),
        "final_no_anc_ap": class_ap[0],
        "final_1_7_ap": class_ap[1],
        "final_ge8_ap": class_ap[2],
    }

baseline = evaluate(q0, q2, q2)

reference_ap = {
    "stage1_no_anc_ap": 0.235306,
    "stage2_ge8_ap": 0.161610,
    "final_macro_ap": 0.434322,
}

print("\nORIGINAL WEIGHTED OOF AVERAGE PRECISION")

for name, value in baseline.items():
    print(f"{name:<22} {value:.6f}")

for name, expected in reference_ap.items():
    if abs(baseline[name] - expected) > 0.00005:
        raise RuntimeError(
            f"Baseline mismatch with locked Step 20: {name}"
        )

OUTPUT.mkdir(parents=True, exist_ok=True)

checkpoint = OUTPUT / "primary_permutation_repeat_level.csv"
spec_path = OUTPUT / "primary_permutation_run_spec.json"

sources = [master_path, outer_path, oof_path] + [
    REFITS / f"outer_{fold}_stage_{stage}.joblib"
    for fold in range(5)
    for stage in (1, 2)
]

spec = {
    "method": "unconditional within-outer-fold grouped predictor permutation",
    "seed": SEED,
    "repeats": N_REPEATS,
    "features": features,
    "inputs_sha256": {
        str(p.relative_to(ROOT)): hash_file(p)
        for p in sources
    },
}

if spec_path.exists():
    if json.loads(
        spec_path.read_text(encoding="utf-8")
    ) != spec:
        raise RuntimeError(
            "Existing XAI checkpoint belongs to a different run"
        )

elif checkpoint.exists():
    raise RuntimeError(
        "Existing result checkpoint has no run specification"
    )

else:
    spec_path.write_text(
        json.dumps(spec, indent=2),
        encoding="utf-8"
    )

columns = (
    ["feature", "repeat"]
    + ["delta_" + x for x in baseline]
)

results = (
    pd.read_csv(checkpoint)
    if checkpoint.exists()
    else pd.DataFrame(columns=columns)
)

if not results.empty:
    if (
        set(results.columns) != set(columns)
        or results.duplicated(["feature", "repeat"]).any()
        or not set(results["feature"]).issubset(features)
        or not results["repeat"].between(1, N_REPEATS).all()
    ):
        raise RuntimeError(
            "Saved permutation checkpoint has invalid content"
        )

    delta_values = results[
        ["delta_" + x for x in baseline]
    ].to_numpy(float)

    if not np.isfinite(delta_values).all():
        raise RuntimeError("Nonfinite checkpoint importance values")

completed = set(
    zip(
        results["feature"],
        results["repeat"].astype(int)
    )
)

print("\nCROSS-FITTED GROUPED PERMUTATION IMPORTANCE")

for j, feature in enumerate(features):

    for rep in range(1, N_REPEATS + 1):

        if (feature, rep) in completed:
            continue

        pert_q0 = q0.copy()
        pert_q2 = q2.copy()
        pert_cond_q2 = q2.copy()

        for fold in range(5):
            idx, data, eligible_idx = fold_data[fold]

            rng = np.random.default_rng(
                np.random.SeedSequence(
                    [SEED, j, rep, fold, 0]
                )
            )

            shuffled = data.copy()

            shuffled[feature] = data[feature].to_numpy()[
                rng.permutation(len(data))
            ]

            pert_q0[idx] = predict_calibrated(
                models[fold, 1],
                shuffled
            )

            pert_q2[idx] = predict_calibrated(
                models[fold, 2],
                shuffled
            )

            eligible = data.iloc[eligible_idx].copy()

            rng_eligible = np.random.default_rng(
                np.random.SeedSequence(
                    [SEED, j, rep, fold, 1]
                )
            )

            eligible[feature] = eligible[feature].to_numpy()[
                rng_eligible.permutation(len(eligible))
            ]

            pert_cond_q2[idx[eligible_idx]] = predict_calibrated(
                models[fold, 2],
                eligible
            )

        compared = evaluate(
            pert_q0,
            pert_q2,
            pert_cond_q2
        )

        row = {
            "feature": feature,
            "repeat": rep
        }

        row.update({
            "delta_" + k: baseline[k] - compared[k]
            for k in baseline
        })

        results = pd.concat(
            [results, pd.DataFrame([row])],
            ignore_index=True
        )

        tmp = checkpoint.with_suffix(".csv.tmp")

        results.to_csv(tmp, index=False)
        tmp.replace(checkpoint)

        completed.add((feature, rep))

    print(
        f"{j + 1:02d}/{len(features)}: "
        f"{feature} ({N_REPEATS}/{N_REPEATS} repeats)"
    )

if len(completed) != len(features) * N_REPEATS:
    raise RuntimeError("Incomplete grouped permutation run")

metrics = ["delta_" + x for x in baseline]

summary_rows = []

for feature in features:
    subset = results.loc[
        results["feature"] == feature
    ]

    row = {
        "feature": feature,
        "n_repeats": len(subset)
    }

    for col in metrics:
        row[col + "_mean"] = float(
            subset[col].mean()
        )

        row[col + "_sd"] = float(
            subset[col].std(ddof=1)
        )

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values(
    "delta_final_macro_ap_mean",
    ascending=False
)

summary.to_csv(
    OUTPUT / "primary_permutation_summary.csv",
    index=False
)

pd.DataFrame([baseline]).to_csv(
    OUTPUT / "primary_permutation_baseline.csv",
    index=False
)

for metric, label in [
    ("delta_stage1_no_anc_ap", "Stage 1: no ANC"),
    ("delta_stage2_ge8_ap", "Stage 2: >=8 visits among ANC users"),
    ("delta_final_macro_ap", "Final three-class: macro average precision"),
]:
    part = summary.sort_values(metric + "_mean")

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(
        part["feature"],
        part[metric + "_mean"],
        xerr=part[metric + "_sd"],
        capsize=2
    )

    ax.axvline(0, linewidth=0.8)

    ax.set_xlabel(
        "Decrease in survey-weighted average precision"
    )

    ax.set_title(label)

    fig.tight_layout()

    fig.savefig(
        OUTPUT / f"{metric}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

cols_show = [
    "feature",
    "delta_stage1_no_anc_ap_mean",
    "delta_stage2_ge8_ap_mean",
    "delta_final_macro_ap_mean"
]

print("\nGROUPED PERMUTATION COMPLETE")

print(
    summary[cols_show].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nOutput:", OUTPUT)

FROZEN MODEL VERIFICATION
Fold 0: Stage 1=2.22e-16, Stage 2=9.71e-17
Fold 1: Stage 1=2.22e-16, Stage 2=9.71e-17
Fold 2: Stage 1=2.22e-16, Stage 2=1.39e-16
Fold 3: Stage 1=1.67e-16, Stage 2=9.80e-17
Fold 4: Stage 1=2.22e-16, Stage 2=9.71e-17

ORIGINAL WEIGHTED OOF AVERAGE PRECISION
stage1_no_anc_ap       0.235306
stage2_ge8_ap          0.161610
final_macro_ap         0.434322
final_no_anc_ap        0.235306
final_1_7_ap           0.906936
final_ge8_ap           0.160725

CROSS-FITTED GROUPED PERMUTATION IMPORTANCE
01/15: maternal_age_at_index_birth_years (10/10 repeats)
02/15: prior_live_births (10/10 repeats)
03/15: household_size_members (10/10 repeats)
04/15: maternal_education (10/10 repeats)
05/15: wealth_quintile (10/10 repeats)
06/15: partner_education_primary_final (10/10 repeats)
07/15: division (10/10 repeats)
08/15: pregnancy_intention (10/10 repeats)
09/15: healthcare_decision_primary_final (10/10 repeats)
10/15: residence (10/10 repeats)
11/15: weekly_media_exposure (10/10 

In [12]:

import sys
import subprocess
import importlib

print("Python environment:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "shap",
    "--upgrade-strategy",
    "only-if-needed"
])

importlib.invalidate_caches()

import shap

print("\nSHAP VERSION:", shap.__version__)
print("SHAP INSTALLATION: PASS")

Python environment: D:\Anaconda\python.exe

SHAP VERSION: 0.54.1
SHAP INSTALLATION: PASS


In [13]:

from pathlib import Path
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.special import expit
from scipy.stats import spearmanr

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
REFITS = ROOT / "10_explainability" / "XAI_SELECTED_REFITS"
PERMUTATION = ROOT / "10_explainability" / "XAI_PRIMARY_PERMUTATION"
OUTPUT = ROOT / "10_explainability" / "XAI_PRIMARY_SHAP"

EVAL_PER_FOLD = 160
BACKGROUND_SIZE = 32
N_PERMUTATIONS = 2
SEED = 2026
TOL = 1e-10

master_path = ROOT / "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
outer_path = ROOT / "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv"
oof_path = ROOT / "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
manifest_path = REFITS / "reconstruction_manifest.json"
qc_path = REFITS / "reconstruction_qc.csv"
permutation_path = PERMUTATION / "primary_permutation_summary.csv"

model_paths = {
    (fold, stage): REFITS / f"outer_{fold}_stage_{stage}.joblib"
    for fold in range(5)
    for stage in (1, 2)
}

required = [
    master_path, outer_path, oof_path,
    manifest_path, qc_path, permutation_path,
    *model_paths.values()
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
qc = pd.read_csv(qc_path)

if manifest.get("status") != "PASS" or int(manifest.get("number_of_models", 0)) != 10:
    raise RuntimeError("Selected-model reconstruction is not verified")

if (
    len(qc) != 10
    or qc[["Raw_MaxAbsError", "Calibrated_MaxAbsError"]].max().max() > TOL
):
    raise RuntimeError("Reconstruction QC is incomplete")

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

for label, file in [
    ("master", master_path),
    ("outer", outer_path),
    ("oof", oof_path)
]:
    expected_hash = manifest.get("original_data_sha256", {}).get(label)
    if expected_hash and sha256(file) != expected_hash:
        raise RuntimeError(f"Frozen input hash mismatch: {label}")

master = pd.read_csv(master_path)
outer = pd.read_csv(outer_path)
oof = pd.read_csv(oof_path)

def normalize_id(values):
    numeric = pd.to_numeric(values, errors="coerce")
    if numeric.notna().all() and np.allclose(numeric, np.round(numeric)):
        return pd.Series(
            np.round(numeric.to_numpy(float)).astype(np.int64),
            index=values.index
        )
    return values.astype(str).str.strip()

for df, col in [
    (master, "_ROW_ID"),
    (outer, "_ROW_ID"),
    (oof, "_ROW_KEY")
]:
    df["__KEY"] = normalize_id(df[col])
    if df["__KEY"].duplicated().any():
        raise RuntimeError(f"Nonunique IDs: {col}")

if len(master) != 5063 or not (
    set(master.__KEY) == set(outer.__KEY) == set(oof.__KEY)
):
    raise RuntimeError("Frozen participant IDs disagree")

master["__FOLD"] = master["__KEY"].map(
    outer.set_index("__KEY")["OUTER_FOLD"]
)

if master["__FOLD"].isna().any():
    raise RuntimeError("Missing outer folds")

master["__FOLD"] = master["__FOLD"].astype(int)

if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError("Outer fold numbering differs")

oof_index = oof.set_index("__KEY")

if not oof_index.index.is_unique:
    raise RuntimeError("OOF IDs not unique")

oof = oof_index.loc[master["__KEY"]].reset_index(drop=True)

true_classes = master["ANC_CLASS"].to_numpy(int)
weights = master["SURVEY_WEIGHT"].to_numpy(float)

if not np.array_equal(
    true_classes, oof["ANC_CLASS_TRUE"].to_numpy(int)
):
    raise RuntimeError("OOF class labels disagree")

if not np.array_equal(
    master["__FOLD"],
    oof["Outer_Fold"].to_numpy(int)
):
    raise RuntimeError("OOF fold assignments disagree")

if not np.allclose(
    weights,
    oof["SURVEY_WEIGHT"].to_numpy(float),
    rtol=0,
    atol=1e-12
):
    raise RuntimeError("Survey weights differ")

if not np.isfinite(weights).all() or (weights <= 0).any():
    raise RuntimeError("Invalid survey weights")

models = {
    key: joblib.load(path)
    for key, path in model_paths.items()
}

num = list(models[0, 1]["numeric_predictors"])
cat = list(models[0, 1]["categorical_predictors"])
features = num + cat

if len(features) != 15 or not set(features).issubset(master.columns):
    raise RuntimeError("Expected 15 original predictors")

for (fold, stage), model in models.items():
    if (
        int(model["outer_fold"]) != fold
        or model["stage"] != f"Stage {stage}"
        or model["numeric_predictors"] + model["categorical_predictors"] != features
    ):
        raise RuntimeError("Model provenance or predictor-order mismatch")

levels = {
    feature: np.sort(master[feature].astype(str).unique())
    for feature in cat
}

X_original = np.empty(
    (len(master), len(features)),
    dtype=np.float64
)

for j, feature in enumerate(features):
    if feature in num:
        X_original[:, j] = master[feature].astype(float).to_numpy()
    else:
        codes = pd.Categorical(
            master[feature].astype(str),
            categories=levels[feature]
        ).codes

        if (codes < 0).any():
            raise RuntimeError(f"Unmapped categorical values: {feature}")

        X_original[:, j] = codes

if not np.isfinite(X_original).all():
    raise RuntimeError("SHAP input contains nonfinite values")

def recover_frame(array):
    z = np.asarray(array, dtype=np.float64)

    if z.ndim == 1:
        z = z.reshape(1, -1)

    if z.shape[1] != len(features):
        raise RuntimeError("Unexpected SHAP feature count")

    output = pd.DataFrame(z, columns=features)

    for feature in cat:
        col = features.index(feature)
        codes = np.rint(z[:, col]).astype(int)

        if not np.allclose(
            z[:, col], codes, atol=1e-8, rtol=0
        ):
            raise RuntimeError("Masker produced a noncategorical level")

        if (codes < 0).any() or (codes >= len(levels[feature])).any():
            raise RuntimeError(f"Invalid category code: {feature}")

        output[feature] = levels[feature][codes]

    return output

def predict_probability(bundle, encoded_original):
    df = recover_frame(encoded_original)

    xn = df[
        bundle["numeric_predictors"]
    ].astype(float).to_numpy()

    if bundle["preprocessing"] == "Scaled OHE":
        xn = bundle["scaler"].transform(xn)
    elif bundle["preprocessing"] != "Unscaled OHE":
        raise RuntimeError("Unknown preprocessing representation")

    xc = bundle["encoder"].transform(
        df[bundle["categorical_predictors"]].astype(str)
    )

    xt = np.hstack([xn, xc])

    classes = np.asarray(bundle["estimator"].classes_)
    pos = np.flatnonzero(classes == 1)

    if len(pos) != 1:
        raise RuntimeError("Invalid positive-class index")

    raw = bundle["estimator"].predict_proba(
        xt
    )[:, int(pos[0])]

    raw = np.clip(raw, 1e-6, 1 - 1e-6)

    return expit(
        float(bundle["calibration_alpha"])
        + float(bundle["calibration_beta"])
        * np.log(raw / (1 - raw))
    )

for fold in range(5):
    positions = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    for stage, col in [
        (1, "q0_CALIBRATED"),
        (2, "q2_CALIBRATED_CONDITIONAL")
    ]:
        current = predict_probability(
            models[fold, stage],
            X_original[positions]
        )

        error = np.max(
            np.abs(
                current - oof[col].to_numpy(float)[positions]
            )
        )

        if error > TOL:
            raise RuntimeError(
                f"Verified model no longer reproduces OOF: {fold}/{stage}"
            )

print("FROZEN MODEL REPLAY: PASS | 10/10")

run_spec = {
    "mode": "original-feature permutation SHAP of calibrated probabilities",
    "eval_per_outer_fold": EVAL_PER_FOLD,
    "background_size": BACKGROUND_SIZE,
    "n_permutations": N_PERMUTATIONS,
    "seed": SEED,
    "features": features,
    "model_sha256": {
        f"outer_{f}_stage_{s}": sha256(p)
        for (f, s), p in model_paths.items()
    },
    "master_sha256": sha256(master_path),
    "outer_sha256": sha256(outer_path),
    "oof_sha256": sha256(oof_path),
    "shap_version": shap.__version__,
}

OUTPUT.mkdir(parents=True, exist_ok=True)
spec_path = OUTPUT / "shap_run_spec.json"

if spec_path.is_file():
    if json.loads(
        spec_path.read_text(encoding="utf-8")
    ) != run_spec:
        raise RuntimeError(
            "Prior SHAP checkpoint belongs to different inputs/settings"
        )
else:
    if list(OUTPUT.glob("fold_*_stage_*.npz")):
        raise RuntimeError(
            "SHAP checkpoints exist without a matching run specification"
        )

    spec_path.write_text(
        json.dumps(run_spec, indent=2),
        encoding="utf-8"
    )

records = []
fold_records = []
level_records = []

for fold in range(5):
    test_all = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )
    train_all = np.flatnonzero(
        master["__FOLD"].to_numpy() != fold
    )

    for stage in (1, 2):

        if stage == 1:
            test_idx = test_all
            train_idx = train_all
        else:
            test_idx = test_all[
                true_classes[test_all] != 0
            ]
            train_idx = train_all[
                true_classes[train_all] != 0
            ]

        if not len(test_idx) or not len(train_idx):
            raise RuntimeError(
                f"Empty explanation population: {fold}/{stage}"
            )

        rng = np.random.default_rng(
            SEED + 100 * fold + stage
        )

        n_eval = min(EVAL_PER_FOLD, len(test_idx))

        selected = np.sort(
            rng.choice(
                test_idx,
                size=n_eval,
                replace=False
            )
        )

        probs = weights[train_idx] / weights[train_idx].sum()

        background_idx = rng.choice(
            train_idx,
            size=BACKGROUND_SIZE,
            replace=True,
            p=probs
        )

        checkpoint = OUTPUT / f"fold_{fold}_stage_{stage}.npz"

        if checkpoint.is_file():
            with np.load(
                checkpoint, allow_pickle=False
            ) as old:
                values = old["values"]
                base_values = old["base_values"]
                prior_idx = old["selected"]
                prior_bg = old["background_idx"]

            if (
                not np.array_equal(prior_idx, selected)
                or not np.array_equal(prior_bg, background_idx)
                or values.shape != (n_eval, len(features))
            ):
                raise RuntimeError(
                    f"Checkpoint does not match design: {checkpoint}"
                )

            print(
                f"Fold {fold} Stage {stage}: "
                f"restored {n_eval} explanations"
            )

        else:
            def f(z, bundle=models[fold, stage]):
                return predict_probability(bundle, z)

            masker = shap.maskers.Independent(
                X_original[background_idx],
                max_samples=BACKGROUND_SIZE
            )

            explainer = shap.Explainer(
                f,
                masker,
                algorithm="permutation",
                feature_names=features,
                seed=SEED + 100 * fold + stage
            )

            explanation = explainer(
                X_original[selected],
                max_evals=N_PERMUTATIONS * (
                    2 * len(features) + 1
                ),
                silent=True
            )

            values = np.asarray(
                explanation.values, dtype=float
            )
            base_values = np.asarray(
                explanation.base_values, dtype=float
            ).reshape(-1)

            if (
                values.shape != (n_eval, len(features))
                or base_values.shape != (n_eval,)
            ):
                raise RuntimeError(
                    f"Unexpected SHAP dimensions: {fold}/{stage}"
                )

            if (
                not np.isfinite(values).all()
                or not np.isfinite(base_values).all()
            ):
                raise RuntimeError(
                    f"Nonfinite SHAP values: {fold}/{stage}"
                )

            error = np.max(
                np.abs(
                    base_values
                    + values.sum(axis=1)
                    - f(X_original[selected])
                )
            )

            if error > 1e-7:
                raise RuntimeError(
                    f"SHAP additivity failure in fold {fold}/{stage}: "
                    f"{error:.2e}"
                )

            temporary = checkpoint.with_suffix(".npz.part")

            with temporary.open("wb") as output_file:
                np.savez_compressed(
                    output_file,
                    values=values,
                    base_values=base_values,
                    selected=selected,
                    background_idx=background_idx
                )

            temporary.replace(checkpoint)

            print(
                f"Fold {fold} Stage {stage}: "
                f"SHAP computed ({n_eval} held-out rows, "
                f"additivity {error:.1e})"
            )

        model_prob = predict_probability(
            models[fold, stage],
            X_original[selected]
        )

        if np.max(
            np.abs(
                base_values
                + values.sum(axis=1)
                - model_prob
            )
        ) > 1e-7:
            raise RuntimeError(
                f"Stored SHAP additivity mismatch: {fold}/{stage}"
            )

        expansion_factor = len(test_idx) / n_eval
        aw = weights[selected] * expansion_factor

        for j, feature in enumerate(features):
            avg_abs = np.average(
                np.abs(values[:, j]),
                weights=aw
            )
            avg_signed = np.average(
                values[:, j],
                weights=aw
            )

            fold_records.append({
                "fold": fold,
                "stage": stage,
                "feature": feature,
                "n_explained": n_eval,
                "mean_abs_shap": avg_abs,
                "mean_signed_shap": avg_signed
            })

        for i, row_index in enumerate(selected):
            result = {
                "row_key": str(master["__KEY"].iloc[row_index]),
                "fold": fold,
                "stage": stage,
                "anc_class": int(true_classes[row_index]),
                "survey_weight": float(weights[row_index]),
                "analysis_weight": float(aw[i]),
                "base_probability": float(base_values[i]),
                "predicted_probability": float(model_prob[i])
            }

            for j, feature in enumerate(features):
                result[f"SHAP__{feature}"] = float(
                    values[i, j]
                )

            records.append(result)

        for feature in cat:
            col = features.index(feature)
            selected_codes = X_original[
                selected, col
            ].astype(int)

            for code, category in enumerate(levels[feature]):
                mask = selected_codes == code

                if mask.any():
                    level_records.append({
                        "fold": fold,
                        "stage": stage,
                        "feature": feature,
                        "category": str(category),
                        "n": int(mask.sum()),
                        "sum_weight": float(aw[mask].sum()),
                        "weighted_signed_shap": float(
                            np.average(
                                values[mask, col],
                                weights=aw[mask]
                            )
                        )
                    })

row_results = pd.DataFrame(records)
fold_results = pd.DataFrame(fold_records)

if len(fold_results) != 2 * 5 * len(features):
    raise RuntimeError("Incomplete fold-level importance")

stage_summaries = []

for stage in (1, 2):
    segment = row_results[
        row_results.stage == stage
    ]

    wf = segment["analysis_weight"].to_numpy(float)

    for feature in features:
        sv = segment[
            f"SHAP__{feature}"
        ].to_numpy(float)

        stage_summaries.append({
            "stage": stage,
            "feature": feature,
            "n_explained": len(segment),
            "weighted_mean_abs_shap": float(
                np.average(np.abs(sv), weights=wf)
            ),
            "weighted_mean_signed_shap": float(
                np.average(sv, weights=wf)
            )
        })

summary = pd.DataFrame(stage_summaries)

permutation = pd.read_csv(permutation_path)
comparisons = []

for stage, perm_col in [
    (1, "delta_stage1_no_anc_ap_mean"),
    (2, "delta_stage2_ge8_ap_mean")
]:
    a = summary[
        summary.stage == stage
    ].set_index("feature")["weighted_mean_abs_shap"]

    b = permutation.set_index("feature")[perm_col]

    if set(a.index) != set(b.index):
        raise RuntimeError(
            "SHAP and permutation feature sets differ"
        )

    b = b.reindex(a.index)
    rho, _ = spearmanr(
        a.to_numpy(float),
        b.to_numpy(float)
    )

    top_a = set(a.nlargest(5).index)
    top_b = set(b.nlargest(5).index)

    comparisons.append({
        "stage": stage,
        "spearman_rho": float(rho),
        "top5_overlap": len(top_a & top_b),
        "top5_shap": " | ".join(
            a.nlargest(5).index
        ),
        "top5_permutation": " | ".join(
            b.nlargest(5).index
        )
    })

stability = []

for stage in (1, 2):
    pivot = fold_results[
        fold_results.stage == stage
    ].pivot(
        index="feature",
        columns="fold",
        values="mean_abs_shap"
    )

    for left in range(5):
        for right in range(left + 1, 5):
            rho, _ = spearmanr(
                pivot[left],
                pivot[right]
            )

            stability.append({
                "stage": stage,
                "fold_a": left,
                "fold_b": right,
                "rank_spearman": float(rho)
            })

row_results.to_csv(
    OUTPUT / "heldout_original_feature_shap.csv",
    index=False
)
fold_results.to_csv(
    OUTPUT / "fold_shap_importance.csv",
    index=False
)
summary.to_csv(
    OUTPUT / "stage_shap_importance.csv",
    index=False
)
pd.DataFrame(level_records).to_csv(
    OUTPUT / "categorical_level_shap.csv",
    index=False
)
pd.DataFrame(comparisons).to_csv(
    OUTPUT / "shap_permutation_rank_agreement.csv",
    index=False
)
pd.DataFrame(stability).to_csv(
    OUTPUT / "fold_shap_rank_stability.csv",
    index=False
)

for stage in (1, 2):
    segment = summary[
        summary.stage == stage
    ].sort_values("weighted_mean_abs_shap")

    fig, ax = plt.subplots(figsize=(9, 6.3))

    ax.barh(
        segment.feature,
        segment.weighted_mean_abs_shap
    )

    ax.set_xlabel(
        "Survey-weighted mean |SHAP| (predicted probability units)"
    )
    ax.set_title(
        f"Stage {stage} — "
        f"{'No ANC' if stage == 1 else '≥8 visits among ANC users'}"
    )

    fig.tight_layout()

    for ext in ("png", "tiff"):
        fig.savefig(
            OUTPUT / f"stage_{stage}_shap_global.{ext}",
            dpi=300,
            bbox_inches="tight"
        )

    plt.close(fig)

    display_order = segment.feature.tolist()

    sampled = row_results[
        row_results.stage == stage
    ]

    fig, ax = plt.subplots(figsize=(9, 6.8))
    jitter_rng = np.random.default_rng(
        SEED + 20000 + stage
    )

    for row, feature in enumerate(display_order):
        x = sampled[
            f"SHAP__{feature}"
        ].to_numpy(float)

        jitter = jitter_rng.uniform(
            -0.32, 0.32, size=len(x)
        )

        ax.scatter(
            x,
            row + jitter,
            s=7,
            alpha=0.23,
            rasterized=True
        )

    ax.axvline(0, linewidth=0.8)
    ax.set_yticks(
        range(len(display_order)),
        display_order
    )
    ax.set_xlabel(
        "SHAP contribution to calibrated predicted probability"
    )
    ax.set_title(
        f"Stage {stage} — held-out SHAP distributions"
    )

    fig.tight_layout()

    for ext in ("png", "tiff"):
        fig.savefig(
            OUTPUT / f"stage_{stage}_shap_distribution.{ext}",
            dpi=300,
            bbox_inches="tight"
        )

    plt.close(fig)

print("\nSTAGE-SPECIFIC SHAP: COMPLETE")

for stage in (1, 2):
    print(f"\nStage {stage} — top 10 original predictors")

    print(
        summary[
            summary.stage == stage
        ].nlargest(
            10, "weighted_mean_abs_shap"
        )[
            [
                "feature",
                "weighted_mean_abs_shap",
                "weighted_mean_signed_shap"
            ]
        ].to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}"
        )
    )

print("\nSHAP-PERMUTATION AGREEMENT")

print(
    pd.DataFrame(comparisons)[
        ["stage", "spearman_rho", "top5_overlap"]
    ].to_string(index=False)
)

print("\nCROSS-FOLD STABILITY")

print(
    pd.DataFrame(stability)
    .groupby("stage")["rank_spearman"]
    .agg(["mean", "min", "max"])
    .to_string()
)

print("\nSaved:", OUTPUT)

FROZEN MODEL REPLAY: PASS | 10/10
Fold 0 Stage 1: SHAP computed (160 held-out rows, additivity 2.2e-16)
Fold 0 Stage 2: SHAP computed (160 held-out rows, additivity 1.9e-16)
Fold 1 Stage 1: SHAP computed (160 held-out rows, additivity 3.3e-16)
Fold 1 Stage 2: SHAP computed (160 held-out rows, additivity 1.4e-16)
Fold 2 Stage 1: SHAP computed (160 held-out rows, additivity 2.8e-16)
Fold 2 Stage 2: SHAP computed (160 held-out rows, additivity 1.7e-16)
Fold 3 Stage 1: SHAP computed (160 held-out rows, additivity 2.8e-16)
Fold 3 Stage 2: SHAP computed (160 held-out rows, additivity 1.4e-16)
Fold 4 Stage 1: SHAP computed (160 held-out rows, additivity 1.9e-16)
Fold 4 Stage 2: SHAP computed (160 held-out rows, additivity 1.7e-16)

STAGE-SPECIFIC SHAP: COMPLETE

Stage 1 — top 10 original predictors
                          feature  weighted_mean_abs_shap  weighted_mean_signed_shap
                  wealth_quintile                0.027382                   0.002580
                prior_live_

In [14]:

from pathlib import Path
import json
import hashlib

import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from scipy.special import expit
from scipy.stats import spearmanr

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
REFITS = ROOT / "10_explainability" / "XAI_SELECTED_REFITS"
STAGE_SHAP = ROOT / "10_explainability" / "XAI_PRIMARY_SHAP"
OUT = ROOT / "10_explainability" / "XAI_FINAL_MULTICLASS_SHAP"

EVAL_PER_FOLD = 160
BACKGROUND_SIZE = 32
N_PERMUTATIONS = 2
SEED = 2026
TOL = 1e-10

master_file = ROOT / "01_processed_data" / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
fold_file = ROOT / "04_cv_splits" / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
oof_file = ROOT / "06_predictions" / "STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
manifest_file = REFITS / "reconstruction_manifest.json"

model_files = {
    (f, s): REFITS / f"outer_{f}_stage_{s}.joblib"
    for f in range(5)
    for s in (1, 2)
}

stage_files = {
    f: STAGE_SHAP / f"fold_{f}_stage_1.npz"
    for f in range(5)
}

required = [
    master_file, fold_file, oof_file, manifest_file,
    *model_files.values(), *stage_files.values()
]

for file in required:
    if not file.is_file():
        raise FileNotFoundError(file)

manifest = json.loads(
    manifest_file.read_text(encoding="utf-8")
)

if (
    manifest.get("status") != "PASS"
    or manifest.get("number_of_models") != 10
):
    raise RuntimeError("Verified model reconstruction is required")

def sha256(file):
    h = hashlib.sha256()
    with file.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for name, file in [
    ("master", master_file),
    ("outer", fold_file),
    ("oof", oof_file)
]:
    expected = manifest.get(
        "original_data_sha256", {}
    ).get(name)

    if expected and expected != sha256(file):
        raise RuntimeError(f"Frozen input hash mismatch: {name}")

master = pd.read_csv(master_file)
folds = pd.read_csv(fold_file)
oof = pd.read_csv(oof_file)

def key(series):
    n = pd.to_numeric(series, errors="coerce")

    if n.notna().all() and np.allclose(
        n.to_numpy(float),
        np.round(n.to_numpy(float))
    ):
        return pd.Series(
            np.round(
                n.to_numpy(float)
            ).astype(np.int64),
            index=series.index
        )

    return series.astype(str).str.strip()

for frame, name in [
    (master, "_ROW_ID"),
    (folds, "_ROW_ID"),
    (oof, "_ROW_KEY")
]:
    frame["__KEY"] = key(frame[name])

    if frame["__KEY"].duplicated().any():
        raise RuntimeError(f"Repeated participant IDs: {name}")

if len(master) != 5063 or not (
    set(master.__KEY)
    == set(folds.__KEY)
    == set(oof.__KEY)
):
    raise RuntimeError("Master/outer/OOF rows not identical")

master["__FOLD"] = master["__KEY"].map(
    folds.set_index("__KEY")["OUTER_FOLD"]
)

if master["__FOLD"].isna().any():
    raise RuntimeError("Missing CV fold ID")

master["__FOLD"] = master["__FOLD"].astype(int)

if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError("Unexpected CV fold IDs")

oof = (
    oof.set_index("__KEY")
    .loc[master["__KEY"]]
    .reset_index(drop=True)
)

y = master["ANC_CLASS"].to_numpy(int)
w = master["SURVEY_WEIGHT"].to_numpy(float)

if not np.array_equal(
    y, oof["ANC_CLASS_TRUE"].to_numpy(int)
):
    raise RuntimeError("OOF outcome labels are misaligned")

if not np.array_equal(
    master["__FOLD"],
    oof["Outer_Fold"].to_numpy(int)
):
    raise RuntimeError("OOF fold IDs are misaligned")

if not np.allclose(
    w,
    oof["SURVEY_WEIGHT"].to_numpy(float),
    atol=1e-12,
    rtol=0
):
    raise RuntimeError("Survey weights are misaligned")

if not np.isfinite(w).all() or (w <= 0).any():
    raise RuntimeError("Invalid DHS survey weights")

models = {
    k: joblib.load(file)
    for k, file in model_files.items()
}

num = list(models[0, 1]["numeric_predictors"])
cat = list(models[0, 1]["categorical_predictors"])
features = num + cat

if len(features) != 15 or not set(features).issubset(master.columns):
    raise RuntimeError("Incorrect original-feature specification")

for (f, s), bundle in models.items():
    if (
        int(bundle["outer_fold"]) != f
        or bundle["stage"] != f"Stage {s}"
        or (
            list(bundle["numeric_predictors"])
            + list(bundle["categorical_predictors"])
        ) != features
    ):
        raise RuntimeError(
            "Verified model metadata differs across outer folds"
        )

levels = {
    col: np.sort(master[col].astype(str).unique())
    for col in cat
}

X = np.zeros(
    (len(master), len(features)),
    dtype=np.float64
)

for j, col in enumerate(features):
    if col in num:
        X[:, j] = master[col].astype(float).to_numpy()
    else:
        X[:, j] = pd.Categorical(
            master[col].astype(str),
            categories=levels[col]
        ).codes

if not np.isfinite(X).all():
    raise RuntimeError("Incorrect SHAP input representation")

def decode(z):
    z = np.asarray(z, dtype=np.float64)

    if z.ndim == 1:
        z = z[None, :]

    if z.ndim != 2 or z.shape[1] != len(features):
        raise ValueError("Feature shape mismatch")

    frame = pd.DataFrame(z, columns=features)

    for col in cat:
        codes = np.rint(
            frame[col].to_numpy(float)
        ).astype(int)

        if (
            not np.allclose(
                frame[col].to_numpy(float),
                codes,
                atol=1e-8,
                rtol=0
            )
            or (codes < 0).any()
            or (codes >= len(levels[col])).any()
        ):
            raise RuntimeError(
                f"Invalid encoded category: {col}"
            )

        frame[col] = levels[col][codes]

    return frame

def calibrated(bundle, frame):
    xn = frame[
        bundle["numeric_predictors"]
    ].astype(float).to_numpy()

    if bundle["preprocessing"] == "Scaled OHE":
        xn = bundle["scaler"].transform(xn)

    elif bundle["preprocessing"] != "Unscaled OHE":
        raise RuntimeError("Preprocessing mismatch")

    xc = bundle["encoder"].transform(
        frame[
            bundle["categorical_predictors"]
        ].astype(str)
    )

    estimator = bundle["estimator"]

    idx = np.flatnonzero(
        np.asarray(estimator.classes_) == 1
    )

    if len(idx) != 1:
        raise RuntimeError("Positive class not found")

    p = estimator.predict_proba(
        np.hstack([xn, xc])
    )[:, int(idx[0])]

    p = np.clip(p, 1e-6, 1 - 1e-6)

    return expit(
        bundle["calibration_alpha"]
        + bundle["calibration_beta"]
        * np.log(p / (1 - p))
    )

def final_prob(fold, z):
    df = decode(z)

    q0 = calibrated(models[fold, 1], df)
    q2 = calibrated(models[fold, 2], df)

    return np.column_stack([
        q0,
        (1 - q0) * (1 - q2),
        (1 - q0) * q2
    ])

saved_probs = oof[
    ["P_NO_ANC", "P_ANC_1_7", "P_ANC_GE8"]
].to_numpy(float)

for fold in range(5):
    ix = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    error = np.max(
        np.abs(
            final_prob(fold, X[ix])
            - saved_probs[ix]
        )
    )

    if error > TOL:
        raise RuntimeError(
            f"Frozen three-class replay failed: {fold}: {error}"
        )

print("FROZEN THREE-CLASS MODEL REPLAY: PASS | 5/5 folds")

spec = {
    "design": "Sampled outer-test multiclass probability SHAP, original 15 predictors",
    "seed": SEED,
    "evaluation_rows_per_fold": EVAL_PER_FOLD,
    "background_size": BACKGROUND_SIZE,
    "permutations": N_PERMUTATIONS,
    "shap_version": shap.__version__,
    "features": features,
    "sha256": {
        str(p.relative_to(ROOT)): sha256(p)
        for p in [
            master_file,
            fold_file,
            oof_file,
            *model_files.values(),
            *stage_files.values()
        ]
    },
}

OUT.mkdir(parents=True, exist_ok=True)
spec_file = OUT / "run_spec.json"

if spec_file.is_file():
    previous = json.loads(
        spec_file.read_text(encoding="utf-8")
    )

    if previous != spec:
        raise RuntimeError(
            "Existing checkpoints have different SHAP settings"
        )

else:
    if list(OUT.glob("fold_*_multiclass.npz")):
        raise RuntimeError(
            "Existing checkpoints lack a verified run specification"
        )

    spec_file.write_text(
        json.dumps(spec, indent=2),
        encoding="utf-8"
    )

sample_indices = []
all_values = []
all_baselines = []
comparison_errors = []

for fold in range(5):
    train_idx = np.flatnonzero(
        master["__FOLD"].to_numpy() != fold
    )

    test_idx = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    with np.load(
        stage_files[fold],
        allow_pickle=False
    ) as prior:
        selected = prior["selected"].astype(int)
        background = prior["background_idx"].astype(int)
        stage1_values = prior["values"].astype(float)

    if (
        len(selected) != min(EVAL_PER_FOLD, len(test_idx))
        or len(background) != BACKGROUND_SIZE
        or stage1_values.shape != (
            len(selected), len(features)
        )
        or not np.isin(selected, test_idx).all()
        or not np.isin(background, train_idx).all()
        or len(np.unique(selected)) != len(selected)
    ):
        raise RuntimeError(
            f"Previous Stage 1 SHAP sample mismatch: fold {fold}"
        )

    checkpoint = OUT / f"fold_{fold}_multiclass.npz"

    if checkpoint.is_file():
        with np.load(
            checkpoint,
            allow_pickle=False
        ) as previous:
            values = previous["values"]
            base = previous["base"]
            last_selected = previous["selected"]
            last_background = previous["background"]

        if (
            not np.array_equal(selected, last_selected)
            or not np.array_equal(
                background, last_background
            )
        ):
            raise RuntimeError(
                f"Checkpoint sample changed: fold {fold}"
            )

        print(
            f"Fold {fold}: multiclass SHAP checkpoint restored"
        )

    else:
        explainer = shap.Explainer(
            lambda z, current_fold=fold: final_prob(
                current_fold, z
            ),
            shap.maskers.Independent(
                X[background],
                max_samples=BACKGROUND_SIZE
            ),
            algorithm="permutation",
            feature_names=features,
            seed=SEED + 100 * fold + 1,
        )

        explanation = explainer(
            X[selected],
            max_evals=N_PERMUTATIONS * (
                2 * len(features) + 1
            ),
            silent=True
        )

        values = np.asarray(
            explanation.values,
            dtype=float
        )

        base = np.asarray(
            explanation.base_values,
            dtype=float
        )

        if (
            values.shape != (
                len(selected), len(features), 3
            )
            or base.shape != (len(selected), 3)
        ):
            raise RuntimeError(
                f"Unexpected SHAP dimensions: fold {fold}"
            )

        if (
            not np.isfinite(values).all()
            or not np.isfinite(base).all()
        ):
            raise RuntimeError("Non-finite SHAP values")

        temporary = checkpoint.with_suffix(".npz.part")

        with temporary.open("wb") as dest:
            np.savez_compressed(
                dest,
                values=values,
                base=base,
                selected=selected,
                background=background
            )

        temporary.replace(checkpoint)

        print(
            f"Fold {fold}: computed "
            f"{len(selected)} held-out three-class explanations"
        )

    predicted = final_prob(fold, X[selected])

    additivity_error = float(
        np.max(
            np.abs(
                base + values.sum(axis=1) - predicted
            )
        )
    )

    base_sum_error = float(
        np.max(
            np.abs(base.sum(axis=1) - 1)
        )
    )

    feature_sum_error = float(
        np.max(
            np.abs(values.sum(axis=2))
        )
    )

    if max(
        additivity_error,
        base_sum_error,
        feature_sum_error
    ) > 1e-7:
        raise RuntimeError(
            f"Multiclass SHAP QC failed: fold {fold}"
        )

    class0_error = float(
        np.max(
            np.abs(
                values[:, :, 0] - stage1_values
            )
        )
    )

    comparison_errors.append(class0_error)

    print(
        f"  Additivity={additivity_error:.2e} | "
        f"Sum-to-one={base_sum_error:.2e} | "
        f"Class-0 comparison={class0_error:.2e}"
    )

    sample_indices.append(selected)
    all_values.append(values)
    all_baselines.append(base)

indices = np.concatenate(sample_indices)
values = np.concatenate(all_values, axis=0)
bases = np.concatenate(all_baselines, axis=0)

if (
    len(indices) != 5 * EVAL_PER_FOLD
    or len(np.unique(indices)) != len(indices)
):
    raise RuntimeError(
        "Incomplete or repeated held-out explanation samples"
    )

fold_ids = (
    master["__FOLD"]
    .to_numpy(int)[indices]
)

expansion = np.array([
    (master["__FOLD"] == int(f)).sum()
    / EVAL_PER_FOLD
    for f in fold_ids
])

analysis_weights = w[indices] * expansion

class_names = [
    "No ANC",
    "1-7 visits",
    ">=8 visits"
]

individual = pd.DataFrame({
    "row_key": master["__KEY"].iloc[indices].astype(str).to_numpy(),
    "outer_fold": fold_ids,
    "true_class": y[indices],
    "survey_weight": w[indices],
    "analysis_weight": analysis_weights,
})

for cls in range(3):
    individual[f"class{cls}_base"] = bases[:, cls]

    individual[f"class{cls}_predicted"] = (
        saved_probs[indices, cls]
    )

    for j, feat in enumerate(features):
        individual[f"class{cls}__{feat}"] = (
            values[:, j, cls]
        )

individual.to_csv(
    OUT / "heldout_multiclass_original_feature_shap.csv",
    index=False
)

summary = []
fold_results = []

for cls in range(3):
    for j, feat in enumerate(features):
        sh = values[:, j, cls]

        summary.append({
            "class_id": cls,
            "class_label": class_names[cls],
            "feature": feat,
            "mean_abs_shap": float(
                np.average(
                    np.abs(sh),
                    weights=analysis_weights
                )
            ),
            "mean_signed_shap": float(
                np.average(
                    sh,
                    weights=analysis_weights
                )
            ),
            "n_explained": len(indices),
        })

        for fold in range(5):
            mask = fold_ids == fold

            fold_results.append({
                "class_id": cls,
                "outer_fold": fold,
                "feature": feat,
                "mean_abs_shap": float(
                    np.average(
                        np.abs(sh[mask]),
                        weights=analysis_weights[mask]
                    )
                )
            })

summary = pd.DataFrame(summary)
fold_results = pd.DataFrame(fold_results)

summary.to_csv(
    OUT / "multiclass_feature_importance.csv",
    index=False
)

fold_results.to_csv(
    OUT / "multiclass_fold_importance.csv",
    index=False
)

stability = []

for cls in range(3):
    subset = fold_results[
        fold_results.class_id == cls
    ].pivot(
        index="feature",
        columns="outer_fold",
        values="mean_abs_shap"
    )

    for a in range(5):
        for b in range(a + 1, 5):
            rho, _ = spearmanr(
                subset[a],
                subset[b]
            )

            stability.append({
                "class_id": cls,
                "fold_a": a,
                "fold_b": b,
                "spearman_rho": float(rho)
            })

stability = pd.DataFrame(stability)

stability.to_csv(
    OUT / "multiclass_fold_rank_stability.csv",
    index=False
)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11
})

for cls in range(3):
    selected_summary = summary[
        summary.class_id == cls
    ].sort_values("mean_abs_shap")

    fig, ax = plt.subplots(figsize=(9, 6.6))

    ax.barh(
        selected_summary.feature,
        selected_summary.mean_abs_shap
    )

    ax.set_xlabel(
        "Survey-weighted mean |SHAP| (probability units)"
    )

    ax.set_title(
        f"Final multiclass probability - {class_names[cls]}"
    )

    fig.tight_layout()

    fig.savefig(
        OUT / f"class{cls}_global_shap.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        OUT / f"class{cls}_global_shap.svg",
        bbox_inches="tight"
    )

    plt.close(fig)

    eligible = np.flatnonzero(
        y[indices] == cls
    )

    if len(eligible):
        probs = saved_probs[
            indices[eligible], cls
        ]

        representative = eligible[
            int(
                np.argmin(
                    np.abs(
                        probs - np.median(probs)
                    )
                )
            )
        ]

        position = int(
            indices[representative]
        )

        original = master.iloc[
            position
        ][features].to_numpy(dtype=object)

        local = shap.Explanation(
            values=values[
                representative, :, cls
            ],
            base_values=float(
                bases[representative, cls]
            ),
            data=original,
            feature_names=features,
        )

        shap.plots.waterfall(
            local,
            max_display=12,
            show=False
        )

        fig = plt.gcf()
        fig.set_size_inches(11, 7.5)

        plt.title(
            f"Representative held-out example: {class_names[cls]}"
        )

        fig.savefig(
            OUT / f"class{cls}_representative_waterfall.png",
            dpi=300,
            bbox_inches="tight"
        )

        fig.savefig(
            OUT / f"class{cls}_representative_waterfall.svg",
            bbox_inches="tight"
        )

        plt.close(fig)

print("\nFINAL MULTICLASS SHAP: COMPLETE")

for cls in range(3):
    print(
        f"\n{class_names[cls]} - top 7 original predictors"
    )

    cols = [
        "feature",
        "mean_abs_shap",
        "mean_signed_shap"
    ]

    print(
        summary[
            summary.class_id == cls
        ].nlargest(
            7,
            "mean_abs_shap"
        )[cols].to_string(index=False)
    )

print("\nCross-fold rank stability:")

print(
    stability.groupby("class_id")
    .spearman_rho
    .agg(["mean", "min", "max"])
    .to_string()
)

print(
    "\nStage 1 vs class-0 SHAP max-difference by fold:",
    comparison_errors
)

print("\nSaved:", OUT)

FROZEN THREE-CLASS MODEL REPLAY: PASS | 5/5 folds
Fold 0: computed 160 held-out three-class explanations
  Additivity=8.88e-16 | Sum-to-one=1.11e-16 | Class-0 comparison=0.00e+00
Fold 1: computed 160 held-out three-class explanations
  Additivity=7.77e-16 | Sum-to-one=1.11e-16 | Class-0 comparison=0.00e+00
Fold 2: computed 160 held-out three-class explanations
  Additivity=7.77e-16 | Sum-to-one=1.11e-16 | Class-0 comparison=0.00e+00
Fold 3: computed 160 held-out three-class explanations
  Additivity=7.77e-16 | Sum-to-one=2.22e-16 | Class-0 comparison=0.00e+00
Fold 4: computed 160 held-out three-class explanations
  Additivity=8.88e-16 | Sum-to-one=0.00e+00 | Class-0 comparison=0.00e+00

FINAL MULTICLASS SHAP: COMPLETE

No ANC - top 7 original predictors
                          feature  mean_abs_shap  mean_signed_shap
                  wealth_quintile       0.027382          0.002580
                prior_live_births       0.019629          0.001932
            weekly_media_exposure  

In [15]:

from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
MASTER_FILE = ROOT / "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
STAGE_FILE = ROOT / "10_explainability/XAI_PRIMARY_SHAP/heldout_original_feature_shap.csv"
FINAL_FILE = ROOT / "10_explainability/XAI_FINAL_MULTICLASS_SHAP/heldout_multiclass_original_feature_shap.csv"
MODEL_FILE = ROOT / "10_explainability/XAI_SELECTED_REFITS/outer_0_stage_1.joblib"
OUTPUT = ROOT / "10_explainability/XAI_DIRECTIONAL_EFFECTS"

MIN_N = 15
MIN_ESS = 10
TOP_PLOTS = 6
ADD_TOL = 1e-7

for p in (MASTER_FILE, STAGE_FILE, FINAL_FILE, MODEL_FILE):
    if not p.is_file():
        raise FileNotFoundError(p)

bundle = joblib.load(MODEL_FILE)
numeric = list(bundle["numeric_predictors"])
categorical = list(bundle["categorical_predictors"])
features = numeric + categorical

if len(features) != 15 or len(set(features)) != 15:
    raise RuntimeError("Expected 15 distinct original predictors")

master = pd.read_csv(MASTER_FILE)
stage = pd.read_csv(STAGE_FILE)
final = pd.read_csv(FINAL_FILE)

def normalize_key(s):
    v = pd.to_numeric(s, errors="coerce")
    if v.notna().all() and np.allclose(
        v.to_numpy(float),
        np.round(v.to_numpy(float))
    ):
        return pd.Series(
            np.round(v.to_numpy(float)).astype(np.int64).astype(str),
            index=s.index
        )
    return s.astype(str).str.strip()

required_master = features + [
    "_ROW_ID", "ANC_CLASS", "SURVEY_WEIGHT"
]

if not set(required_master).issubset(master.columns):
    raise RuntimeError("Missing master columns")

master["__KEY"] = normalize_key(master["_ROW_ID"])

if master["__KEY"].duplicated().any():
    raise RuntimeError("Duplicate master participant IDs")

for frame in (stage, final):
    if "row_key" not in frame:
        raise RuntimeError("Saved SHAP table has no row_key")
    frame["__KEY"] = normalize_key(frame["row_key"])

if final["__KEY"].duplicated().any():
    raise RuntimeError("Duplicate multiclass explanation IDs")

if stage.duplicated(["__KEY", "stage"]).any():
    raise RuntimeError("Duplicate stage explanation IDs")

if set(stage["stage"].astype(int)) != {1, 2}:
    raise RuntimeError("Missing stage explanations")

base = master[
    ["__KEY", "ANC_CLASS", "SURVEY_WEIGHT", *features]
].copy()

for feature in numeric:
    base[feature] = pd.to_numeric(
        base[feature], errors="raise"
    )

    if not np.isfinite(base[feature].to_numpy(float)).all():
        raise RuntimeError(
            f"Invalid numeric predictor: {feature}"
        )

specs = [
    (
        "Stage1_NoANC",
        stage.loc[stage["stage"] == 1].copy(),
        "SHAP__",
        "base_probability",
        "predicted_probability",
        0,
        "fold"
    ),
    (
        "Stage2_GE8_conditional",
        stage.loc[stage["stage"] == 2].copy(),
        "SHAP__",
        "base_probability",
        "predicted_probability",
        2,
        "fold"
    ),
    (
        "Final_NoANC",
        final.copy(),
        "class0__",
        "class0_base",
        "class0_predicted",
        0,
        "outer_fold"
    ),
    (
        "Final_1to7",
        final.copy(),
        "class1__",
        "class1_base",
        "class1_predicted",
        1,
        "outer_fold"
    ),
    (
        "Final_GE8",
        final.copy(),
        "class2__",
        "class2_base",
        "class2_predicted",
        2,
        "outer_fold"
    ),
]

views = {}

for target, source, prefix, baseline, prediction, positive, fold_col in specs:

    mandatory = [
        "__KEY", "analysis_weight",
        "survey_weight", fold_col,
        baseline, prediction
    ] + [prefix + f for f in features]

    if not set(mandatory).issubset(source.columns):
        raise RuntimeError(
            f"Incomplete SHAP table: {target}"
        )

    if source["__KEY"].duplicated().any():
        raise RuntimeError(
            f"Duplicate explained participants: {target}"
        )

    source = source[mandatory].merge(
        base,
        on="__KEY",
        how="left",
        validate="one_to_one",
        indicator=True
    )

    if (source["_merge"] != "both").any():
        raise RuntimeError(
            f"Unmatched master participant: {target}"
        )

    source = source.drop(columns="_merge")

    if (
        target == "Stage2_GE8_conditional"
        and not (source["ANC_CLASS"].astype(int) != 0).all()
    ):
        raise RuntimeError(
            "Stage 2 explanation includes ANC nonusers"
        )

    if not np.allclose(
        source["survey_weight"],
        source["SURVEY_WEIGHT"],
        rtol=0,
        atol=1e-12
    ):
        raise RuntimeError(
            f"Survey weight mismatch: {target}"
        )

    aw = source["analysis_weight"].to_numpy(float)

    if not np.isfinite(aw).all() or (aw <= 0).any():
        raise RuntimeError(
            f"Invalid analysis weights: {target}"
        )

    shap_values = source[
        [prefix + f for f in features]
    ].to_numpy(float)

    if not np.isfinite(shap_values).all():
        raise RuntimeError(
            f"Nonfinite SHAP values: {target}"
        )

    difference = (
        source[prediction].to_numpy(float)
        - source[baseline].to_numpy(float)
    )

    if np.max(
        np.abs(shap_values.sum(axis=1) - difference)
    ) > ADD_TOL:
        raise RuntimeError(
            f"SHAP additivity failure: {target}"
        )

    source.attrs.update({
        "prefix": prefix,
        "prediction": prediction,
        "positive": positive,
        "fold_col": fold_col
    })

    views[target] = source

s1 = views["Stage1_NoANC"].set_index("__KEY")
f0 = views["Final_NoANC"].set_index("__KEY")

if set(s1.index) != set(f0.index):
    raise RuntimeError(
        "Stage 1 and final class-0 samples differ"
    )

max_class0_difference = max(
    float(
        np.max(
            np.abs(
                s1.loc[
                    f0.index, "SHAP__" + f
                ].to_numpy(float)
                - f0["class0__" + f].to_numpy(float)
            )
        )
    )
    for f in features
)

if max_class0_difference > ADD_TOL:
    raise RuntimeError(
        "Stage 1 and class-0 attributions differ"
    )

print(
    "INPUT QC: PASS | Five explanation targets | "
    "15 original predictors"
)

print(
    "STAGE 1 vs FINAL CLASS 0:",
    f"{max_class0_difference:.2e}"
)

numeric_bins = {}

for f in numeric:
    full_values = base[f].to_numpy(float)
    distinct = np.unique(full_values)

    if len(distinct) <= 12:
        numeric_bins[f] = ("exact", distinct)

    else:
        edges = np.unique(
            np.quantile(
                full_values,
                np.linspace(0, 1, 6)
            )[1:-1]
        )

        numeric_bins[f] = ("quintile", edges)

def groups_for(source, feature):
    if feature in categorical:
        labels = source[feature].astype(str)
        order = sorted(labels.unique().tolist())

        return labels, order, "categorical"

    values = source[feature].to_numpy(float)
    method, rule = numeric_bins[feature]

    if method == "exact":
        labels = pd.Series(
            [f"{v:g}" for v in values],
            index=source.index
        )

        return (
            labels,
            [f"{v:g}" for v in rule],
            "numeric_exact"
        )

    bins = [-np.inf, *rule.tolist(), np.inf]

    grouped = pd.cut(
        values,
        bins=bins,
        include_lowest=True
    )

    labels = pd.Series(
        grouped.astype(str),
        index=source.index
    )

    return (
        labels,
        [str(c) for c in grouped.categories],
        "numeric_quantiles"
    )

def weighted_mean(arr, w):
    return float(
        np.average(
            np.asarray(arr, dtype=float),
            weights=np.asarray(w, dtype=float)
        )
    )

def kish_ess(w):
    w = np.asarray(w, dtype=float)
    return float(w.sum() ** 2 / np.square(w).sum())

feature_results = []
level_results = []
numeric_rows = []

for target, data in views.items():

    prefix = data.attrs["prefix"]
    prediction = data.attrs["prediction"]
    positive = data.attrs["positive"]

    total_weight = float(
        data["analysis_weight"].sum()
    )

    for feature in features:

        shap_col = prefix + feature

        shap_values = data[
            shap_col
        ].to_numpy(float)

        aw = data[
            "analysis_weight"
        ].to_numpy(float)

        feature_results.append({
            "target": target,
            "feature": feature,
            "n": len(data),
            "weighted_mean_abs_shap": weighted_mean(
                np.abs(shap_values), aw
            ),
            "weighted_mean_signed_shap": weighted_mean(
                shap_values, aw
            ),
            "weighted_positive_shap_fraction": weighted_mean(
                (shap_values > 0).astype(float), aw
            ),
        })

        labels, order, group_type = groups_for(
            data, feature
        )

        for level in order:

            selected = (
                labels.to_numpy() == level
            )

            if not selected.any():
                continue

            local = data.loc[selected]

            local_weights = local[
                "analysis_weight"
            ].to_numpy(float)

            n = len(local)
            ess = kish_ess(local_weights)

            record = {
                "target": target,
                "feature": feature,
                "group_type": group_type,
                "level": str(level),
                "n": n,
                "kish_ess": ess,
                "weighted_sample_share": float(
                    local_weights.sum() / total_weight
                ),
                "weighted_mean_signed_shap": weighted_mean(
                    local[shap_col], local_weights
                ),
                "weighted_mean_abs_shap": weighted_mean(
                    np.abs(local[shap_col]), local_weights
                ),
                "weighted_mean_predicted_probability": weighted_mean(
                    local[prediction], local_weights
                ),
                "weighted_observed_rate": weighted_mean(
                    (
                        local["ANC_CLASS"].to_numpy(int)
                        == positive
                    ).astype(float),
                    local_weights
                ),
                "limited_support": bool(
                    n < MIN_N or ess < MIN_ESS
                ),
                "weighted_mean_numeric_value": (
                    weighted_mean(
                        local[feature], local_weights
                    )
                    if feature in numeric
                    else np.nan
                ),
            }

            level_results.append(record)

        if feature in numeric:

            subset = data[
                [
                    "__KEY",
                    "analysis_weight",
                    feature,
                    shap_col
                ]
            ].copy()

            subset["target"] = target
            subset["feature"] = feature

            subset = subset.rename(columns={
                feature: "original_value",
                shap_col: "shap_value"
            })

            numeric_rows.append(subset)

feature_summary = pd.DataFrame(feature_results)
levels_summary = pd.DataFrame(level_results)
numeric_points = pd.concat(
    numeric_rows,
    ignore_index=True
)

if len(feature_summary) != len(views) * len(features):
    raise RuntimeError(
        "Incomplete original-feature analysis"
    )

ranks = []

for first, second in [
    (
        "Stage1_NoANC",
        "Stage2_GE8_conditional"
    ),
    (
        "Stage1_NoANC",
        "Final_GE8"
    ),
    (
        "Stage2_GE8_conditional",
        "Final_GE8"
    ),
]:

    a = feature_summary.loc[
        feature_summary.target == first
    ].set_index("feature")[
        "weighted_mean_abs_shap"
    ].reindex(features)

    b = feature_summary.loc[
        feature_summary.target == second
    ].set_index("feature")[
        "weighted_mean_abs_shap"
    ].reindex(features)

    rho, _ = spearmanr(
        a.to_numpy(float),
        b.to_numpy(float)
    )

    ranks.append({
        "comparison": f"{first} vs {second}",
        "rank_spearman": float(rho),
        "top5_overlap": len(
            set(a.nlargest(5).index)
            & set(b.nlargest(5).index)
        )
    })

rank_table = pd.DataFrame(ranks)

OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

feature_summary.to_csv(
    OUTPUT / "directionality_feature_summary.csv",
    index=False
)

levels_summary.to_csv(
    OUTPUT / "directionality_category_and_numeric_group_summary.csv",
    index=False
)

numeric_points.to_csv(
    OUTPUT / "numeric_shap_dependence_points.csv",
    index=False
)

rank_table.to_csv(
    OUTPUT / "cross_target_shap_rank_comparison.csv",
    index=False
)

plot_targets = [
    "Stage1_NoANC",
    "Stage2_GE8_conditional",
    "Final_1to7",
    "Final_GE8"
]

for target in plot_targets:

    best = feature_summary.loc[
        feature_summary.target == target
    ].nlargest(
        TOP_PLOTS,
        "weighted_mean_abs_shap"
    )

    for feature in best["feature"]:

        part = levels_summary[
            (levels_summary.target == target)
            & (levels_summary.feature == feature)
        ].copy()

        part = part.loc[
            ~part.limited_support
        ]

        if len(part) < 2:
            continue

        fig, ax = plt.subplots(
            figsize=(
                9,
                max(3.8, 0.48 * len(part) + 1.8)
            )
        )

        if feature in numeric:

            part = part.sort_values(
                "weighted_mean_numeric_value"
            )

            original = numeric_points[
                (numeric_points.target == target)
                & (numeric_points.feature == feature)
            ]

            if len(original):
                ax.scatter(
                    original["original_value"],
                    original["shap_value"],
                    s=9,
                    alpha=0.18
                )

            ax.plot(
                part["weighted_mean_numeric_value"],
                part["weighted_mean_signed_shap"],
                marker="o",
                linewidth=1.5
            )

            ax.set_xlabel(feature)
            ax.set_ylabel(
                "SHAP contribution to predicted probability"
            )
            ax.axhline(0, linewidth=0.8)

        else:
            part = part.sort_values(
                "weighted_mean_signed_shap"
            )

            ypos = np.arange(len(part))

            ax.barh(
                ypos,
                part["weighted_mean_signed_shap"]
            )

            ax.set_yticks(ypos)

            ax.set_yticklabels([
                f"{level} (n={n})"
                for level, n in zip(
                    part.level,
                    part.n
                )
            ])

            ax.axvline(0, linewidth=0.8)

            ax.set_xlabel(
                "Mean signed SHAP (predicted probability units)"
            )

        ax.set_title(
            f"{target} | {feature}"
        )

        fig.tight_layout()

        fig.savefig(
            OUTPUT / f"{target}__{feature}.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(fig)

print("\nDIRECTIONAL SHAP ANALYSIS: COMPLETE")

for target in plot_targets:

    best = feature_summary.loc[
        feature_summary.target == target
    ].nlargest(
        5,
        "weighted_mean_abs_shap"
    )

    print(
        f"\n{target} | "
        f"n={int(best['n'].iloc[0])}"
    )

    print(
        best[
            [
                "feature",
                "weighted_mean_abs_shap",
                "weighted_mean_signed_shap",
                "weighted_positive_shap_fraction"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.5f}"
        )
    )

print("\nCROSS-TARGET RANK COMPARISONS")

print(
    rank_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print(
    "\nGroups with limited support:",
    int(levels_summary["limited_support"].sum()),
    "/",
    len(levels_summary)
)

print("\nSaved:", OUTPUT)

INPUT QC: PASS | Five explanation targets | 15 original predictors
STAGE 1 vs FINAL CLASS 0: 0.00e+00

DIRECTIONAL SHAP ANALYSIS: COMPLETE

Stage1_NoANC | n=800
                        feature  weighted_mean_abs_shap  weighted_mean_signed_shap  weighted_positive_shap_fraction
                wealth_quintile                 0.02738                    0.00258                          0.50615
              prior_live_births                 0.01963                    0.00193                          0.36513
          weekly_media_exposure                 0.01610                    0.00181                          0.56399
             maternal_education                 0.01479                    0.00059                          0.29186
partner_education_primary_final                 0.00992                    0.00081                          0.42544

Stage2_GE8_conditional | n=800
                          feature  weighted_mean_abs_shap  weighted_mean_signed_shap  weighted_positive_shap_fr

In [16]:

from pathlib import Path
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn
from scipy.special import expit
from scipy.stats import spearmanr

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")
REFITS = ROOT / "10_explainability" / "XAI_SELECTED_REFITS"
PRIOR = ROOT / "10_explainability" / "XAI_PRIMARY_SHAP"
OUT = ROOT / "10_explainability" / "XAI_SHAP_SENSITIVITY"

SEED = 2026
EVAL_PER_FOLD = 80
BASE_BACKGROUND = 32
ALT_BACKGROUND = 64
N_PERMUTATIONS = 2
TOL = 1e-10

master_path = ROOT / "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
outer_path = ROOT / "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv"
oof_path = ROOT / "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
manifest_path = REFITS / "reconstruction_manifest.json"

model_paths = {
    (fold, stage): REFITS / f"outer_{fold}_stage_{stage}.joblib"
    for fold in range(5)
    for stage in (1, 2)
}

prior_paths = {
    (fold, stage): PRIOR / f"fold_{fold}_stage_{stage}.npz"
    for fold in range(5)
    for stage in (1, 2)
}

required = [
    master_path,
    outer_path,
    oof_path,
    manifest_path,
    *model_paths.values(),
    *prior_paths.values()
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

if (
    manifest.get("status") != "PASS"
    or int(manifest.get("number_of_models", 0)) != 10
):
    raise RuntimeError("Model reconstruction has not passed QC")

for label, path in [
    ("master", master_path),
    ("outer", outer_path),
    ("oof", oof_path)
]:
    expected = manifest.get(
        "original_data_sha256", {}
    ).get(label)

    if expected and sha256(path) != expected:
        raise RuntimeError(
            f"Frozen input integrity failure: {label}"
        )

master = pd.read_csv(master_path)
outer = pd.read_csv(outer_path)
oof = pd.read_csv(oof_path)

def normalize_key(series):
    numerical = pd.to_numeric(
        series, errors="coerce"
    )

    if numerical.notna().all() and np.allclose(
        numerical,
        np.round(numerical)
    ):
        return pd.Series(
            np.round(
                numerical.to_numpy(float)
            ).astype(np.int64),
            index=series.index
        )

    return series.astype(str).str.strip()

for table, label in [
    (master, "_ROW_ID"),
    (outer, "_ROW_ID"),
    (oof, "_ROW_KEY")
]:
    table["__KEY"] = normalize_key(table[label])

    if table["__KEY"].duplicated().any():
        raise RuntimeError(
            f"Duplicate identifiers in {label}"
        )

if (
    len(master) != 5063
    or not (
        set(master.__KEY)
        == set(outer.__KEY)
        == set(oof.__KEY)
    )
):
    raise RuntimeError(
        "Frozen cohort identifiers disagree"
    )

master["__FOLD"] = master["__KEY"].map(
    outer.set_index("__KEY")["OUTER_FOLD"]
)

if master["__FOLD"].isna().any():
    raise RuntimeError(
        "Missing frozen CV assignments"
    )

master["__FOLD"] = master["__FOLD"].astype(int)

if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError(
        "Expected five outer folds numbered 0-4"
    )

oof = (
    oof.set_index("__KEY")
    .loc[master["__KEY"]]
    .reset_index(drop=True)
)

labels = master["ANC_CLASS"].to_numpy(int)
weights = master["SURVEY_WEIGHT"].to_numpy(float)

if not np.array_equal(
    labels,
    oof["ANC_CLASS_TRUE"].to_numpy(int)
):
    raise RuntimeError(
        "OOF outcome alignment failure"
    )

if not np.array_equal(
    master["__FOLD"],
    oof["Outer_Fold"].to_numpy(int)
):
    raise RuntimeError(
        "OOF fold alignment failure"
    )

if not np.allclose(
    weights,
    oof["SURVEY_WEIGHT"].to_numpy(float),
    rtol=0,
    atol=1e-12
):
    raise RuntimeError(
        "Survey weight alignment failure"
    )

if (
    not np.isfinite(weights).all()
    or (weights <= 0).any()
):
    raise RuntimeError(
        "Invalid survey weights"
    )

models = {
    key: joblib.load(path)
    for key, path in model_paths.items()
}

numeric = list(
    models[0, 1]["numeric_predictors"]
)

categorical = list(
    models[0, 1]["categorical_predictors"]
)

features = numeric + categorical

if (
    len(features) != 15
    or not set(features).issubset(master.columns)
):
    raise RuntimeError(
        "Original predictor specification mismatch"
    )

for (fold, stage), bundle in models.items():
    if (
        int(bundle["outer_fold"]) != fold
        or bundle["stage"] != f"Stage {stage}"
        or (
            list(bundle["numeric_predictors"])
            + list(bundle["categorical_predictors"])
        ) != features
    ):
        raise RuntimeError(
            "Serialized model metadata mismatch"
        )

levels = {
    feature: np.sort(
        master[feature].astype(str).unique()
    )
    for feature in categorical
}

X = np.zeros(
    (len(master), len(features)),
    dtype=float
)

for j, feature in enumerate(features):
    if feature in numeric:
        X[:, j] = master[
            feature
        ].astype(float).to_numpy()

    else:
        X[:, j] = pd.Categorical(
            master[feature].astype(str),
            categories=levels[feature]
        ).codes

if not np.isfinite(X).all():
    raise RuntimeError(
        "Nonfinite original-feature values"
    )

def decode(array):
    array = np.asarray(
        array, dtype=float
    )

    if array.ndim == 1:
        array = array[None, :]

    if (
        array.ndim != 2
        or array.shape[1] != len(features)
    ):
        raise RuntimeError(
            "Incorrect feature dimensions"
        )

    data = pd.DataFrame(
        array, columns=features
    )

    for feature in categorical:
        codes = np.rint(
            data[feature].to_numpy(float)
        ).astype(int)

        if (
            not np.allclose(
                data[feature].to_numpy(float),
                codes,
                rtol=0,
                atol=1e-8
            )
            or (codes < 0).any()
            or (
                codes >= len(levels[feature])
            ).any()
        ):
            raise RuntimeError(
                f"Invalid masked category: {feature}"
            )

        data[feature] = levels[feature][codes]

    return data

def predict(bundle, array):
    data = decode(array)

    xn = data[
        bundle["numeric_predictors"]
    ].astype(float).to_numpy()

    if bundle["preprocessing"] == "Scaled OHE":
        xn = bundle["scaler"].transform(xn)

    elif bundle["preprocessing"] != "Unscaled OHE":
        raise RuntimeError(
            "Unknown feature representation"
        )

    xc = bundle["encoder"].transform(
        data[
            bundle["categorical_predictors"]
        ].astype(str)
    )

    cls = np.asarray(
        bundle["estimator"].classes_
    )

    positive = np.flatnonzero(
        cls == 1
    )

    if len(positive) != 1:
        raise RuntimeError(
            "Invalid binary class representation"
        )

    raw = bundle[
        "estimator"
    ].predict_proba(
        np.hstack([xn, xc])
    )[:, int(positive[0])]

    raw = np.clip(
        raw, 1e-6, 1 - 1e-6
    )

    return expit(
        bundle["calibration_alpha"]
        + bundle["calibration_beta"]
        * np.log(raw / (1 - raw))
    )

for fold in range(5):
    ix = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    for stage, col in [
        (1, "q0_CALIBRATED"),
        (2, "q2_CALIBRATED_CONDITIONAL")
    ]:
        predicted = predict(
            models[fold, stage],
            X[ix]
        )

        reference = oof[
            col
        ].to_numpy(float)[ix]

        error = np.max(
            np.abs(predicted - reference)
        )

        if error > TOL:
            raise RuntimeError(
                f"OOF replay failed: "
                f"fold={fold}, stage={stage}, "
                f"error={error}"
            )

print(
    "FROZEN MODEL REPLAY: PASS | 10/10"
)

scenarios = [
    "original_cached",
    "reference_32",
    "permutation_seed",
    "alternate_background_32",
    "alternate_background_64"
]

spec = {
    "method": (
        "Paired held-out original-feature "
        "probability SHAP sensitivity"
    ),
    "seed": SEED,
    "eval_per_fold": EVAL_PER_FOLD,
    "original_background_size": BASE_BACKGROUND,
    "alternative_background_size": ALT_BACKGROUND,
    "permutations_per_recalculation": N_PERMUTATIONS,
    "scenarios": scenarios,
    "features": features,
    "versions": {
        "shap": shap.__version__,
        "sklearn": sklearn.__version__
    },
    "sha256": {
        str(p.relative_to(ROOT)): sha256(p)
        for p in [
            master_path,
            outer_path,
            oof_path,
            *model_paths.values(),
            *prior_paths.values()
        ]
    }
}

OUT.mkdir(
    parents=True,
    exist_ok=True
)

spec_file = OUT / "sensitivity_run_spec.json"

if spec_file.is_file():
    if json.loads(
        spec_file.read_text(encoding="utf-8")
    ) != spec:
        raise RuntimeError(
            "Existing sensitivity results use "
            "different settings"
        )

else:
    if list(
        OUT.glob("fold_*_stage_*_*.npz")
    ):
        raise RuntimeError(
            "Sensitivity checkpoints exist "
            "without a matching specification"
        )

    spec_file.write_text(
        json.dumps(spec, indent=2),
        encoding="utf-8"
    )

importance_rows = []
difference_rows = []

for fold in range(5):

    all_train = np.flatnonzero(
        master["__FOLD"].to_numpy() != fold
    )

    all_test = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    for stage in (1, 2):

        train = (
            all_train
            if stage == 1
            else all_train[
                labels[all_train] != 0
            ]
        )

        test = (
            all_test
            if stage == 1
            else all_test[
                labels[all_test] != 0
            ]
        )

        with np.load(
            prior_paths[fold, stage],
            allow_pickle=False
        ) as previous:

            original_selected = previous[
                "selected"
            ].astype(int)

            original_background = previous[
                "background_idx"
            ].astype(int)

            original_values = previous[
                "values"
            ].astype(float)

            original_bases = previous[
                "base_values"
            ].astype(float).reshape(-1)

        if (
            len(original_background) != BASE_BACKGROUND
            or len(
                np.unique(original_selected)
            ) != len(original_selected)
            or not np.isin(
                original_selected, test
            ).all()
            or not np.isin(
                original_background, train
            ).all()
            or original_values.shape != (
                len(original_selected),
                len(features)
            )
            or original_bases.shape != (
                len(original_selected),
            )
        ):
            raise RuntimeError(
                f"Incompatible Stage SHAP checkpoint: "
                f"{fold}/{stage}"
            )

        if len(original_selected) < EVAL_PER_FOLD:
            raise RuntimeError(
                f"Too few saved SHAP rows: {fold}/{stage}"
            )

        sample_rng = np.random.default_rng(
            np.random.SeedSequence([
                SEED, 550, fold, stage
            ])
        )

        selected = np.sort(
            sample_rng.choice(
                original_selected,
                EVAL_PER_FOLD,
                replace=False
            )
        )

        old_position = pd.Index(
            original_selected
        ).get_indexer(selected)

        if (old_position < 0).any():
            raise RuntimeError(
                "Reference SHAP selection failed"
            )

        cached_values = original_values[
            old_position
        ]

        cached_bases = original_bases[
            old_position
        ]

        row_weights = (
            weights[selected]
            * len(test)
            / len(selected)
        )

        if (
            not np.isfinite(row_weights).all()
            or (row_weights <= 0).any()
        ):
            raise RuntimeError(
                "Invalid analysis weights"
            )

        rng_background = np.random.default_rng(
            np.random.SeedSequence([
                SEED, 660, fold, stage
            ])
        )

        probabilities = (
            weights[train]
            / weights[train].sum()
        )

        alternate_64 = rng_background.choice(
            train,
            size=ALT_BACKGROUND,
            replace=True,
            p=probabilities
        )

        alternate_32 = alternate_64[
            :BASE_BACKGROUND
        ]

        seed_original = (
            SEED + 100 * fold + stage
        )

        seed_alternative = (
            SEED + 800000 + 100 * fold + stage
        )

        configurations = {
            "original_cached": (
                original_background,
                seed_original
            ),
            "reference_32": (
                original_background,
                seed_original
            ),
            "permutation_seed": (
                original_background,
                seed_alternative
            ),
            "alternate_background_32": (
                alternate_32,
                seed_original
            ),
            "alternate_background_64": (
                alternate_64,
                seed_original
            )
        }

        target = predict(
            models[fold, stage],
            X[selected]
        )

        reference_error = float(
            np.max(
                np.abs(
                    cached_bases
                    + cached_values.sum(axis=1)
                    - target
                )
            )
        )

        if reference_error > 1e-7:
            raise RuntimeError(
                f"Saved SHAP additivity failed: "
                f"{fold}/{stage}"
            )

        scenario_values = {
            "original_cached": cached_values
        }

        scenario_bases = {
            "original_cached": cached_bases
        }

        for scenario, (
            background,
            explainer_seed
        ) in configurations.items():

            if scenario == "original_cached":
                continue

            checkpoint = (
                OUT
                / f"fold_{fold}_stage_{stage}_{scenario}.npz"
            )

            if checkpoint.is_file():

                with np.load(
                    checkpoint,
                    allow_pickle=False
                ) as previous:

                    values = previous[
                        "values"
                    ].astype(float)

                    bases = previous[
                        "bases"
                    ].astype(float).reshape(-1)

                    saved_selected = previous[
                        "selected"
                    ].astype(int)

                    saved_background = previous[
                        "background"
                    ].astype(int)

                if (
                    not np.array_equal(
                        saved_selected,
                        selected
                    )
                    or not np.array_equal(
                        saved_background,
                        background
                    )
                ):
                    raise RuntimeError(
                        f"Checkpoint design mismatch: "
                        f"{checkpoint}"
                    )

            else:

                explainer = shap.Explainer(
                    lambda z, bundle=models[
                        fold, stage
                    ]: predict(bundle, z),

                    shap.maskers.Independent(
                        X[background],
                        max_samples=len(background)
                    ),

                    algorithm="permutation",
                    feature_names=features,
                    seed=explainer_seed
                )

                explanation = explainer(
                    X[selected],
                    max_evals=(
                        N_PERMUTATIONS
                        * (2 * len(features) + 1)
                    ),
                    silent=True
                )

                values = np.asarray(
                    explanation.values,
                    dtype=float
                )

                bases = np.asarray(
                    explanation.base_values,
                    dtype=float
                ).reshape(-1)

                if (
                    values.shape != (
                        len(selected),
                        len(features)
                    )
                    or bases.shape != (
                        len(selected),
                    )
                ):
                    raise RuntimeError(
                        f"SHAP dimension mismatch: "
                        f"{fold}/{stage}/{scenario}"
                    )

                if (
                    not np.isfinite(values).all()
                    or not np.isfinite(bases).all()
                ):
                    raise RuntimeError(
                        "Nonfinite SHAP values"
                    )

                error = float(
                    np.max(
                        np.abs(
                            bases
                            + values.sum(axis=1)
                            - target
                        )
                    )
                )

                if error > 1e-7:
                    raise RuntimeError(
                        f"SHAP additivity failed: "
                        f"{fold}/{stage}/{scenario}"
                    )

                temporary = checkpoint.with_suffix(
                    ".npz.part"
                )

                with temporary.open(
                    "wb"
                ) as handle:

                    np.savez_compressed(
                        handle,
                        values=values,
                        bases=bases,
                        selected=selected,
                        background=background
                    )

                temporary.replace(checkpoint)

            if (
                values.shape != (
                    len(selected),
                    len(features)
                )
                or not np.isfinite(values).all()
            ):
                raise RuntimeError(
                    f"Invalid cached SHAP values: "
                    f"{fold}/{stage}/{scenario}"
                )

            error = float(
                np.max(
                    np.abs(
                        bases
                        + values.sum(axis=1)
                        - target
                    )
                )
            )

            if error > 1e-7:
                raise RuntimeError(
                    f"Saved SHAP additivity failed: "
                    f"{fold}/{stage}/{scenario}"
                )

            scenario_values[scenario] = values
            scenario_bases[scenario] = bases

        for scenario in scenarios:

            values = scenario_values[
                scenario
            ]

            bases = scenario_bases[
                scenario
            ]

            reference = scenario_values[
                "reference_32"
            ]

            for j, feature in enumerate(features):

                importance_rows.append({
                    "fold": fold,
                    "stage": stage,
                    "scenario": scenario,
                    "feature": feature,
                    "n_eval": len(selected),
                    "weighted_sum": float(
                        row_weights.sum()
                    ),
                    "weighted_abs_shap_sum": float(
                        np.sum(
                            row_weights
                            * np.abs(values[:, j])
                        )
                    ),
                    "weighted_signed_shap_sum": float(
                        np.sum(
                            row_weights
                            * values[:, j]
                        )
                    )
                })

            difference_rows.append({
                "fold": fold,
                "stage": stage,
                "scenario": scenario,
                "n_eval": len(selected),
                "weighted_mean_abs_shap_difference": float(
                    np.average(
                        np.abs(
                            values - reference
                        ).mean(axis=1),
                        weights=row_weights
                    )
                ),
                "weighted_mean_abs_base_difference": float(
                    np.average(
                        np.abs(
                            bases
                            - scenario_bases[
                                "reference_32"
                            ]
                        ),
                        weights=row_weights
                    )
                )
            })

        print(
            f"Fold {fold}, Stage {stage}: "
            f"{len(selected)} paired observations, "
            f"5 scenarios PASS"
        )

fold_results = pd.DataFrame(
    importance_rows
)

case_differences = pd.DataFrame(
    difference_rows
)

grouped = fold_results.groupby(
    ["stage", "scenario", "feature"],
    as_index=False
).agg(
    weighted_sum=(
        "weighted_sum", "sum"
    ),
    weighted_abs_shap_sum=(
        "weighted_abs_shap_sum", "sum"
    ),
    weighted_signed_shap_sum=(
        "weighted_signed_shap_sum", "sum"
    )
)

grouped["mean_abs_shap"] = (
    grouped["weighted_abs_shap_sum"]
    / grouped["weighted_sum"]
)

grouped["mean_signed_shap"] = (
    grouped["weighted_signed_shap_sum"]
    / grouped["weighted_sum"]
)

comparisons = []

for stage in (1, 2):

    reference = grouped.loc[
        (grouped["stage"] == stage)
        & (
            grouped["scenario"]
            == "reference_32"
        )
    ].set_index(
        "feature"
    )["mean_abs_shap"].reindex(features)

    for scenario in scenarios:

        current = grouped.loc[
            (grouped["stage"] == stage)
            & (
                grouped["scenario"]
                == scenario
            )
        ].set_index(
            "feature"
        )["mean_abs_shap"].reindex(features)

        rho, _ = spearmanr(
            reference.to_numpy(float),
            current.to_numpy(float)
        )

        if not np.isfinite(rho):
            raise RuntimeError(
                f"Rank correlation undefined: "
                f"{stage}/{scenario}"
            )

        comparisons.append({
            "stage": stage,
            "scenario": scenario,
            "rank_spearman_vs_reference32": float(
                rho
            ),
            "top5_overlap": len(
                set(
                    reference.nlargest(5).index
                )
                & set(
                    current.nlargest(5).index
                )
            ),
            "relative_total_importance_change": float(
                np.abs(
                    current.to_numpy(float)
                    - reference.to_numpy(float)
                ).sum()
                / max(
                    reference.to_numpy(float).sum(),
                    1e-12
                )
            ),
            "mean_abs_shap_change": float(
                case_differences.loc[
                    (
                        case_differences["stage"]
                        == stage
                    )
                    & (
                        case_differences["scenario"]
                        == scenario
                    ),
                    "weighted_mean_abs_shap_difference"
                ].mean()
            )
        })

comparison_table = pd.DataFrame(
    comparisons
)

fold_results.to_csv(
    OUT / "sensitivity_fold_importance.csv",
    index=False
)

grouped.to_csv(
    OUT / "sensitivity_global_importance.csv",
    index=False
)

case_differences.to_csv(
    OUT / "sensitivity_fold_attribution_differences.csv",
    index=False
)

comparison_table.to_csv(
    OUT / "sensitivity_rank_agreement.csv",
    index=False
)

for stage in (1, 2):

    reference = grouped.loc[
        (grouped["stage"] == stage)
        & (
            grouped["scenario"]
            == "reference_32"
        )
    ].nlargest(
        10, "mean_abs_shap"
    )

    ordered = reference[
        "feature"
    ].tolist()[::-1]

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    for i, scenario in enumerate(scenarios):

        subset = grouped.loc[
            (grouped["stage"] == stage)
            & (
                grouped["scenario"]
                == scenario
            )
        ].set_index(
            "feature"
        ).loc[ordered]

        ax.plot(
            subset[
                "mean_abs_shap"
            ].to_numpy(float),
            np.arange(len(ordered))
            + (i - 2) * 0.14,
            "o",
            markersize=5,
            label=scenario
        )

    ax.set_yticks(
        range(len(ordered)),
        ordered
    )

    ax.set_xlabel(
        "Survey-weighted mean absolute SHAP "
        "(probability units)"
    )

    ax.set_title(
        f"Stage {stage}: "
        "SHAP background and permutation sensitivity"
    )

    ax.legend(
        fontsize=8,
        loc="lower right"
    )

    fig.tight_layout()

    fig.savefig(
        OUT / f"stage_{stage}_shap_sensitivity.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        OUT / f"stage_{stage}_shap_sensitivity.svg",
        bbox_inches="tight"
    )

    plt.close(fig)

print(
    "\nSHAP SENSITIVITY ANALYSIS: COMPLETE"
)

print(
    "\nRanking agreement against paired "
    "32-background reference:"
)

print(
    comparison_table[
        [
            "stage",
            "scenario",
            "rank_spearman_vs_reference32",
            "top5_overlap",
            "relative_total_importance_change"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    "\nReference top-five features by stage:"
)

for stage in (1, 2):

    reference = grouped.loc[
        (grouped["stage"] == stage)
        & (
            grouped["scenario"]
            == "reference_32"
        )
    ].nlargest(
        5, "mean_abs_shap"
    )

    print(
        f"Stage {stage}:",
        ", ".join(reference["feature"])
    )

print("\nSaved:", OUT)

FROZEN MODEL REPLAY: PASS | 10/10
Fold 0, Stage 1: 80 paired observations, 5 scenarios PASS
Fold 0, Stage 2: 80 paired observations, 5 scenarios PASS
Fold 1, Stage 1: 80 paired observations, 5 scenarios PASS
Fold 1, Stage 2: 80 paired observations, 5 scenarios PASS
Fold 2, Stage 1: 80 paired observations, 5 scenarios PASS
Fold 2, Stage 2: 80 paired observations, 5 scenarios PASS
Fold 3, Stage 1: 80 paired observations, 5 scenarios PASS
Fold 3, Stage 2: 80 paired observations, 5 scenarios PASS
Fold 4, Stage 1: 80 paired observations, 5 scenarios PASS
Fold 4, Stage 2: 80 paired observations, 5 scenarios PASS

SHAP SENSITIVITY ANALYSIS: COMPLETE

Ranking agreement against paired 32-background reference:
 stage                scenario  rank_spearman_vs_reference32  top5_overlap  relative_total_importance_change
     1         original_cached                        1.0000             5                            0.0030
     1            reference_32                        1.0000            

In [17]:

from pathlib import Path
from itertools import product
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import spearmanr

ROOT_CANDIDATES = [
    Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026"),
    Path(r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"),
]

ROOT = next(
    (p for p in ROOT_CANDIDATES if p.is_dir()),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Connect the external drive containing the ANC project"
    )

REFITS = ROOT / "10_explainability" / "XAI_SELECTED_REFITS"
OUT = ROOT / "10_explainability" / "XAI_PAIRWISE_INTERACTIONS"
PARTS = OUT / "checkpoints"

SEED = 2026
N_BACKGROUND = 48
MAX_CATEGORIES = 8
MIN_CELL_N = 5
MIN_CELL_ESS = 3.0
TOL = 1e-10

pairs = [
    ("wealth_quintile", "maternal_education"),
    ("wealth_quintile", "division"),
    ("wealth_quintile", "prior_live_births"),
    ("wealth_quintile", "maternal_age_at_index_birth_years"),
    ("wealth_quintile", "weekly_media_exposure"),
    ("maternal_education", "division"),
    ("maternal_education", "maternal_age_at_index_birth_years"),
    ("maternal_age_at_index_birth_years", "prior_live_births"),
    ("division", "residence"),
    ("maternal_education", "partner_education_primary_final"),
    ("weekly_media_exposure", "maternal_education"),
    ("prior_live_births", "pregnancy_intention"),
]

master_path = ROOT / "01_processed_data" / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
outer_path = ROOT / "04_cv_splits" / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
oof_path = ROOT / "06_predictions" / "STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
manifest_path = REFITS / "reconstruction_manifest.json"

model_paths = {
    (f, s): REFITS / f"outer_{f}_stage_{s}.joblib"
    for f in range(5)
    for s in (1, 2)
}

for p in [
    master_path, outer_path, oof_path,
    manifest_path, *model_paths.values()
]:
    if not p.is_file():
        raise FileNotFoundError(p)

def sha256(path):
    h = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1 << 20),
            b""
        ):
            h.update(block)

    return h.hexdigest()

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

if (
    manifest.get("status") != "PASS"
    or int(manifest.get("number_of_models", 0)) != 10
):
    raise RuntimeError(
        "Selected-model reconstruction QC is not PASS"
    )

for k, path in [
    ("master", master_path),
    ("outer", outer_path),
    ("oof", oof_path)
]:
    h = manifest.get(
        "original_data_sha256", {}
    ).get(k)

    if h is not None and sha256(path) != h:
        raise RuntimeError(f"Frozen input changed: {k}")

master = pd.read_csv(master_path)
outer = pd.read_csv(outer_path)
oof = pd.read_csv(oof_path)

def normalize_ids(x):
    z = pd.to_numeric(x, errors="coerce")

    if z.notna().all() and np.allclose(
        z.to_numpy(float),
        np.round(z.to_numpy(float))
    ):
        return pd.Series(
            np.round(
                z.to_numpy(float)
            ).astype(np.int64),
            index=x.index
        )

    return x.astype(str).str.strip()

for frame, col in [
    (master, "_ROW_ID"),
    (outer, "_ROW_ID"),
    (oof, "_ROW_KEY")
]:
    frame["__KEY"] = normalize_ids(frame[col])

    if frame["__KEY"].duplicated().any():
        raise RuntimeError(f"Duplicate IDs: {col}")

if len(master) != 5063 or not (
    set(master.__KEY)
    == set(outer.__KEY)
    == set(oof.__KEY)
):
    raise RuntimeError(
        "Frozen sample alignment failure"
    )

master["__FOLD"] = master["__KEY"].map(
    outer.set_index("__KEY")["OUTER_FOLD"]
)

if master["__FOLD"].isna().any():
    raise RuntimeError("Outer folds missing")

master["__FOLD"] = master["__FOLD"].astype(int)

if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError(
        "Expected folds 0 through 4"
    )

oof = (
    oof.set_index("__KEY")
    .loc[master["__KEY"]]
    .reset_index(drop=True)
)

y = master["ANC_CLASS"].to_numpy(int)
w = master["SURVEY_WEIGHT"].to_numpy(float)

if not np.array_equal(
    y, oof["ANC_CLASS_TRUE"].to_numpy(int)
):
    raise RuntimeError("OOF outcome mismatch")

if not np.array_equal(
    master["__FOLD"].to_numpy(),
    oof["Outer_Fold"].to_numpy(int)
):
    raise RuntimeError("OOF fold mismatch")

if not np.allclose(
    w,
    oof["SURVEY_WEIGHT"].to_numpy(float),
    rtol=0,
    atol=1e-12
):
    raise RuntimeError("Survey weight mismatch")

if not np.isfinite(w).all() or (w <= 0).any():
    raise RuntimeError("Invalid survey weights")

models = {
    k: joblib.load(p)
    for k, p in model_paths.items()
}

numeric = list(
    models[0, 1]["numeric_predictors"]
)

categorical = list(
    models[0, 1]["categorical_predictors"]
)

features = numeric + categorical

if (
    len(features) != 15
    or not set(features).issubset(master.columns)
):
    raise RuntimeError(
        "Expected 15 original predictors"
    )

for (f, s), bundle in models.items():

    if (
        int(bundle["outer_fold"]) != f
        or bundle["stage"] != f"Stage {s}"
        or (
            list(bundle["numeric_predictors"])
            + list(bundle["categorical_predictors"])
        ) != features
    ):
        raise RuntimeError(
            "Verified model metadata mismatch"
        )

if any(
    a not in features
    or b not in features
    or a == b
    for a, b in pairs
):
    raise RuntimeError(
        "Pair specification mismatch"
    )

def calibrated(bundle, df):

    xnum = df[
        bundle["numeric_predictors"]
    ].astype(float).to_numpy()

    if bundle["preprocessing"] == "Scaled OHE":
        xnum = bundle["scaler"].transform(xnum)

    elif bundle["preprocessing"] != "Unscaled OHE":
        raise RuntimeError(
            "Unexpected preprocessing"
        )

    xcat = bundle["encoder"].transform(
        df[
            bundle["categorical_predictors"]
        ].astype(str)
    )

    cls = np.asarray(
        bundle["estimator"].classes_
    )

    positive = np.flatnonzero(cls == 1)

    if len(positive) != 1:
        raise RuntimeError(
            "Invalid positive class"
        )

    raw = bundle[
        "estimator"
    ].predict_proba(
        np.hstack([xnum, xcat])
    )[:, int(positive[0])]

    raw = np.clip(
        raw, 1e-6, 1 - 1e-6
    )

    return expit(
        float(bundle["calibration_alpha"])
        + float(bundle["calibration_beta"])
        * logit(raw)
    )

def predictions(fold, df):

    q0 = calibrated(
        models[fold, 1],
        df
    )

    q2 = calibrated(
        models[fold, 2],
        df
    )

    return np.column_stack([
        q0,
        q2,
        (1 - q0) * (1 - q2),
        (1 - q0) * q2
    ])

for fold in range(5):

    idx = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    p = predictions(
        fold,
        master.iloc[idx][features]
    )

    expected = oof.iloc[idx][[
        "q0_CALIBRATED",
        "q2_CALIBRATED_CONDITIONAL",
        "P_ANC_1_7",
        "P_ANC_GE8"
    ]].to_numpy(float)

    if np.max(np.abs(p - expected)) > TOL:
        raise RuntimeError(
            f"Frozen OOF replay failed in fold {fold}"
        )

print("FROZEN MODEL REPLAY: PASS | 10/10")

def grid_for(df, feature):

    xw = df[
        "SURVEY_WEIGHT"
    ].to_numpy(float)

    if feature in categorical:

        x = df[
            feature
        ].astype(str).to_numpy()

        frame = pd.DataFrame({
            "level": x,
            "weight": xw
        })

        weights_by = (
            frame.groupby("level")["weight"]
            .sum()
            .sort_values(ascending=False)
        )

        selected = sorted(
            weights_by.index[
                :MAX_CATEGORIES
            ].tolist()
        )

        grids = np.asarray(
            selected,
            dtype=str
        )

        mapping = {
            v: j
            for j, v in enumerate(grids)
        }

        codes = np.asarray([
            mapping.get(v, -1)
            for v in x
        ], dtype=int)

    else:

        x = df[
            feature
        ].to_numpy(float)

        order = np.argsort(
            x, kind="stable"
        )

        xsorted = x[order]

        cumulative = np.cumsum(
            xw[order]
        )

        cuts = np.array([
            .1, .3, .5, .7, .9
        ]) * cumulative[-1]

        grids = np.unique(
            xsorted[
                np.searchsorted(
                    cumulative,
                    cuts,
                    side="left"
                )
            ]
        )

        if len(grids) == 0:
            raise RuntimeError(
                f"No quantile levels for {feature}"
            )

        codes = np.argmin(
            np.abs(
                x[:, None]
                - grids[None, :]
            ),
            axis=1
        )

    counts = np.bincount(
        codes[codes >= 0],
        weights=xw[codes >= 0],
        minlength=len(grids)
    )

    mass = counts / counts.sum()

    coverage = float(
        xw[codes >= 0].sum()
        / xw.sum()
    )

    return grids, mass, codes, coverage

def interaction_components(
    surface, ma, mb
):

    wa = ma[:, None] * mb[None, :]

    a = (
        surface * mb[None, :]
    ).sum(axis=1)

    b = (
        surface * ma[:, None]
    ).sum(axis=0)

    grand = float(
        (surface * wa).sum()
    )

    residual = (
        surface
        - a[:, None]
        - b[None, :]
        + grand
    )

    centering_error = max(
        np.max(
            np.abs(
                (
                    residual * mb[None, :]
                ).sum(axis=1)
            )
        ),
        np.max(
            np.abs(
                (
                    residual * ma[:, None]
                ).sum(axis=0)
            )
        )
    )

    if centering_error > 1e-10:
        raise RuntimeError(
            "Interaction centering failed"
        )

    variance = float(
        np.sum(
            wa * residual ** 2
        )
    )

    total = float(
        np.sum(
            wa * (surface - grand) ** 2
        )
    )

    rms = float(
        np.sqrt(max(0, variance))
    )

    h = (
        float(
            np.sqrt(variance / total)
        )
        if total > 1e-12
        else float("nan")
    )

    return residual, rms, h

scope_targets = {
    "all_women": [
        (0, "Stage1_NoANC"),
        (2, "Final_1to7"),
        (3, "Final_GE8")
    ],
    "anc_users": [
        (1, "Stage2_GE8_conditional")
    ]
}

spec = {
    "method": (
        "survey-PPS train-background "
        "interventional 2D PD functional ANOVA"
    ),
    "seed": SEED,
    "n_background": N_BACKGROUND,
    "max_categories": MAX_CATEGORIES,
    "numeric_quantiles": [
        .1, .3, .5, .7, .9
    ],
    "cell_min_n": MIN_CELL_N,
    "cell_min_ess": MIN_CELL_ESS,
    "pairs": [
        list(pair)
        for pair in pairs
    ],
    "sha256": {
        str(p.relative_to(ROOT)): sha256(p)
        for p in [
            master_path,
            outer_path,
            oof_path,
            *model_paths.values()
        ]
    }
}

OUT.mkdir(
    parents=True,
    exist_ok=True
)

PARTS.mkdir(
    parents=True,
    exist_ok=True
)

spec_path = OUT / "run_spec.json"

if spec_path.exists():

    if json.loads(
        spec_path.read_text(
            encoding="utf-8"
        )
    ) != spec:
        raise RuntimeError(
            "Existing interaction run uses "
            "different inputs/settings"
        )

elif list(PARTS.glob("*.npz")):

    raise RuntimeError(
        "Checkpoints exist without "
        "a verified run specification"
    )

else:

    spec_path.write_text(
        json.dumps(
            spec, indent=2
        ),
        encoding="utf-8"
    )

fold_records = []
cell_records = []

for fold in range(5):

    train_all = np.flatnonzero(
        master["__FOLD"].to_numpy() != fold
    )

    test_all = np.flatnonzero(
        master["__FOLD"].to_numpy() == fold
    )

    for scope, targets in scope_targets.items():

        if scope == "all_women":

            training = train_all
            holdout = test_all

        else:

            training = train_all[
                y[train_all] != 0
            ]

            holdout = test_all[
                y[test_all] != 0
            ]

        if (
            len(training) < N_BACKGROUND
            or len(holdout) < 5
        ):
            raise RuntimeError(
                f"Insufficient population: "
                f"fold={fold}, {scope}"
            )

        training_frame = (
            master.iloc[training]
            .reset_index(drop=True)
        )

        rng = np.random.default_rng(
            np.random.SeedSequence([
                SEED,
                897,
                fold,
                int(scope == "anc_users")
            ])
        )

        sampled_local = rng.choice(
            len(training),
            size=N_BACKGROUND,
            replace=True,
            p=(
                w[training]
                / w[training].sum()
            )
        )

        background = (
            training_frame.iloc[
                sampled_local
            ][features]
            .copy()
            .reset_index(drop=True)
        )

        holdout_weight = float(
            w[holdout].sum()
        )

        for a, b in pairs:

            ga, ma, ca, cover_a = grid_for(
                training_frame, a
            )

            gb, mb, cb, cover_b = grid_for(
                training_frame, b
            )

            if (
                len(ga) < 2
                or len(gb) < 2
            ):
                print(
                    f"Fold {fold}, {scope}: "
                    f"skipped constant-grid pair "
                    f"{a} x {b}"
                )
                continue

            na, nb = len(ga), len(gb)

            mask = (
                (ca >= 0)
                & (cb >= 0)
            )

            code = (
                ca[mask] * nb
                + cb[mask]
            )

            n_grid = na * nb

            counts = np.bincount(
                code,
                minlength=n_grid
            ).reshape(na, nb)

            weighted = np.bincount(
                code,
                weights=w[training][mask],
                minlength=n_grid
            )

            squared = np.bincount(
                code,
                weights=(
                    w[training][mask] ** 2
                ),
                minlength=n_grid
            )

            ess = np.divide(
                weighted ** 2,
                squared,
                out=np.zeros(
                    n_grid,
                    dtype=float
                ),
                where=squared > 0
            ).reshape(na, nb)

            supported = (
                (counts >= MIN_CELL_N)
                & (ess >= MIN_CELL_ESS)
            )

            grid_weights = (
                ma[:, None]
                * mb[None, :]
            )

            supported_mass = float(
                np.sum(
                    grid_weights
                    * supported
                )
            )

            joint_coverage = float(
                w[training][mask].sum()
                / w[training].sum()
            )

            ck = (
                PARTS
                / f"fold_{fold}_{scope}_{a}__{b}.npz"
            )

            if ck.is_file():

                with np.load(
                    ck, allow_pickle=False
                ) as previous:

                    surface_prob = previous[
                        "prob"
                    ]

                    surface_logit = previous[
                        "logit"
                    ]

                    saved_ga = previous[
                        "ga"
                    ].astype(str)

                    saved_gb = previous[
                        "gb"
                    ].astype(str)

                    saved_bg = previous[
                        "background_idx"
                    ].astype(int)

                if (
                    surface_prob.shape
                    != (4, na, nb)
                    or surface_logit.shape
                    != (4, na, nb)
                    or not np.array_equal(
                        saved_ga,
                        ga.astype(str)
                    )
                    or not np.array_equal(
                        saved_gb,
                        gb.astype(str)
                    )
                    or not np.array_equal(
                        saved_bg,
                        training[sampled_local]
                    )
                ):
                    raise RuntimeError(
                        f"Interaction checkpoint "
                        f"mismatch: {ck}"
                    )

            else:

                combinations = list(
                    product(
                        ga.tolist(),
                        gb.tolist()
                    )
                )

                modified = pd.concat(
                    [background]
                    * len(combinations),
                    ignore_index=True
                )

                modified[a] = np.repeat(
                    [
                        item[0]
                        for item in combinations
                    ],
                    N_BACKGROUND
                )

                modified[b] = np.repeat(
                    [
                        item[1]
                        for item in combinations
                    ],
                    N_BACKGROUND
                )

                p = predictions(
                    fold,
                    modified
                ).reshape(
                    na,
                    nb,
                    N_BACKGROUND,
                    4
                )

                surface_prob = (
                    p.mean(axis=2)
                    .transpose(2, 0, 1)
                )

                surface_logit = (
                    logit(
                        np.clip(
                            p,
                            1e-10,
                            1 - 1e-10
                        )
                    )
                    .mean(axis=2)
                    .transpose(2, 0, 1)
                )

                if (
                    not np.isfinite(
                        surface_prob
                    ).all()
                    or not np.isfinite(
                        surface_logit
                    ).all()
                ):
                    raise RuntimeError(
                        "Nonfinite interaction surfaces"
                    )

                tmp = ck.with_suffix(
                    ".npz.part"
                )

                with tmp.open("wb") as handle:

                    np.savez_compressed(
                        handle,
                        prob=surface_prob,
                        logit=surface_logit,
                        ga=ga.astype(str),
                        gb=gb.astype(str),
                        background_idx=(
                            training[sampled_local]
                        )
                    )

                tmp.replace(ck)

            for target_idx, target in targets:

                residual, rms, h = (
                    interaction_components(
                        surface_prob[
                            target_idx
                        ],
                        ma,
                        mb
                    )
                )

                _, lrms, lh = (
                    interaction_components(
                        surface_logit[
                            target_idx
                        ],
                        ma,
                        mb
                    )
                )

                supported_rms = (
                    float(
                        np.sqrt(
                            np.sum(
                                grid_weights
                                * supported
                                * residual ** 2
                            )
                            / supported_mass
                        )
                    )
                    if supported_mass > 0
                    else float("nan")
                )

                fold_records.append({
                    "fold": fold,
                    "population": scope,
                    "target": target,
                    "feature_a": a,
                    "feature_b": b,
                    "holdout_weight": holdout_weight,
                    "n_grid_a": na,
                    "n_grid_b": nb,
                    "joint_observed_coverage": (
                        joint_coverage
                    ),
                    "supported_product_grid_mass": (
                        supported_mass
                    ),
                    "risk_interaction_rms_pp": (
                        100 * rms
                    ),
                    "supported_risk_interaction_rms_pp": (
                        100 * supported_rms
                    ),
                    "risk_H": h,
                    "logit_interaction_rms": lrms,
                    "logit_H": lh,
                    "interpretation_limited": bool(
                        joint_coverage < .9
                        or supported_mass < .75
                    ),
                })

                for i in range(na):
                    for j in range(nb):

                        cell_records.append({
                            "fold": fold,
                            "target": target,
                            "feature_a": a,
                            "feature_b": b,
                            "level_a": str(
                                ga[i]
                            ),
                            "level_b": str(
                                gb[j]
                            ),
                            "grid_product_weight": float(
                                grid_weights[i, j]
                            ),
                            "observed_joint_n": int(
                                counts[i, j]
                            ),
                            "observed_joint_ess": float(
                                ess[i, j]
                            ),
                            "sufficient_joint_support": bool(
                                supported[i, j]
                            ),
                            "pd_predicted_risk": float(
                                surface_prob[
                                    target_idx, i, j
                                ]
                            ),
                            "centered_interaction_pp": float(
                                residual[i, j] * 100
                            ),
                        })

        print(
            f"Fold {fold} | {scope} | "
            f"{len(pairs)} pairs processed"
        )

fold_table = pd.DataFrame(
    fold_records
)

cells = pd.DataFrame(
    cell_records
)

if fold_table.empty:
    raise RuntimeError(
        "No pairwise interaction results"
    )

summary = []

for (
    target, a, b
), group in fold_table.groupby(
    [
        "target",
        "feature_a",
        "feature_b"
    ],
    sort=False
):

    ww = group[
        "holdout_weight"
    ].to_numpy(float)

    def avg(col):

        x = group[
            col
        ].to_numpy(float)

        ok = np.isfinite(x)

        return (
            float(
                np.average(
                    x[ok],
                    weights=ww[ok]
                )
            )
            if ok.any()
            else float("nan")
        )

    pooled_rms = float(
        np.sqrt(
            np.average(
                group[
                    "risk_interaction_rms_pp"
                ].to_numpy(float) ** 2,
                weights=ww
            )
        )
    )

    summary.append({
        "target": target,
        "feature_a": a,
        "feature_b": b,
        "n_folds": len(group),
        "interaction_rms_pp": pooled_rms,
        "supported_interaction_rms_pp": avg(
            "supported_risk_interaction_rms_pp"
        ),
        "logit_interaction_rms": avg(
            "logit_interaction_rms"
        ),
        "mean_risk_H": avg("risk_H"),
        "mean_logit_H": avg("logit_H"),
        "mean_supported_grid_mass": avg(
            "supported_product_grid_mass"
        ),
        "mean_joint_observed_coverage": avg(
            "joint_observed_coverage"
        ),
        "limited_fold_count": int(
            group[
                "interpretation_limited"
            ].sum()
        ),
    })

pooled = pd.DataFrame(
    summary
)

fold_table.to_csv(
    OUT / "fold_pair_nonadditivity.csv",
    index=False
)

pooled.to_csv(
    OUT / "pooled_pair_nonadditivity.csv",
    index=False
)

cells.to_csv(
    OUT / "pairwise_probability_surfaces.csv",
    index=False
)

stability_rows = []

for target, group in fold_table.groupby(
    "target"
):

    scores = group.pivot(
        index=[
            "feature_a",
            "feature_b"
        ],
        columns="fold",
        values="risk_interaction_rms_pp"
    )

    for left in range(5):

        for right in range(
            left + 1, 5
        ):

            rho, _ = spearmanr(
                scores[left].to_numpy(float),
                scores[right].to_numpy(float)
            )

            stability_rows.append({
                "target": target,
                "fold_a": left,
                "fold_b": right,
                "rho": float(rho)
            })

pd.DataFrame(
    stability_rows
).to_csv(
    OUT / "pair_rank_stability.csv",
    index=False
)

targets = [
    "Stage1_NoANC",
    "Stage2_GE8_conditional",
    "Final_1to7",
    "Final_GE8"
]

for target in targets:

    subset = pooled.loc[
        pooled["target"] == target
    ].copy()

    eligible = subset[
        (
            subset[
                "mean_supported_grid_mass"
            ] >= .75
        )
        & (
            subset[
                "limited_fold_count"
            ] == 0
        )
    ]

    top = (
        eligible
        if not eligible.empty
        else subset
    ).nlargest(
        3,
        "interaction_rms_pp"
    )

    for _, pair in top.iterrows():

        aa = pair["feature_a"]
        bb = pair["feature_b"]

        frows = fold_table[
            (
                fold_table["target"]
                == target
            )
            & (
                fold_table["feature_a"]
                == aa
            )
            & (
                fold_table["feature_b"]
                == bb
            )
        ]

        chosen = int(
            frows.sort_values(
                "holdout_weight",
                ascending=False
            ).iloc[0]["fold"]
        )

        selected = cells[
            (
                cells["target"]
                == target
            )
            & (
                cells["feature_a"]
                == aa
            )
            & (
                cells["feature_b"]
                == bb
            )
            & (
                cells["fold"]
                == chosen
            )
        ]

        scope = (
            "anc_users"
            if target
            == "Stage2_GE8_conditional"
            else "all_women"
        )

        source_ck = (
            PARTS
            / f"fold_{chosen}_{scope}_{aa}__{bb}.npz"
        )

        with np.load(
            source_ck,
            allow_pickle=False
        ) as saved_grid:

            ordered_a = saved_grid[
                "ga"
            ].astype(str).tolist()

            ordered_b = saved_grid[
                "gb"
            ].astype(str).tolist()

        heat = selected.pivot(
            index="level_a",
            columns="level_b",
            values="centered_interaction_pp"
        ).reindex(
            index=ordered_a,
            columns=ordered_b
        )

        support = selected.pivot(
            index="level_a",
            columns="level_b",
            values="sufficient_joint_support"
        ).reindex(
            index=ordered_a,
            columns=ordered_b
        )

        values = heat.to_numpy(float)

        supported_cells = support.to_numpy(
            bool
        )

        if not supported_cells.any():
            continue

        show_values = np.ma.masked_where(
            ~supported_cells,
            values
        )

        max_abs = max(
            float(
                np.max(
                    np.abs(
                        values[supported_cells]
                    )
                )
            ),
            1e-9
        )

        fig, ax = plt.subplots(
            figsize=(
                max(
                    8,
                    .72 * len(heat.columns) + 3
                ),
                max(
                    5,
                    .52 * len(heat.index) + 2
                )
            )
        )

        im = ax.imshow(
            show_values,
            aspect="auto",
            cmap="coolwarm",
            vmin=-max_abs,
            vmax=max_abs
        )

        ax.set_xticks(
            np.arange(
                len(heat.columns)
            ),
            heat.columns,
            rotation=40,
            ha="right"
        )

        ax.set_yticks(
            np.arange(
                len(heat.index)
            ),
            heat.index
        )

        ax.set_xlabel(bb)
        ax.set_ylabel(aa)

        ax.set_title(
            f"{target} | fold {chosen} | "
            "centered 2D PD interaction"
        )

        for i in range(
            len(heat.index)
        ):
            for j in range(
                len(heat.columns)
            ):

                ax.text(
                    j,
                    i,
                    (
                        f"{values[i, j]:+.2f}"
                        if support.iloc[i, j]
                        else "x"
                    ),
                    ha="center",
                    va="center",
                    fontsize=8
                )

        fig.colorbar(
            im,
            ax=ax,
            label=(
                "Interaction deviation "
                "(percentage points)"
            )
        )

        fig.tight_layout()

        fig.savefig(
            OUT
            / f"{target}__{aa}__{bb}__example_fold{chosen}.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(fig)

print(
    "\nTARGETED PAIRWISE NONADDITIVITY: COMPLETE"
)

for target in targets:

    group = pooled[
        pooled["target"] == target
    ].nlargest(
        6,
        "interaction_rms_pp"
    )

    print(
        f"\n{target} - highest 6 "
        "probability-scale pairwise nonadditivities"
    )

    print(
        group[
            [
                "feature_a",
                "feature_b",
                "interaction_rms_pp",
                "logit_interaction_rms",
                "mean_supported_grid_mass",
                "limited_fold_count"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

stage1_max = pooled.loc[
    pooled["target"]
    == "Stage1_NoANC",
    "logit_interaction_rms"
].max()

print(
    "\nStage 1 maximum logit-scale interaction RMS:",
    f"{stage1_max:.3e}"
)

print(
    "Pairs with limited support in at least one fold:",
    int(
        (
            pooled["limited_fold_count"] > 0
        ).sum()
    ),
    "/",
    len(pooled)
)

print("Output:", OUT)

FROZEN MODEL REPLAY: PASS | 10/10
Fold 0 | all_women | 12 pairs processed
Fold 0 | anc_users | 12 pairs processed
Fold 1 | all_women | 12 pairs processed
Fold 1 | anc_users | 12 pairs processed
Fold 2 | all_women | 12 pairs processed
Fold 2 | anc_users | 12 pairs processed
Fold 3 | all_women | 12 pairs processed
Fold 3 | anc_users | 12 pairs processed
Fold 4 | all_women | 12 pairs processed
Fold 4 | anc_users | 12 pairs processed

TARGETED PAIRWISE NONADDITIVITY: COMPLETE

Stage1_NoANC - highest 6 probability-scale pairwise nonadditivities
            feature_a                       feature_b  interaction_rms_pp  logit_interaction_rms  mean_supported_grid_mass  limited_fold_count
      wealth_quintile           weekly_media_exposure              0.6102                 0.0000                    1.0000                   0
      wealth_quintile               prior_live_births              0.6081                 0.0000                    1.0000                   0
      wealth_quintile    

In [18]:

from pathlib import Path
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")

if not ROOT.is_dir():
    ROOT = Path(r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026")

DATA = ROOT / "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
OOF = ROOT / "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
OUTER = ROOT / "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv"
MANIFEST = ROOT / "10_explainability/XAI_SELECTED_REFITS/reconstruction_manifest.json"
MODEL = ROOT / "10_explainability/XAI_SELECTED_REFITS/outer_0_stage_1.joblib"
STAGE = ROOT / "10_explainability/XAI_PRIMARY_SHAP/heldout_original_feature_shap.csv"
FINAL = ROOT / "10_explainability/XAI_FINAL_MULTICLASS_SHAP/heldout_multiclass_original_feature_shap.csv"
OUTPUT = ROOT / "10_explainability/XAI_SUBGROUP_HETEROGENEITY"

DOMAINS = [
    "division",
    "wealth_quintile",
    "maternal_education",
    "residence",
    "weekly_media_exposure"
]

MIN_EXPLAINED_N = 50
MIN_EXPLAINED_PSU = 25
MIN_KISH_ESS = 30
MIN_FOLDS = 4
MIN_FULL_EVENTS = 5

for path in [DATA, OOF, OUTER, MANIFEST, MODEL, STAGE, FINAL]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

if manifest.get("status") != "PASS" or int(manifest.get("number_of_models", 0)) != 10:
    raise RuntimeError("Frozen reconstruction manifest must pass")

def hash_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for label, path in [
    ("master", DATA),
    ("outer", OUTER),
    ("oof", OOF)
]:
    expected_hash = manifest.get("original_data_sha256", {}).get(label)
    if expected_hash and expected_hash != hash_file(path):
        raise RuntimeError(f"Frozen input changed: {label}")

master = pd.read_csv(DATA)
oof = pd.read_csv(OOF)
outer = pd.read_csv(OUTER)
stage = pd.read_csv(STAGE)
final = pd.read_csv(FINAL)

bundle = joblib.load(MODEL)

FEATURES = (
    list(bundle["numeric_predictors"])
    + list(bundle["categorical_predictors"])
)

if len(FEATURES) != 15 or not set(DOMAINS).issubset(FEATURES):
    raise RuntimeError("Frozen feature/domain specification mismatch")

for col in [
    "_ROW_ID", "ANC_CLASS", "SURVEY_WEIGHT",
    "PSU", "STRATUM", *DOMAINS
]:
    if col not in master.columns:
        raise RuntimeError(f"Missing master column: {col}")

if master[["PSU", "STRATUM"]].isna().any().any():
    raise RuntimeError("Missing survey PSU or strata")

def keys(series):
    n = pd.to_numeric(series, errors="coerce")

    if n.notna().all() and np.allclose(
        n.to_numpy(float),
        np.round(n.to_numpy(float))
    ):
        return pd.Series(
            np.round(n.to_numpy(float)).astype(np.int64).astype(str),
            index=series.index
        )

    return series.astype(str).str.strip()

for frame, name in [
    (master, "_ROW_ID"),
    (oof, "_ROW_KEY"),
    (outer, "_ROW_ID")
]:
    frame["__KEY"] = keys(frame[name])

    if frame["__KEY"].duplicated().any():
        raise RuntimeError(f"Duplicate IDs: {name}")

if len(master) != 5063 or not (
    set(master["__KEY"])
    == set(oof["__KEY"])
    == set(outer["__KEY"])
):
    raise RuntimeError("Frozen cohort alignment failure")

master["__FOLD"] = master["__KEY"].map(
    outer.set_index("__KEY")["OUTER_FOLD"]
)

if master["__FOLD"].isna().any():
    raise RuntimeError("Missing outer-fold assignments")

master["__FOLD"] = master["__FOLD"].astype(int)

if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError("Unexpected outer folds")

oof = (
    oof.set_index("__KEY")
    .loc[master["__KEY"]]
    .reset_index()
)

if not np.array_equal(
    oof["ANC_CLASS_TRUE"].to_numpy(int),
    master["ANC_CLASS"].to_numpy(int)
):
    raise RuntimeError("Outcome alignment mismatch")

if not np.array_equal(
    oof["Outer_Fold"].to_numpy(int),
    master["__FOLD"].to_numpy(int)
):
    raise RuntimeError("Outer-fold alignment mismatch")

if not np.allclose(
    oof["SURVEY_WEIGHT"],
    master["SURVEY_WEIGHT"],
    atol=1e-12,
    rtol=0
):
    raise RuntimeError("Survey weight mismatch")

survey_w = master["SURVEY_WEIGHT"].to_numpy(float)

if not np.isfinite(survey_w).all() or (survey_w <= 0).any():
    raise RuntimeError("Invalid survey weights")

meta = master.set_index("__KEY")[
    [
        "ANC_CLASS", "SURVEY_WEIGHT",
        "PSU", "STRATUM", "__FOLD", *DOMAINS
    ]
]

full = master[
    [
        "__KEY", "ANC_CLASS", "SURVEY_WEIGHT",
        "PSU", "STRATUM", "__FOLD", *DOMAINS
    ]
].copy()

for domain in DOMAINS:
    full[domain] = full[domain].astype(str)

full["q0"] = oof["q0_CALIBRATED"].to_numpy(float)
full["q2"] = oof["q2_CALIBRATED_CONDITIONAL"].to_numpy(float)
full["p1"] = oof["P_ANC_1_7"].to_numpy(float)
full["p2"] = oof["P_ANC_GE8"].to_numpy(float)

reconstructed = np.column_stack([
    full["q0"],
    (1 - full["q0"]) * (1 - full["q2"]),
    (1 - full["q0"]) * full["q2"]
])

reference = oof[
    ["P_NO_ANC", "P_ANC_1_7", "P_ANC_GE8"]
].to_numpy(float)

if np.max(np.abs(reconstructed - reference)) > 1e-10:
    raise RuntimeError("Frozen multiclass probabilities disagree")

stage["__KEY"] = keys(stage["row_key"])
final["__KEY"] = keys(final["row_key"])

if (
    stage.duplicated(["__KEY", "stage"]).any()
    or final["__KEY"].duplicated().any()
):
    raise RuntimeError("Duplicate SHAP participant IDs")

sources = {
    "Stage1_NoANC": (
        stage.loc[stage["stage"] == 1].copy(),
        "SHAP__", "predicted_probability",
        "base_probability", "q0", 0, False, "fold"
    ),
    "Stage2_GE8_conditional": (
        stage.loc[stage["stage"] == 2].copy(),
        "SHAP__", "predicted_probability",
        "base_probability", "q2", 2, True, "fold"
    ),
    "Final_1to7": (
        final.copy(),
        "class1__", "class1_predicted",
        "class1_base", "p1", 1, False, "outer_fold"
    ),
    "Final_GE8": (
        final.copy(),
        "class2__", "class2_predicted",
        "class2_base", "p2", 2, False, "outer_fold"
    )
}

def kish_ess(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0 or values.sum() <= 0:
        return 0.0

    return float(
        values.sum() ** 2 / np.square(values).sum()
    )

def cluster_ids(df):
    return (
        df["STRATUM"].astype(str)
        + "::"
        + df["PSU"].astype(str)
    )

def weighted_mean(values, weights):
    return float(
        np.average(
            np.asarray(values, dtype=float),
            weights=np.asarray(weights, dtype=float)
        )
    )

clean = {}
global_importance = {}

for target, (
    raw, prefix, prob_col, base_col,
    full_prob_col, event, conditional, fold_col
) in sources.items():

    needed = [
        "__KEY", fold_col, "survey_weight",
        "analysis_weight", prob_col, base_col
    ] + [prefix + feature for feature in FEATURES]

    if not set(needed).issubset(raw.columns):
        raise RuntimeError(f"Incomplete SHAP source: {target}")

    raw = raw[needed].copy()

    if raw["__KEY"].duplicated().any() or len(raw) < 100:
        raise RuntimeError(f"Invalid explanation sample: {target}")

    if not raw["__KEY"].isin(meta.index).all():
        raise RuntimeError(f"Unknown participant in {target}")

    raw = raw.join(
        meta,
        on="__KEY",
        how="left",
        validate="one_to_one"
    )

    for domain in DOMAINS:
        raw[domain] = raw[domain].astype(str)

    if not np.array_equal(
        raw[fold_col].to_numpy(int),
        raw["__FOLD"].to_numpy(int)
    ):
        raise RuntimeError(f"Fold mismatch: {target}")

    if not np.allclose(
        raw["survey_weight"],
        raw["SURVEY_WEIGHT"],
        rtol=0,
        atol=1e-12
    ):
        raise RuntimeError(f"Weight mismatch: {target}")

    if conditional and (
        raw["ANC_CLASS"].to_numpy(int) == 0
    ).any():
        raise RuntimeError("Stage 2 includes nonusers")

    expected_prob = (
        full.set_index("__KEY")
        .loc[raw["__KEY"], full_prob_col]
        .to_numpy(float)
    )

    if np.max(
        np.abs(
            raw[prob_col].to_numpy(float)
            - expected_prob
        )
    ) > 1e-10:
        raise RuntimeError(f"OOF probability mismatch: {target}")

    values = raw[
        [prefix + feature for feature in FEATURES]
    ].to_numpy(float)

    if (
        not np.isfinite(values).all()
        or np.max(
            np.abs(
                raw[base_col].to_numpy(float)
                + values.sum(axis=1)
                - expected_prob
            )
        ) > 1e-7
    ):
        raise RuntimeError(f"SHAP additivity failure: {target}")

    if (
        not np.isfinite(raw["analysis_weight"]).all()
        or (raw["analysis_weight"] <= 0).any()
    ):
        raise RuntimeError(f"Invalid analysis weights: {target}")

    for fold, group in raw.groupby("__FOLD"):

        population = (
            (full["__FOLD"] == fold)
            & (
                (full["ANC_CLASS"] != 0)
                if conditional
                else True
            )
        )

        n_full = int(population.sum())

        ratio = (
            group["analysis_weight"].to_numpy(float)
            / group["SURVEY_WEIGHT"].to_numpy(float)
        )

        if not np.allclose(
            ratio,
            n_full / len(group),
            atol=1e-7,
            rtol=1e-7
        ):
            raise RuntimeError(
                f"Sampling expansion mismatch: {target}/{fold}"
            )

    importance = np.average(
        np.abs(values),
        axis=0,
        weights=raw["analysis_weight"].to_numpy(float)
    )

    global_importance[target] = dict(
        zip(FEATURES, importance)
    )

    raw.attrs = {
        "prefix": prefix,
        "prob_col": prob_col,
        "full_prob_col": full_prob_col,
        "event": event,
        "conditional": conditional
    }

    clean[target] = raw

stage1 = clean["Stage1_NoANC"].set_index("__KEY")
class0 = final.set_index("__KEY")

if set(stage1.index) != set(class0.index):
    raise RuntimeError("Stage 1/class 0 samples differ")

for feature in FEATURES:

    difference = np.max(
        np.abs(
            stage1.loc[
                class0.index,
                "SHAP__" + feature
            ].to_numpy(float)
            - class0[
                "class0__" + feature
            ].to_numpy(float)
        )
    )

    if difference > 1e-7:
        raise RuntimeError(
            f"Stage 1/class 0 attribution mismatch: {feature}"
        )

print(
    "SUBGROUP INPUT QC: PASS | "
    "4 unique targets | 15 predictors | 5 domains"
)

risk_records = []
subgroup_records = []
importance_records = []

for target, sample in clean.items():

    prob_col = sample.attrs["prob_col"]
    pred_full = sample.attrs["full_prob_col"]
    positive = sample.attrs["event"]
    conditional = sample.attrs["conditional"]
    prefix = sample.attrs["prefix"]

    pool = (
        full.loc[full["ANC_CLASS"] != 0]
        if conditional
        else full
    )

    overall = pd.Series(
        global_importance[target]
    )

    overall_share = overall / overall.sum()

    for domain in DOMAINS:

        for level in sorted(pool[domain].unique().tolist()):

            population = pool.loc[
                pool[domain] == level
            ]

            explained = sample.loc[
                sample[domain] == level
            ]

            pop_w = population[
                "SURVEY_WEIGHT"
            ].to_numpy(float)

            pop_pred = population[
                pred_full
            ].to_numpy(float)

            event_pop = (
                population["ANC_CLASS"].to_numpy(int)
                == positive
            ).astype(float)

            obs_rate = weighted_mean(event_pop, pop_w)
            pred_rate = weighted_mean(pop_pred, pop_w)
            event_n = int(event_pop.sum())

            risk_records.append({
                "target": target,
                "domain": domain,
                "level": level,
                "n_full": len(population),
                "full_event_n": event_n,
                "full_psu_n": cluster_ids(population).nunique(),
                "full_strata_n": population["STRATUM"].nunique(),
                "full_weighted_observed_rate": obs_rate,
                "full_weighted_predicted_rate": pred_rate,
                "full_observed_minus_predicted_pp": 100 * (
                    obs_rate - pred_rate
                ),
                "full_OE_ratio": (
                    obs_rate / pred_rate
                    if pred_rate > 0
                    else np.nan
                ),
                "full_weighted_brier": weighted_mean(
                    (event_pop - pop_pred) ** 2,
                    pop_w
                ),
                "full_few_positive_events": (
                    event_n < MIN_FULL_EVENTS
                )
            })

            if explained.empty:

                subgroup_records.append({
                    "target": target,
                    "domain": domain,
                    "level": level,
                    "n_explained": 0,
                    "support_flag": "NO_EXPLAINED_ROWS"
                })

                continue

            ew = explained[
                "analysis_weight"
            ].to_numpy(float)

            shap_abs = np.array([
                weighted_mean(
                    np.abs(
                        explained[prefix + feature]
                    ),
                    ew
                )
                for feature in FEATURES
            ])

            shap_signed = np.array([
                weighted_mean(
                    explained[prefix + feature],
                    ew
                )
                for feature in FEATURES
            ])

            shap_positive = np.array([
                weighted_mean(
                    (
                        explained[
                            prefix + feature
                        ].to_numpy(float) > 0
                    ).astype(float),
                    ew
                )
                for feature in FEATURES
            ])

            imp = pd.Series(
                shap_abs,
                index=FEATURES
            )

            share = (
                imp / imp.sum()
                if imp.sum() > 0
                else imp * np.nan
            )

            rho, _ = spearmanr(
                overall.to_numpy(float),
                shap_abs
            )

            top = imp.nlargest(5).index.tolist()

            n_psu = cluster_ids(explained).nunique()
            n_strata = explained["STRATUM"].nunique()
            n_folds = explained["__FOLD"].nunique()
            ess = kish_ess(ew)

            supported = (
                len(explained) >= MIN_EXPLAINED_N
                and n_psu >= MIN_EXPLAINED_PSU
                and ess >= MIN_KISH_ESS
                and n_folds >= MIN_FOLDS
            )

            support_flag = (
                "INTERPRETABLE_DESCRIPTIVELY"
                if supported
                else "LIMITED_SAMPLE_SUPPORT"
            )

            subgroup_records.append({
                "target": target,
                "domain": domain,
                "level": level,
                "n_explained": len(explained),
                "n_explained_psu": n_psu,
                "n_explained_strata": n_strata,
                "folds_represented": n_folds,
                "kish_weight_ess_not_design_ess": ess,
                "explained_sample_mean_probability": weighted_mean(
                    explained[prob_col],
                    ew
                ),
                "full_OOF_mean_probability": pred_rate,
                "explained_minus_full_probability_pp": 100 * (
                    weighted_mean(
                        explained[prob_col],
                        ew
                    ) - pred_rate
                ),
                "rank_spearman_vs_target_overall": float(rho),
                "top5_overlap_vs_target_overall": len(
                    set(top)
                    & set(
                        overall.nlargest(5).index
                    )
                ),
                "top5_features": " | ".join(top),
                "support_flag": support_flag
            })

            for i, feature in enumerate(FEATURES):

                importance_records.append({
                    "target": target,
                    "domain": domain,
                    "level": level,
                    "feature": feature,
                    "n_explained": len(explained),
                    "support_flag": support_flag,
                    "mean_abs_shap_probability": shap_abs[i],
                    "mean_signed_shap_probability": shap_signed[i],
                    "fraction_positive_shap": shap_positive[i],
                    "relative_importance_share": float(
                        share.loc[feature]
                    ),
                    "overall_target_importance_share": float(
                        overall_share.loc[feature]
                    ),
                    "share_difference_pp": 100 * (
                        float(share.loc[feature])
                        - float(overall_share.loc[feature])
                    )
                })

risk = pd.DataFrame(risk_records)
subgroups = pd.DataFrame(subgroup_records)
importance = pd.DataFrame(importance_records)

if risk.empty or importance.empty or subgroups.empty:
    raise RuntimeError("Subgroup results unexpectedly empty")

comparisons = []

two_stages = importance.loc[
    importance["target"].isin([
        "Stage1_NoANC",
        "Stage2_GE8_conditional"
    ])
]

for (domain, level), group in two_stages.groupby(
    ["domain", "level"]
):

    a = (
        group.loc[
            group["target"] == "Stage1_NoANC"
        ]
        .set_index("feature")[
            "mean_abs_shap_probability"
        ]
        .reindex(FEATURES)
    )

    b = (
        group.loc[
            group["target"] == "Stage2_GE8_conditional"
        ]
        .set_index("feature")[
            "mean_abs_shap_probability"
        ]
        .reindex(FEATURES)
    )

    if a.isna().any() or b.isna().any():
        continue

    rho, _ = spearmanr(a, b)

    s1 = subgroups.loc[
        (subgroups["target"] == "Stage1_NoANC")
        & (subgroups["domain"] == domain)
        & (subgroups["level"] == level)
    ].iloc[0]

    s2 = subgroups.loc[
        (subgroups["target"] == "Stage2_GE8_conditional")
        & (subgroups["domain"] == domain)
        & (subgroups["level"] == level)
    ].iloc[0]

    comparisons.append({
        "domain": domain,
        "level": level,
        "stage1_n_explained": int(s1["n_explained"]),
        "stage2_n_explained": int(s2["n_explained"]),
        "stage1_support": s1["support_flag"],
        "stage2_support": s2["support_flag"],
        "importance_spearman_stage1_vs_stage2": float(rho),
        "top5_overlap_stage1_vs_stage2": len(
            set(a.nlargest(5).index)
            & set(b.nlargest(5).index)
        )
    })

comparisons = pd.DataFrame(comparisons)

spec = {
    "method": (
        "Domain-specific survey-weighted held-out SHAP "
        "with full-cohort OOF risk context"
    ),
    "domains": DOMAINS,
    "targets": list(clean),
    "n_original_predictors": len(FEATURES),
    "minimum_explained_n": MIN_EXPLAINED_N,
    "minimum_explained_psus": MIN_EXPLAINED_PSU,
    "minimum_kish_weight_ess": MIN_KISH_ESS,
    "minimum_folds": MIN_FOLDS,
    "source_hashes": {
        str(p.relative_to(ROOT)): hash_file(p)
        for p in [
            DATA, OOF, OUTER, MANIFEST,
            STAGE, FINAL
        ]
    }
}

OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

spec_path = OUTPUT / "subgroup_run_spec.json"

if spec_path.is_file():

    previous = json.loads(
        spec_path.read_text(encoding="utf-8")
    )

    if previous != spec:
        raise RuntimeError(
            "Existing subgroup outputs use different inputs/settings"
        )

spec_path.write_text(
    json.dumps(spec, indent=2),
    encoding="utf-8"
)

risk.to_csv(
    OUTPUT / "subgroup_full_oof_risk_calibration.csv",
    index=False
)

subgroups.to_csv(
    OUTPUT / "subgroup_shap_ranking_and_support.csv",
    index=False
)

importance.to_csv(
    OUTPUT / "subgroup_feature_importance.csv",
    index=False
)

comparisons.to_csv(
    OUTPUT / "subgroup_stage1_stage2_rank_comparison.csv",
    index=False
)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 10
})

for target in [
    "Stage1_NoANC",
    "Stage2_GE8_conditional",
    "Final_1to7",
    "Final_GE8"
]:

    overall = pd.Series(
        global_importance[target]
    ).nlargest(8)

    for domain in [
        "division",
        "wealth_quintile"
    ]:

        selected = subgroups.loc[
            (subgroups["target"] == target)
            & (subgroups["domain"] == domain)
        ]

        levels = sorted(
            selected["level"].astype(str).tolist()
        )

        if not levels:
            continue

        table = (
            importance.loc[
                (importance["target"] == target)
                & (importance["domain"] == domain)
            ]
            .pivot_table(
                index="level",
                columns="feature",
                values="relative_importance_share",
                aggfunc="first"
            )
            .reindex(
                index=levels,
                columns=overall.index
            )
        )

        for row in selected.itertuples(index=False):

            if row.support_flag != "INTERPRETABLE_DESCRIPTIVELY":
                table.loc[row.level, :] = np.nan

        values = table.to_numpy(float) * 100

        fig, ax = plt.subplots(
            figsize=(
                10,
                max(4, 0.48 * len(levels) + 2)
            )
        )

        image = ax.imshow(
            np.ma.masked_invalid(values),
            cmap="viridis",
            aspect="auto",
            vmin=0
        )

        ax.set_xticks(
            np.arange(len(overall)),
            [
                feature.replace("_", " ")
                for feature in overall.index
            ],
            rotation=45,
            ha="right"
        )

        ax.set_yticks(
            np.arange(len(levels)),
            levels
        )

        ax.set_xlabel("Original predictor")
        ax.set_ylabel(domain.replace("_", " ").title())

        ax.set_title(
            f"{target}: within-group share of absolute SHAP"
        )

        fig.colorbar(
            image,
            ax=ax,
            label="Share of total group importance (%)"
        )

        fig.tight_layout()

        fig.savefig(
            OUTPUT / f"{target}__{domain}__shap_share.png",
            dpi=300,
            bbox_inches="tight"
        )

        fig.savefig(
            OUTPUT / f"{target}__{domain}__shap_share.svg",
            bbox_inches="tight"
        )

        plt.close(fig)

print("\nGEOGRAPHIC AND SOCIOECONOMIC XAI: COMPLETE")

print("Full-cohort OOF participants:", len(master))

print(
    "Stage 1/Stage 2 explanations:",
    len(clean["Stage1_NoANC"]),
    len(clean["Stage2_GE8_conditional"])
)

print("Domains:", ", ".join(DOMAINS))

print(
    "Groups with limited or no SHAP sample support:",
    int(
        (
            subgroups["support_flag"]
            != "INTERPRETABLE_DESCRIPTIVELY"
        ).sum()
    ),
    "/",
    len(subgroups)
)

for target in [
    "Stage1_NoANC",
    "Stage2_GE8_conditional",
    "Final_GE8"
]:

    subset = subgroups.loc[
        (subgroups["target"] == target)
        & subgroups["domain"].isin([
            "division",
            "wealth_quintile"
        ])
    ].copy()

    if subset.empty:
        continue

    shown = subset.merge(
        risk[
            [
                "target", "domain", "level",
                "n_full", "full_event_n",
                "full_weighted_observed_rate",
                "full_weighted_predicted_rate"
            ]
        ],
        on=["target", "domain", "level"],
        validate="one_to_one"
    )

    print(
        f"\n{target} | division and wealth quintile"
    )

    columns = [
        "domain",
        "level",
        "n_full",
        "full_event_n",
        "n_explained",
        "n_explained_psu",
        "full_weighted_observed_rate",
        "full_weighted_predicted_rate",
        "rank_spearman_vs_target_overall",
        "top5_overlap_vs_target_overall",
        "support_flag"
    ]

    print(
        shown[columns].to_string(
            index=False,
            float_format=lambda x: f"{x:.3f}"
        )
    )

print("\nOutput:", OUTPUT)

SUBGROUP INPUT QC: PASS | 4 unique targets | 15 predictors | 5 domains

GEOGRAPHIC AND SOCIOECONOMIC XAI: COMPLETE
Full-cohort OOF participants: 5063
Stage 1/Stage 2 explanations: 800 800
Domains: division, wealth_quintile, maternal_education, residence, weekly_media_exposure
Groups with limited or no SHAP sample support: 4 / 84

Stage1_NoANC | division and wealth quintile
         domain      level  n_full  full_event_n  n_explained  n_explained_psu  full_weighted_observed_rate  full_weighted_predicted_rate  rank_spearman_vs_target_overall  top5_overlap_vs_target_overall                support_flag
       division   Barishal     544            40           85               49                        0.088                         0.088                            0.946                               5 INTERPRETABLE_DESCRIPTIVELY
       division Chattogram     880            74          143               73                        0.091                         0.089                         

In [20]:

from pathlib import Path
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")

if not ROOT.is_dir():
    ROOT = Path(r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026")

REFITS = ROOT / "10_explainability/XAI_SELECTED_REFITS"
OUT = ROOT / "10_explainability/XAI_PSU_BOOTSTRAP_UNCERTAINTY"

MASTER = ROOT / "01_processed_data/FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
OOF = ROOT / "06_predictions/STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
OUTER = ROOT / "04_cv_splits/STEP13_FROZEN_outer_5fold_row_assignments.csv"

STAGE = ROOT / "10_explainability/XAI_PRIMARY_SHAP/heldout_original_feature_shap.csv"
FINAL = ROOT / "10_explainability/XAI_FINAL_MULTICLASS_SHAP/heldout_multiclass_original_feature_shap.csv"

MANIFEST = REFITS / "reconstruction_manifest.json"
MODEL = REFITS / "outer_0_stage_1.joblib"

SUBGROUP = ROOT / "10_explainability/XAI_SUBGROUP_HETEROGENEITY/subgroup_full_oof_risk_calibration.csv"
SUPPORT = ROOT / "10_explainability/XAI_SUBGROUP_HETEROGENEITY/subgroup_shap_ranking_and_support.csv"

B = 2000
SEED = 2026
ALPHA = 0.05
TOL = 1e-10

DOMAINS = [
    "division",
    "wealth_quintile",
    "maternal_education",
    "residence",
    "weekly_media_exposure"
]

required = [
    MASTER, OOF, OUTER, STAGE, FINAL,
    MANIFEST, MODEL, SUBGROUP, SUPPORT
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

if (
    manifest.get("status") != "PASS"
    or int(manifest.get("number_of_models", 0)) != 10
):
    raise RuntimeError(
        "Verified model reconstruction is required"
    )

def sha256(path):
    h = hashlib.sha256()

    with path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(1 << 20),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

for name, path in [
    ("master", MASTER),
    ("outer", OUTER),
    ("oof", OOF)
]:
    expected = manifest.get(
        "original_data_sha256", {}
    ).get(name)

    if expected and sha256(path) != expected:
        raise RuntimeError(
            f"Frozen file hash mismatch: {name}"
        )

master = pd.read_csv(MASTER)
oof = pd.read_csv(OOF)
outer = pd.read_csv(OUTER)

stage = pd.read_csv(STAGE)
final = pd.read_csv(FINAL)

prior_subgroups = pd.read_csv(SUBGROUP)
support_table = pd.read_csv(SUPPORT)

bundle = joblib.load(MODEL)

features = (
    list(bundle["numeric_predictors"])
    + list(bundle["categorical_predictors"])
)

if len(features) != 15 or len(set(features)) != 15:
    raise RuntimeError(
        "Expected 15 distinct original predictors"
    )

def canonical(series):
    x = pd.to_numeric(series, errors="coerce")

    if x.notna().all() and np.allclose(
        x.to_numpy(float),
        np.round(x.to_numpy(float))
    ):
        return pd.Series(
            np.round(
                x.to_numpy(float)
            ).astype(np.int64).astype(str),
            index=series.index
        )

    return series.astype(str).str.strip()

for frame, column in [
    (master, "_ROW_ID"),
    (oof, "_ROW_KEY"),
    (outer, "_ROW_ID")
]:
    frame["__KEY"] = canonical(frame[column])

    if frame["__KEY"].duplicated().any():
        raise RuntimeError(
            f"Duplicate identifiers: {column}"
        )

if (
    len(master) != 5063
    or not (
        set(master["__KEY"])
        == set(oof["__KEY"])
        == set(outer["__KEY"])
    )
):
    raise RuntimeError(
        "Master, OOF and outer-fold IDs disagree"
    )

master["__FOLD"] = master["__KEY"].map(
    outer.set_index("__KEY")["OUTER_FOLD"]
)

if master["__FOLD"].isna().any():
    raise RuntimeError(
        "Missing outer-fold assignment"
    )

master["__FOLD"] = master["__FOLD"].astype(int)

if set(master["__FOLD"]) != set(range(5)):
    raise RuntimeError(
        "Expected five outer folds numbered 0-4"
    )

oof = (
    oof.set_index("__KEY")
    .loc[master["__KEY"]]
    .reset_index()
)

y = master["ANC_CLASS"].to_numpy(int)
w = master["SURVEY_WEIGHT"].to_numpy(float)
fold_ids = master["__FOLD"].to_numpy(int)

if not np.array_equal(
    y,
    oof["ANC_CLASS_TRUE"].to_numpy(int)
):
    raise RuntimeError(
        "Frozen OOF labels are misaligned"
    )

if not np.array_equal(
    fold_ids,
    oof["Outer_Fold"].to_numpy(int)
):
    raise RuntimeError(
        "Frozen OOF folds are misaligned"
    )

if not np.allclose(
    w,
    oof["SURVEY_WEIGHT"].to_numpy(float),
    rtol=0,
    atol=1e-12
):
    raise RuntimeError(
        "Survey weights are misaligned"
    )

if not np.isfinite(w).all() or (w <= 0).any():
    raise RuntimeError(
        "Invalid survey weights"
    )

q0 = oof["q0_CALIBRATED"].to_numpy(float)
q2 = oof["q2_CALIBRATED_CONDITIONAL"].to_numpy(float)

expected_probs = np.column_stack([
    q0,
    (1 - q0) * (1 - q2),
    (1 - q0) * q2
])

saved_probs = oof[
    ["P_NO_ANC", "P_ANC_1_7", "P_ANC_GE8"]
].to_numpy(float)

if np.max(
    np.abs(expected_probs - saved_probs)
) > TOL:
    raise RuntimeError(
        "Frozen three-class probability fusion mismatch"
    )

for column in [
    "PSU", "STRATUM", *DOMAINS
]:
    if (
        column not in master.columns
        or master[column].isna().any()
    ):
        raise RuntimeError(
            f"Missing survey design/subgroup variable: {column}"
        )

pair_ids = list(
    zip(
        master["STRATUM"].astype(str),
        master["PSU"].astype(str)
    )
)

clusters = sorted(set(pair_ids))

cluster_map = {
    pair: i
    for i, pair in enumerate(clusters)
}

cluster_codes = np.fromiter(
    (cluster_map[pair] for pair in pair_ids),
    dtype=int,
    count=len(master)
)

strata = sorted({
    pair[0]
    for pair in clusters
})

K = len(clusters)

if K != 672 or len(strata) != 16:
    raise RuntimeError(
        f"Expected 672 PSUs and 16 strata; "
        f"found {K} PSUs and {len(strata)} strata"
    )

rng = np.random.default_rng(SEED)

factors = np.empty(
    (B, K),
    dtype=np.float64
)

for stratum in strata:
    idx = np.array([
        i
        for i, pair in enumerate(clusters)
        if pair[0] == stratum
    ], dtype=int)

    nh = len(idx)

    if nh < 2:
        raise RuntimeError(
            f"Singleton analytic stratum: {stratum}"
        )

    multiplicities = rng.multinomial(
        nh - 1,
        np.full(nh, 1 / nh),
        size=B
    )

    factors[:, idx] = (
        multiplicities * nh / (nh - 1)
    )

    deviation = np.max(
        np.abs(
            factors[:, idx].sum(axis=1)
            - nh
        )
    )

    if deviation > 1e-9:
        raise RuntimeError(
            f"Rao-Wu rescaling failed: {stratum}"
        )

print(
    f"FROZEN DATA: PASS | N={len(master)} | "
    f"PSUs={K} | strata={len(strata)}"
)

print(
    f"RAO-WU FACTORS: PASS | B={B} | seed={SEED}"
)

key_to_position = pd.Series(
    np.arange(len(master)),
    index=master["__KEY"]
)

shap_sources = {
    "Stage1_NoANC": (
        stage.loc[stage["stage"] == 1].copy(),
        "SHAP__",
        "base_probability",
        "predicted_probability",
        q0,
        False,
        "fold"
    ),
    "Stage2_GE8_conditional": (
        stage.loc[stage["stage"] == 2].copy(),
        "SHAP__",
        "base_probability",
        "predicted_probability",
        q2,
        True,
        "fold"
    ),
    "Final_1to7": (
        final.copy(),
        "class1__",
        "class1_base",
        "class1_predicted",
        saved_probs[:, 1],
        False,
        "outer_fold"
    ),
    "Final_GE8": (
        final.copy(),
        "class2__",
        "class2_base",
        "class2_predicted",
        saved_probs[:, 2],
        False,
        "outer_fold"
    )
}

shap_rows = []
shap_reps = {}
point_share = {}
rank_reps = {}

for target, (
    source,
    prefix,
    base_col,
    prob_col,
    reference,
    conditional,
    fold_col
) in shap_sources.items():

    required_columns = [
        "row_key",
        "analysis_weight",
        "survey_weight",
        fold_col,
        base_col,
        prob_col
    ] + [
        prefix + feature
        for feature in features
    ]

    if not set(required_columns).issubset(
        source.columns
    ):
        raise RuntimeError(
            f"Missing SHAP columns: {target}"
        )

    source["__KEY"] = canonical(
        source["row_key"]
    )

    if (
        source["__KEY"].duplicated().any()
        or len(source) != 800
    ):
        raise RuntimeError(
            f"Unexpected explanation sample: {target}"
        )

    positions = source["__KEY"].map(
        key_to_position
    )

    if positions.isna().any():
        raise RuntimeError(
            f"Unmatched SHAP row IDs: {target}"
        )

    pos = positions.to_numpy(int)

    if conditional and (y[pos] == 0).any():
        raise RuntimeError(
            "Stage 2 explanation includes ANC nonusers"
        )

    if not np.array_equal(
        source[fold_col].to_numpy(int),
        fold_ids[pos]
    ):
        raise RuntimeError(
            f"SHAP outer-fold mismatch: {target}"
        )

    if not np.allclose(
        source["survey_weight"].to_numpy(float),
        w[pos],
        rtol=0,
        atol=1e-12
    ):
        raise RuntimeError(
            f"SHAP survey-weight mismatch: {target}"
        )

    aw = source[
        "analysis_weight"
    ].to_numpy(float)

    if (
        not np.isfinite(aw).all()
        or (aw <= 0).any()
    ):
        raise RuntimeError(
            f"Invalid analysis weights: {target}"
        )

    for fold in range(5):
        selected = fold_ids[pos] == fold

        population = (
            (fold_ids == fold)
            & (
                (y != 0)
                if conditional
                else np.ones(len(y), dtype=bool)
            )
        )

        if not selected.any():
            raise RuntimeError(
                f"Empty explanation fold: {target}/{fold}"
            )

        expansion = (
            population.sum() / selected.sum()
        )

        if not np.allclose(
            aw[selected] / w[pos[selected]],
            expansion,
            rtol=1e-7,
            atol=1e-7
        ):
            raise RuntimeError(
                f"SHAP sampling expansion mismatch: {target}/{fold}"
            )

    values = source[
        [
            prefix + feature
            for feature in features
        ]
    ].to_numpy(float)

    predictions = source[
        prob_col
    ].to_numpy(float)

    base_values = source[
        base_col
    ].to_numpy(float)

    if (
        not np.isfinite(values).all()
        or not np.isfinite(base_values).all()
        or np.max(
            np.abs(
                predictions - reference[pos]
            )
        ) > TOL
    ):
        raise RuntimeError(
            f"Frozen SHAP prediction mismatch: {target}"
        )

    additivity_error = np.max(
        np.abs(
            base_values
            + values.sum(axis=1)
            - predictions
        )
    )

    if additivity_error > 1e-7:
        raise RuntimeError(
            f"SHAP additivity failed: {target}"
        )

    cid = cluster_codes[pos]

    denominator_cluster = np.bincount(
        cid,
        weights=aw,
        minlength=K
    )

    abs_cluster = np.column_stack([
        np.bincount(
            cid,
            weights=aw * np.abs(values[:, j]),
            minlength=K
        )
        for j in range(len(features))
    ])

    signed_cluster = np.column_stack([
        np.bincount(
            cid,
            weights=aw * values[:, j],
            minlength=K
        )
        for j in range(len(features))
    ])

    denominator_rep = (
        factors @ denominator_cluster
    )

    if (denominator_rep <= 0).any():
        raise RuntimeError(
            f"Empty SHAP bootstrap replicate: {target}"
        )

    rep_abs = (
        factors @ abs_cluster
    ) / denominator_rep[:, None]

    rep_signed = (
        factors @ signed_cluster
    ) / denominator_rep[:, None]

    point_abs = (
        abs_cluster.sum(axis=0)
        / denominator_cluster.sum()
    )

    point_signed = (
        signed_cluster.sum(axis=0)
        / denominator_cluster.sum()
    )

    if (
        point_abs.sum() <= 0
        or (rep_abs.sum(axis=1) <= 0).any()
    ):
        raise RuntimeError(
            f"Invalid SHAP importance denominator: {target}"
        )

    point_share[target] = (
        point_abs / point_abs.sum()
    )

    rep_share = (
        rep_abs
        / rep_abs.sum(axis=1, keepdims=True)
    )

    shap_reps[target] = rep_share

    order = np.argsort(
        -rep_abs,
        axis=1
    )

    ranks = np.empty_like(
        order
    )

    ranks[
        np.arange(B)[:, None],
        order
    ] = np.arange(
        1, len(features) + 1
    )[None, :]

    rank_reps[target] = ranks

    point_order = np.argsort(-point_abs)

    point_rank = np.empty(
        len(features),
        dtype=int
    )

    point_rank[point_order] = np.arange(
        1, len(features) + 1
    )

    for j, feature in enumerate(features):

        abs_ci = np.quantile(
            rep_abs[:, j],
            [ALPHA / 2, 1 - ALPHA / 2]
        )

        signed_ci = np.quantile(
            rep_signed[:, j],
            [ALPHA / 2, 1 - ALPHA / 2]
        )

        share_ci = np.quantile(
            100 * rep_share[:, j],
            [ALPHA / 2, 1 - ALPHA / 2]
        )

        shap_rows.append({
            "target": target,
            "feature": feature,
            "n_explained": len(source),
            "point_mean_abs_shap": point_abs[j],
            "mean_abs_ci_lower": abs_ci[0],
            "mean_abs_ci_upper": abs_ci[1],
            "point_mean_signed_shap": point_signed[j],
            "mean_signed_ci_lower": signed_ci[0],
            "mean_signed_ci_upper": signed_ci[1],
            "point_relative_importance_percent": (
                100 * point_share[target][j]
            ),
            "relative_importance_ci_lower_percent": share_ci[0],
            "relative_importance_ci_upper_percent": share_ci[1],
            "point_rank": point_rank[j],
            "top1_frequency": float(
                np.mean(ranks[:, j] == 1)
            ),
            "top5_frequency": float(
                np.mean(ranks[:, j] <= 5)
            ),
            "rank_95pct_lower": float(
                np.quantile(ranks[:, j], ALPHA / 2)
            ),
            "rank_95pct_upper": float(
                np.quantile(ranks[:, j], 1 - ALPHA / 2)
            )
        })

    print(
        f"SHAP bootstrap: {target} | "
        f"rows={len(source)} | "
        f"top1={features[point_order[0]]}"
    )

shap_summary = pd.DataFrame(
    shap_rows
)

contrast_rows = []

for j, feature in enumerate(features):

    difference = 100 * (
        shap_reps["Stage1_NoANC"][:, j]
        - shap_reps["Stage2_GE8_conditional"][:, j]
    )

    lower, upper = np.quantile(
        difference,
        [ALPHA / 2, 1 - ALPHA / 2]
    )

    contrast_rows.append({
        "feature": feature,
        "stage1_minus_stage2_importance_share_pp": (
            100 * (
                point_share["Stage1_NoANC"][j]
                - point_share["Stage2_GE8_conditional"][j]
            )
        ),
        "conditional_psu_ci_lower_pp": lower,
        "conditional_psu_ci_upper_pp": upper
    })

contrast_summary = pd.DataFrame(
    contrast_rows
)

rank_stability = []

rank_pairs = [
    (
        "Stage1_NoANC",
        "Stage2_GE8_conditional"
    ),
    (
        "Stage1_NoANC",
        "Final_GE8"
    ),
    (
        "Stage2_GE8_conditional",
        "Final_GE8"
    )
]

for first, second in rank_pairs:

    squared_differences = np.square(
        rank_reps[first]
        - rank_reps[second]
    ).sum(axis=1)

    rho = 1 - (
        6 * squared_differences
        / (
            len(features)
            * (len(features) ** 2 - 1)
        )
    )

    overlap = (
        (rank_reps[first] <= 5)
        & (rank_reps[second] <= 5)
    ).sum(axis=1)

    rank_stability.append({
        "comparison": f"{first} vs {second}",
        "rho_median": float(
            np.median(rho)
        ),
        "rho_ci_lower": float(
            np.quantile(rho, ALPHA / 2)
        ),
        "rho_ci_upper": float(
            np.quantile(rho, 1 - ALPHA / 2)
        ),
        "top5_overlap_median": float(
            np.median(overlap)
        ),
        "top5_overlap_range_min": int(
            overlap.min()
        ),
        "top5_overlap_range_max": int(
            overlap.max()
        )
    })

probability_targets = {
    "Stage1_NoANC": (
        q0, 0, False
    ),
    "Stage2_GE8_conditional": (
        q2, 2, True
    ),
    "Final_1to7": (
        saved_probs[:, 1], 1, False
    ),
    "Final_GE8": (
        saved_probs[:, 2], 2, False
    )
}

group_columns = []
aggregated = []
point_data = {}

for target, (
    probability,
    positive_class,
    conditional
) in probability_targets.items():

    eligible = (
        y != 0
        if conditional
        else np.ones(len(y), dtype=bool)
    )

    for domain in DOMAINS:

        categories = master[
            domain
        ].astype(str).to_numpy()

        for level in sorted(
            np.unique(categories[eligible])
        ):

            mask = (
                eligible
                & (categories == level)
            )

            selected = np.flatnonzero(
                mask
            )

            if not len(selected):
                continue

            cid = cluster_codes[selected]
            sw = w[selected]

            outcome = (
                y[selected] == positive_class
            ).astype(float)

            predicted = probability[selected]

            denominator = np.bincount(
                cid,
                weights=sw,
                minlength=K
            )

            observed = np.bincount(
                cid,
                weights=sw * outcome,
                minlength=K
            )

            predicted_sum = np.bincount(
                cid,
                weights=sw * predicted,
                minlength=K
            )

            group_columns.extend([
                (target, domain, level, "denominator"),
                (target, domain, level, "observed"),
                (target, domain, level, "predicted")
            ])

            aggregated.extend([
                denominator,
                observed,
                predicted_sum
            ])

            point_data[
                (target, domain, level)
            ] = {
                "n": len(selected),
                "events": int(outcome.sum()),
                "psus": len(np.unique(cid)),
                "observed": (
                    observed.sum()
                    / denominator.sum()
                ),
                "predicted": (
                    predicted_sum.sum()
                    / denominator.sum()
                )
            }

if not aggregated:
    raise RuntimeError(
        "No subgroup bootstrap data"
    )

matrix = np.column_stack(
    aggregated
)

rep_sums = factors @ matrix

if not np.isfinite(rep_sums).all():
    raise RuntimeError(
        "Nonfinite subgroup bootstrap estimates"
    )

subgroup_rows = []
rep_group = {}

for j in range(
    0, len(group_columns), 3
):

    target, domain, level, _ = (
        group_columns[j]
    )

    denominator = rep_sums[:, j]
    observed_total = rep_sums[:, j + 1]
    predicted_total = rep_sums[:, j + 2]

    observed = np.divide(
        observed_total,
        denominator,
        out=np.full(B, np.nan),
        where=denominator > 0
    )

    predicted = np.divide(
        predicted_total,
        denominator,
        out=np.full(B, np.nan),
        where=denominator > 0
    )

    gap_pp = 100 * (
        observed - predicted
    )

    oe = np.divide(
        observed_total,
        predicted_total,
        out=np.full(B, np.nan),
        where=predicted_total > 0
    )

    group_key = (
        target, domain, level
    )

    rep_group[group_key] = {
        "observed": observed,
        "predicted": predicted,
        "gap_pp": gap_pp,
        "OE": oe
    }

    point = point_data[group_key]

    point_observed = point["observed"]
    point_predicted = point["predicted"]

    row = {
        "target": target,
        "domain": domain,
        "level": level,
        "n_full": point["n"],
        "full_event_n": point["events"],
        "full_psus": point["psus"],
        "point_observed_rate": point_observed,
        "point_predicted_rate": point_predicted,
        "point_gap_pp": 100 * (
            point_observed - point_predicted
        ),
        "point_OE": (
            point_observed / point_predicted
            if point_predicted > 0
            else np.nan
        ),
        "limited_events": (
            point["events"] < 5
        ),
        "ci_method": (
            "Rao-Wu rescaled PSU bootstrap; "
            "fixed OOF predictions"
        )
    }

    metric_values = {
        "observed": observed,
        "predicted": predicted,
        "gap_pp": gap_pp,
        "OE": oe
    }

    for metric, values in metric_values.items():

        valid = values[
            np.isfinite(values)
        ]

        enough = (
            len(valid) >= int(0.95 * B)
        )

        row[
            f"{metric}_valid_replicates"
        ] = len(valid)

        row[
            f"{metric}_limited_replicates"
        ] = not enough

        row[
            f"{metric}_ci_lower"
        ] = (
            float(
                np.quantile(
                    valid,
                    ALPHA / 2
                )
            )
            if enough
            else np.nan
        )

        row[
            f"{metric}_ci_upper"
        ] = (
            float(
                np.quantile(
                    valid,
                    1 - ALPHA / 2
                )
            )
            if enough
            else np.nan
        )

    subgroup_rows.append(row)

subgroup_ci = pd.DataFrame(
    subgroup_rows
)

for _, previous in prior_subgroups.iterrows():

    key = (
        previous["target"],
        previous["domain"],
        str(previous["level"])
    )

    current = point_data.get(key)

    if current is None:
        raise RuntimeError(
            f"Previously reported subgroup missing: {key}"
        )

    if (
        int(previous["n_full"])
        != current["n"]
        or int(previous["full_event_n"])
        != current["events"]
        or abs(
            float(
                previous[
                    "full_weighted_observed_rate"
                ]
            )
            - current["observed"]
        ) > TOL
        or abs(
            float(
                previous[
                    "full_weighted_predicted_rate"
                ]
            )
            - current["predicted"]
        ) > TOL
    ):
        raise RuntimeError(
            f"Subgroup result mismatch: {key}"
        )

planned_contrasts = [
    (
        "Stage1_NoANC",
        "wealth_quintile",
        "Poorest",
        "Richest",
        "observed"
    ),
    (
        "Stage2_GE8_conditional",
        "wealth_quintile",
        "Richest",
        "Poorest",
        "observed"
    ),
    (
        "Final_GE8",
        "wealth_quintile",
        "Richest",
        "Poorest",
        "observed"
    ),
    (
        "Stage2_GE8_conditional",
        "division",
        "Dhaka",
        "Rajshahi",
        "gap_pp"
    ),
    (
        "Final_GE8",
        "division",
        "Dhaka",
        "Rajshahi",
        "gap_pp"
    )
]

paired_contrasts = []

for (
    target,
    domain,
    group_a,
    group_b,
    metric
) in planned_contrasts:

    key_a = (
        target, domain, group_a
    )

    key_b = (
        target, domain, group_b
    )

    if (
        key_a not in rep_group
        or key_b not in rep_group
    ):
        raise RuntimeError(
            f"Prespecified contrast missing: "
            f"{target}/{domain}"
        )

    difference = (
        rep_group[key_a][metric]
        - rep_group[key_b][metric]
    )

    def point_value(key):

        p = point_data[key]

        if metric == "observed":
            return 100 * p["observed"]

        return 100 * (
            p["observed"]
            - p["predicted"]
        )

    scale = (
        100
        if metric == "observed"
        else 1
    )

    finite = (
        difference[
            np.isfinite(difference)
        ] * scale
    )

    if len(finite) >= int(0.95 * B):

        low, high = np.quantile(
            finite,
            [ALPHA / 2, 1 - ALPHA / 2]
        )

    else:
        low, high = np.nan, np.nan

    paired_contrasts.append({
        "target": target,
        "domain": domain,
        "group_A": group_a,
        "group_B": group_b,
        "difference_type": (
            "observed_rate_pp"
            if metric == "observed"
            else "calibration_gap_pp"
        ),
        "point_A_minus_B_pp": (
            point_value(key_a)
            - point_value(key_b)
        ),
        "ci_lower_pp": float(low),
        "ci_upper_pp": float(high)
    })

spec = {
    "method": (
        "Rao-Wu rescaled stratified PSU bootstrap; "
        "original frozen OOF; conditional SHAP sample"
    ),
    "replicates": B,
    "seed": SEED,
    "number_of_psus": K,
    "number_of_strata": len(strata),
    "ci_type": "2.5/97.5 percentile",
    "sha256": {
        str(path.relative_to(ROOT)): sha256(path)
        for path in [
            MASTER,
            OOF,
            OUTER,
            STAGE,
            FINAL,
            SUBGROUP,
            SUPPORT,
            MANIFEST
        ]
    }
}

OUT.mkdir(
    parents=True,
    exist_ok=True
)

spec_path = (
    OUT / "bootstrap_run_spec.json"
)

if spec_path.is_file():

    existing = json.loads(
        spec_path.read_text(
            encoding="utf-8"
        )
    )

    if existing != spec:
        raise RuntimeError(
            "Existing bootstrap outputs have "
            "different inputs or settings"
        )

spec_path.write_text(
    json.dumps(
        spec,
        indent=2
    ),
    encoding="utf-8"
)

shap_summary.to_csv(
    OUT / "shap_psu_bootstrap_global_ci.csv",
    index=False
)

contrast_summary.to_csv(
    OUT / "stage1_stage2_importance_share_contrasts.csv",
    index=False
)

pd.DataFrame(
    rank_stability
).to_csv(
    OUT / "psu_bootstrap_stage_rank_agreement.csv",
    index=False
)

subgroup_ci.to_csv(
    OUT / "subgroup_frozen_oof_psu_bootstrap_ci.csv",
    index=False
)

pd.DataFrame(
    paired_contrasts
).to_csv(
    OUT / "prespecified_subgroup_paired_contrasts.csv",
    index=False
)

for target in shap_sources:

    figure_data = (
        shap_summary.loc[
            shap_summary["target"] == target
        ]
        .nsmallest(
            15,
            "point_rank"
        )
        .iloc[::-1]
    )

    fig, ax = plt.subplots(
        figsize=(9, 6.6)
    )

    positions = np.arange(
        len(figure_data)
    )

    ax.hlines(
        positions,
        figure_data["mean_abs_ci_lower"],
        figure_data["mean_abs_ci_upper"],
        linewidth=2
    )

    ax.plot(
        figure_data["point_mean_abs_shap"],
        positions,
        "o"
    )

    ax.set_yticks(
        positions,
        figure_data["feature"]
    )

    ax.set_xlabel(
        "Survey-weighted mean |SHAP| "
        "(calibrated probability scale)"
    )

    ax.set_title(
        f"{target}: conditional PSU-bootstrap intervals"
    )

    fig.tight_layout()

    fig.savefig(
        OUT / f"{target}_shap_bootstrap_ci.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

print(
    "\nXAI RAO-WU PSU BOOTSTRAP: COMPLETE"
)

print(
    "Global feature CI records:",
    len(shap_summary)
)

print(
    "Full-OOF subgroup CI records:",
    len(subgroup_ci)
)

print(
    "\nTop five SHAP features per target "
    "and top-five bootstrap frequency:"
)

for target in shap_sources:

    print(f"\n{target}")

    subset = (
        shap_summary.loc[
            shap_summary["target"] == target
        ]
        .nsmallest(
            5,
            "point_rank"
        )
    )

    print(
        subset[
            [
                "feature",
                "point_mean_abs_shap",
                "mean_abs_ci_lower",
                "mean_abs_ci_upper",
                "top5_frequency"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.5f}"
        )
    )

print(
    "\nPaired subgroup contrasts "
    "(percentage points):"
)

print(
    pd.DataFrame(
        paired_contrasts
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print(
    "\nStage ranking robustness:"
)

print(
    pd.DataFrame(
        rank_stability
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print(
    "\nLimited-support SHAP subgroups "
    "from Step 8:"
)

flagged = support_table.loc[
    support_table["support_flag"]
    != "INTERPRETABLE_DESCRIPTIVELY",
    [
        "target",
        "domain",
        "level",
        "n_explained",
        "support_flag"
    ]
]

print(
    flagged.to_string(
        index=False
    )
)

print("\nSaved:", OUT)

FROZEN DATA: PASS | N=5063 | PSUs=672 | strata=16
RAO-WU FACTORS: PASS | B=2000 | seed=2026
SHAP bootstrap: Stage1_NoANC | rows=800 | top1=wealth_quintile
SHAP bootstrap: Stage2_GE8_conditional | rows=800 | top1=wealth_quintile
SHAP bootstrap: Final_1to7 | rows=800 | top1=wealth_quintile
SHAP bootstrap: Final_GE8 | rows=800 | top1=wealth_quintile

XAI RAO-WU PSU BOOTSTRAP: COMPLETE
Global feature CI records: 60
Full-OOF subgroup CI records: 84

Top five SHAP features per target and top-five bootstrap frequency:

Stage1_NoANC
                        feature  point_mean_abs_shap  mean_abs_ci_lower  mean_abs_ci_upper  top5_frequency
                wealth_quintile              0.02738            0.02589            0.02890         1.00000
              prior_live_births              0.01963            0.01787            0.02161         1.00000
          weekly_media_exposure              0.01610            0.01559            0.01667         1.00000
             maternal_education          

In [21]:

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

ROOT = Path(r"E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026")

if not ROOT.is_dir():
    ROOT = Path(r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026")

if not ROOT.is_dir():
    raise FileNotFoundError("The verified ANC project is unavailable")

XAI = ROOT / "10_explainability"
OUT = XAI / "XAI_MANUSCRIPT_AUDIT"

paths = {
    "manifest": XAI / "XAI_SELECTED_REFITS/reconstruction_manifest.json",
    "reconstruction": XAI / "XAI_SELECTED_REFITS/reconstruction_qc.csv",
    "stage": XAI / "XAI_PRIMARY_SHAP/stage_shap_importance.csv",
    "multiclass": XAI / "XAI_FINAL_MULTICLASS_SHAP/multiclass_feature_importance.csv",
    "permutation": XAI / "XAI_PRIMARY_PERMUTATION/primary_permutation_summary.csv",
    "direction": XAI / "XAI_DIRECTIONAL_EFFECTS/directionality_feature_summary.csv",
    "sensitivity": XAI / "XAI_SHAP_SENSITIVITY/sensitivity_rank_agreement.csv",
    "interactions": XAI / "XAI_PAIRWISE_INTERACTIONS/pooled_pair_nonadditivity.csv",
    "subgroup_risk": XAI / "XAI_SUBGROUP_HETEROGENEITY/subgroup_full_oof_risk_calibration.csv",
    "subgroup_support": XAI / "XAI_SUBGROUP_HETEROGENEITY/subgroup_shap_ranking_and_support.csv",
    "bootstrap_spec": XAI / "XAI_PSU_BOOTSTRAP_UNCERTAINTY/bootstrap_run_spec.json",
    "importance_ci": XAI / "XAI_PSU_BOOTSTRAP_UNCERTAINTY/shap_psu_bootstrap_global_ci.csv",
    "relative_contrasts": XAI / "XAI_PSU_BOOTSTRAP_UNCERTAINTY/stage1_stage2_importance_share_contrasts.csv",
    "rank_ci": XAI / "XAI_PSU_BOOTSTRAP_UNCERTAINTY/psu_bootstrap_stage_rank_agreement.csv",
    "subgroup_ci": XAI / "XAI_PSU_BOOTSTRAP_UNCERTAINTY/subgroup_frozen_oof_psu_bootstrap_ci.csv",
    "paired_contrasts": XAI / "XAI_PSU_BOOTSTRAP_UNCERTAINTY/prespecified_subgroup_paired_contrasts.csv",
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing completed XAI output ({name}): {path}"
        )

def file_hash(path):
    h = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1 << 20),
            b""
        ):
            h.update(block)

    return h.hexdigest()

manifest = json.loads(
    paths["manifest"].read_text(encoding="utf-8")
)

spec = json.loads(
    paths["bootstrap_spec"].read_text(encoding="utf-8")
)

if (
    manifest.get("status") != "PASS"
    or int(manifest.get("number_of_models", 0)) != 10
):
    raise RuntimeError("Model reconstruction QC has not passed")

if (
    int(spec.get("replicates", -1)),
    int(spec.get("number_of_psus", -1)),
    int(spec.get("number_of_strata", -1))
) != (2000, 672, 16):
    raise RuntimeError("PSU-bootstrap design differs")

D = {
    key: pd.read_csv(path)
    for key, path in paths.items()
    if path.suffix.lower() == ".csv"
}

boot = D["importance_ci"]
stage = D["stage"]
multi = D["multiclass"]

perm = D["permutation"]
sens = D["sensitivity"]
interactions = D["interactions"]

risk = D["subgroup_risk"]
subci = D["subgroup_ci"]
support = D["subgroup_support"]

contrast = D["relative_contrasts"]
ranks = D["rank_ci"]
paired = D["paired_contrasts"]
direction = D["direction"]

targets = [
    "Stage1_NoANC",
    "Stage2_GE8_conditional",
    "Final_1to7",
    "Final_GE8"
]

features = stage.loc[
    stage["stage"] == 1,
    "feature"
].tolist()

def require(frame, cols, name):
    missing = set(cols) - set(frame.columns)

    if missing:
        raise RuntimeError(
            f"Missing columns in {name}: {sorted(missing)}"
        )

def check_unique(frame, by, expected, name):
    if (
        len(frame) != expected
        or frame.duplicated(by).any()
    ):
        raise RuntimeError(
            f"Unexpected size or duplicate keys in {name}: {len(frame)}"
        )

require(
    boot,
    [
        "target", "feature", "point_mean_abs_shap",
        "mean_abs_ci_lower", "mean_abs_ci_upper",
        "point_rank", "top5_frequency"
    ],
    "bootstrap importance"
)

require(
    subci,
    [
        "target", "domain", "level",
        "point_observed_rate",
        "point_predicted_rate",
        "observed_ci_lower",
        "observed_ci_upper",
        "gap_pp_ci_lower",
        "gap_pp_ci_upper",
        "point_gap_pp",
        "full_event_n"
    ],
    "subgroup confidence intervals"
)

require(
    support,
    ["target", "domain", "level", "support_flag"],
    "subgroup SHAP support"
)

require(
    interactions,
    [
        "target", "feature_a", "feature_b",
        "interaction_rms_pp",
        "logit_interaction_rms",
        "limited_fold_count"
    ],
    "pairwise interactions"
)

require(
    sens,
    [
        "stage", "scenario",
        "rank_spearman_vs_reference32",
        "top5_overlap",
        "relative_total_importance_change"
    ],
    "SHAP sensitivity"
)

require(
    contrast,
    [
        "feature",
        "stage1_minus_stage2_importance_share_pp",
        "conditional_psu_ci_lower_pp",
        "conditional_psu_ci_upper_pp"
    ],
    "stage contrast"
)

require(
    paired,
    [
        "target", "domain", "group_A",
        "group_B", "point_A_minus_B_pp",
        "ci_lower_pp", "ci_upper_pp"
    ],
    "paired contrasts"
)

require(
    ranks,
    [
        "comparison", "rho_median",
        "rho_ci_lower", "rho_ci_upper"
    ],
    "stage rank intervals"
)

require(
    stage,
    ["stage", "feature", "weighted_mean_abs_shap"],
    "stage SHAP"
)

require(
    multi,
    ["class_id", "feature", "mean_abs_shap"],
    "multiclass SHAP"
)

require(
    perm,
    [
        "feature",
        "delta_stage1_no_anc_ap_mean",
        "delta_stage2_ge8_ap_mean"
    ],
    "permutation"
)

require(
    direction,
    ["target", "feature", "weighted_mean_abs_shap"],
    "directional SHAP"
)

if len(features) != 15 or len(set(features)) != 15:
    raise RuntimeError("Expected 15 original predictors")

if set(boot["target"]) != set(targets):
    raise RuntimeError("Missing explanation targets")

check_unique(boot, ["target", "feature"], 60, "importance intervals")
check_unique(stage, ["stage", "feature"], 30, "stage SHAP")
check_unique(multi, ["class_id", "feature"], 45, "multiclass SHAP")
check_unique(perm, ["feature"], 15, "permutation")
check_unique(direction, ["target", "feature"], 75, "directionality")
check_unique(sens, ["stage", "scenario"], 10, "sensitivity")

check_unique(
    interactions,
    ["target", "feature_a", "feature_b"],
    48,
    "pairwise interactions"
)

check_unique(
    subci,
    ["target", "domain", "level"],
    84,
    "subgroup uncertainty"
)

check_unique(
    support,
    ["target", "domain", "level"],
    84,
    "subgroup support"
)

check_unique(
    risk,
    ["target", "domain", "level"],
    84,
    "subgroup OOF risks"
)

check_unique(
    contrast,
    ["feature"],
    15,
    "importance-share contrasts"
)

check_unique(
    paired,
    ["target", "domain", "group_A", "group_B"],
    5,
    "paired contrasts"
)

check_unique(
    D["reconstruction"],
    ["Outer_Fold", "Stage"],
    10,
    "reconstruction"
)

if max(
    D["reconstruction"]["Raw_MaxAbsError"].max(),
    D["reconstruction"]["Calibrated_MaxAbsError"].max()
) > 1e-10:
    raise RuntimeError("Reconstruction tolerance exceeded")

if (
    not set(boot["feature"]).issubset(set(features))
    or set(perm["feature"]) != set(features)
):
    raise RuntimeError("Feature specification differs")

if not np.isfinite(
    boot["point_mean_abs_shap"].to_numpy(float)
).all():
    raise RuntimeError("Nonfinite feature importance")

if not boot["top5_frequency"].between(0, 1).all():
    raise RuntimeError("Invalid bootstrap selection frequencies")

if (
    boot["mean_abs_ci_upper"]
    < boot["mean_abs_ci_lower"]
).any():
    raise RuntimeError("Reversed confidence intervals")

for target, stage_id, class_id in [
    ("Stage1_NoANC", 1, 0),
    ("Stage2_GE8_conditional", 2, None),
    ("Final_1to7", None, 1),
    ("Final_GE8", None, 2)
]:

    reference = boot.loc[
        boot["target"] == target
    ].set_index("feature")

    if stage_id is not None:
        other = stage.loc[
            stage["stage"] == stage_id
        ].set_index("feature")["weighted_mean_abs_shap"]

    else:
        other = multi.loc[
            multi["class_id"] == class_id
        ].set_index("feature")["mean_abs_shap"]

    if set(other.index) != set(features):
        raise RuntimeError(f"Feature-set mismatch: {target}")

    error = np.max(
        np.abs(
            reference.loc[
                features, "point_mean_abs_shap"
            ].to_numpy(float)
            - other.loc[features].to_numpy(float)
        )
    )

    if error > 1e-10:
        raise RuntimeError(
            f"SHAP summary mismatch: {target}, error={error}"
        )

stage1 = stage.loc[
    stage["stage"] == 1
].set_index("feature")["weighted_mean_abs_shap"]

class0 = multi.loc[
    multi["class_id"] == 0
].set_index("feature")["mean_abs_shap"]

if np.max(
    np.abs(
        stage1.loc[features].to_numpy(float)
        - class0.loc[features].to_numpy(float)
    )
) > 1e-10:
    raise RuntimeError(
        "Stage 1 and final class-0 importance disagree"
    )

joined = subci.merge(
    risk,
    on=["target", "domain", "level"],
    validate="one_to_one"
)

if len(joined) != 84:
    raise RuntimeError("Incomplete subgroup estimate matching")

for left, right in [
    (
        "point_observed_rate",
        "full_weighted_observed_rate"
    ),
    (
        "point_predicted_rate",
        "full_weighted_predicted_rate"
    )
]:
    if np.max(
        np.abs(
            joined[left].to_numpy(float)
            - joined[right].to_numpy(float)
        )
    ) > 1e-10:
        raise RuntimeError(
            f"OOF and PSU-bootstrap mismatch: {left}"
        )

OUT.mkdir(parents=True, exist_ok=True)

TABLES = OUT / "tables"
FIGURES = OUT / "figures"

TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

names = {
    "Stage1_NoANC": "Stage 1: no ANC",
    "Stage2_GE8_conditional": "Stage 2: >=8 visits | ANC users",
    "Final_1to7": "Final: 1-7 visits",
    "Final_GE8": "Final: >=8 visits"
}

all_shap = boot.copy()

all_shap["prediction_target"] = (
    all_shap["target"].map(names)
)

all_shap.sort_values(
    ["target", "point_rank"]
).to_csv(
    TABLES / "Table_XAI1_all_stage_and_class_SHAP_with_CI.csv",
    index=False
)

contrast.sort_values(
    "stage1_minus_stage2_importance_share_pp",
    ascending=False
).to_csv(
    TABLES / "Table_XAI2_stage_specific_importance_share_differences.csv",
    index=False
)

paired.to_csv(
    TABLES / "Table_XAI3_prespecified_PSU_contrasts.csv",
    index=False
)

subci.to_csv(
    TABLES / "Supplement_XAI_subgroup_OOF_calibration_CI.csv",
    index=False
)

subci.loc[
    subci["domain"].isin([
        "division",
        "wealth_quintile"
    ])
].to_csv(
    TABLES / "Table_XAI4_geographic_and_wealth_OOF_calibration.csv",
    index=False
)

interactions.sort_values(
    ["target", "interaction_rms_pp"],
    ascending=[True, False]
).to_csv(
    TABLES / "Supplement_XAI_pairwise_response_nonadditivity.csv",
    index=False
)

sens.to_csv(
    TABLES / "Supplement_XAI_SHAP_sensitivity.csv",
    index=False
)

support.to_csv(
    TABLES / "Supplement_XAI_subgroup_SHAP_support.csv",
    index=False
)

perm.to_csv(
    TABLES / "Supplement_XAI_grouped_permutation_importance.csv",
    index=False
)

ranks.to_csv(
    TABLES / "Supplement_XAI_cross_stage_rank_uncertainty.csv",
    index=False
)

try:
    font_manager.findfont(
        "Times New Roman",
        fallback_to_default=False
    )
    selected_font = "Times New Roman"

except ValueError:
    selected_font = "DejaVu Serif"

plt.rcParams.update({
    "font.family": selected_font,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})

def save_figure(fig, name):

    fig.savefig(
        FIGURES / (name + ".png"),
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        FIGURES / (name + ".tiff"),
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        FIGURES / (name + ".svg"),
        bbox_inches="tight"
    )

    plt.close(fig)

def forest(ax, data, title, n=10):

    sub = data.nsmallest(
        n, "point_rank"
    ).iloc[::-1]

    point = sub[
        "point_mean_abs_shap"
    ].to_numpy(float)

    lower = sub[
        "mean_abs_ci_lower"
    ].to_numpy(float)

    upper = sub[
        "mean_abs_ci_upper"
    ].to_numpy(float)

    yy = np.arange(len(sub))

    ax.hlines(
        yy,
        lower,
        upper,
        color="0.4",
        linewidth=1.7
    )

    ax.scatter(
        point,
        yy,
        color="black",
        s=25,
        zorder=3
    )

    ax.set_yticks(
        yy,
        [
            feature.replace("_", " ")
            for feature in sub["feature"]
        ]
    )

    ax.set_title(
        title,
        loc="left"
    )

    ax.set_xlabel(
        "Mean absolute SHAP (probability units)"
    )

    ax.grid(
        axis="x",
        alpha=0.18
    )

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6.6),
    layout="constrained"
)

for ax, target in zip(
    axes, targets[:2]
):

    forest(
        ax,
        boot.loc[
            boot["target"] == target
        ],
        names[target],
        n=10
    )

save_figure(
    fig,
    "Figure_XAI1_two_stage_SHAP_PSU_CI"
)

fig, axes = plt.subplots(
    1, 3,
    figsize=(19, 6.8),
    layout="constrained"
)

for ax, target in zip(
    axes,
    [
        "Stage1_NoANC",
        "Final_1to7",
        "Final_GE8"
    ]
):

    forest(
        ax,
        boot.loc[
            boot["target"] == target
        ],
        names[target],
        n=10
    )

save_figure(
    fig,
    "Figure_XAI2_three_class_SHAP_PSU_CI"
)

wealth_order = [
    "Poorest",
    "Poorer",
    "Middle",
    "Richer",
    "Richest"
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4.9),
    layout="constrained"
)

for ax, target in zip(
    axes,
    [
        "Stage1_NoANC",
        "Final_GE8"
    ]
):

    sub = subci.loc[
        (subci["target"] == target)
        & (
            subci["domain"]
            == "wealth_quintile"
        )
    ].copy()

    sub["level"] = sub["level"].astype(str)

    sub = sub.set_index("level")

    if set(wealth_order).issubset(set(sub.index)):
        labels = wealth_order

    elif len(sub) == 5:
        labels = sorted(
            sub.index.astype(str).tolist()
        )

    else:
        raise RuntimeError(
            "Expected five wealth quintile groups"
        )

    sub = sub.loc[labels]

    xx = np.arange(5)

    point = 100 * sub[
        "point_observed_rate"
    ].to_numpy(float)

    lower = 100 * sub[
        "observed_ci_lower"
    ].to_numpy(float)

    upper = 100 * sub[
        "observed_ci_upper"
    ].to_numpy(float)

    predicted = 100 * sub[
        "point_predicted_rate"
    ].to_numpy(float)

    ax.vlines(
        xx,
        lower,
        upper,
        color="0.45",
        linewidth=1.7
    )

    ax.plot(
        xx,
        point,
        "o-",
        color="black",
        label="Observed (+95% PSU CI)"
    )

    ax.plot(
        xx,
        predicted,
        "s--",
        color="0.55",
        label="Frozen OOF predicted"
    )

    ax.set_xticks(
        xx,
        labels,
        rotation=25
    )

    ax.set_ylabel(
        "Survey-weighted percentage"
    )

    ax.set_title(
        names[target],
        loc="left"
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

axes[0].legend(
    frameon=False,
    fontsize=8
)

save_figure(
    fig,
    "Figure_XAI3_wealth_gradient_observed_predicted"
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(14.8, 5.8),
    layout="constrained"
)

for ax, target in zip(
    axes,
    [
        "Stage2_GE8_conditional",
        "Final_GE8"
    ]
):

    sub = subci.loc[
        (subci["target"] == target)
        & (subci["domain"] == "division")
    ].sort_values("point_gap_pp")

    yy = np.arange(len(sub))

    ax.hlines(
        yy,
        sub["gap_pp_ci_lower"],
        sub["gap_pp_ci_upper"],
        color="0.45",
        linewidth=1.8
    )

    ax.scatter(
        sub["point_gap_pp"],
        yy,
        color="black",
        s=25,
        zorder=3
    )

    ax.axvline(
        0,
        color="0.55",
        linestyle="--",
        linewidth=1
    )

    ax.set_yticks(
        yy,
        sub["level"].tolist()
    )

    ax.set_xlabel(
        "Observed minus predicted (percentage points)"
    )

    ax.set_title(
        names[target],
        loc="left"
    )

    ax.grid(
        axis="x",
        alpha=0.18
    )

save_figure(
    fig,
    "Supplement_XAI_division_calibration_gap_PSU_CI"
)

sub = contrast.sort_values(
    "stage1_minus_stage2_importance_share_pp"
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    layout="constrained"
)

yy = np.arange(len(sub))

ax.hlines(
    yy,
    sub["conditional_psu_ci_lower_pp"],
    sub["conditional_psu_ci_upper_pp"],
    color="0.45",
    linewidth=1.7
)

ax.scatter(
    sub["stage1_minus_stage2_importance_share_pp"],
    yy,
    color="black",
    s=24
)

ax.axvline(
    0,
    color="0.55",
    linestyle="--",
    linewidth=1
)

ax.set_yticks(
    yy,
    [
        str(feature).replace("_", " ")
        for feature in sub["feature"]
    ]
)

ax.set_xlabel(
    "Stage 1 minus Stage 2 relative importance share (pp)"
)

ax.set_title(
    "Conditional PSU-bootstrap stage-importance contrasts",
    loc="left"
)

ax.grid(
    axis="x",
    alpha=0.2
)

save_figure(
    fig,
    "Supplement_XAI_stage_importance_share_contrasts"
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4.9),
    layout="constrained"
)

scenario_order = [
    "original_cached",
    "reference_32",
    "permutation_seed",
    "alternate_background_32",
    "alternate_background_64"
]

scenario_labels = [
    "Original",
    "Reference",
    "New seed",
    "New BG32",
    "New BG64"
]

for ax, stage_id in zip(
    axes, [1, 2]
):

    sub = sens.loc[
        sens["stage"] == stage_id
    ].set_index("scenario").loc[scenario_order]

    values = sub[
        "rank_spearman_vs_reference32"
    ]

    ax.plot(
        np.arange(5),
        values,
        "o-",
        color="black"
    )

    ax.set_xticks(
        np.arange(5),
        scenario_labels,
        rotation=28
    )

    ax.set_ylim(
        max(0.8, float(values.min()) - 0.02),
        1.015
    )

    ax.set_ylabel(
        "Global feature-rank Spearman correlation"
    )

    ax.set_title(
        f"Stage {stage_id}",
        loc="left"
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

save_figure(
    fig,
    "Supplement_XAI_SHAP_background_seed_sensitivity"
)

flags = {
    "limited_SHAP_subgroups": support.loc[
        support["support_flag"]
        != "INTERPRETABLE_DESCRIPTIVELY",
        [
            "target",
            "domain",
            "level",
            "n_explained",
            "support_flag"
        ]
    ].to_dict("records"),

    "geographic_contrasts_with_CI_crossing_zero": paired.loc[
        (paired["domain"] == "division")
        & (paired["ci_lower_pp"] <= 0)
        & (paired["ci_upper_pp"] >= 0),
        [
            "target",
            "group_A",
            "group_B",
            "point_A_minus_B_pp",
            "ci_lower_pp",
            "ci_upper_pp"
        ]
    ].to_dict("records"),

    "note": (
        "Fixed original trained models and sampled SHAP attributions; "
        "bootstrap intervals do not include model-development, "
        "explanation-sample or SHAP-background uncertainty."
    )
}

audit = {
    "status": "PASS",
    "n_selected_models": 10,
    "predictor_count": 15,
    "shap_targets": targets,
    "global_importance_records": len(boot),
    "subgroup_CI_records": len(subci),
    "interaction_records": len(interactions),
    "bootstrap_replicates": int(spec["replicates"]),
    "PSUs": int(spec["number_of_psus"]),
    "strata": int(spec["number_of_strata"]),
    "input_sha256": {
        key: file_hash(path)
        for key, path in paths.items()
    },
    "reporting_cautions": flags
}

(OUT / "xai_audit_manifest.json").write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

with (
    OUT / "manuscript_reporting_notes.md"
).open("w", encoding="utf-8") as file:

    file.write("# XAI reporting audit\n\n")

    file.write(
        "All cross-file numerical checks passed. "
        "Primary figures: Figure_XAI1, Figure_XAI2, Figure_XAI3. "
        "Supplemental figures: divisional calibration gaps, "
        "conditional stage-importance contrasts, and sensitivity.\n\n"
    )

    file.write(
        "Use feature reliance or model attribution rather than "
        "causal determinants. SHAP magnitudes are on the calibrated "
        "probability scale. Stage 2 conditional and final >=8 "
        "are distinct estimands.\n\n"
    )

    file.write(
        "PSU-bootstrap intervals for SHAP are conditional on "
        "fixed models, sampled explanations, and SHAP background. "
        "Top-five retention of 100% does not establish retraining "
        "or external-sample stability.\n\n"
    )

    file.write(
        "Do not infer reliable Dhaka-Rajshahi pooled-OOF "
        "calibration-gap differences when paired CIs include zero. "
        "Division-withheld IECV has a different estimand and "
        "must be presented separately.\n\n"
    )

    file.write(
        "Flag limited-coverage maternal-education/no-education "
        "SHAP subgroups; interaction surfaces are descriptive "
        "functional ANOVA and not causal interactions.\n"
    )

print("\nFINAL XAI REPORTING AUDIT: PASS")

print(
    f"Model refits: 10 | Features: {len(features)} "
    f"| Targets: {len(targets)}"
)

print(
    f"Global SHAP CI: {len(boot)} "
    f"| Subgroup CI: {len(subci)} "
    f"| Targeted interactions: {len(interactions)}"
)

print(
    "PSU bootstrap: 2000 replicates | "
    "672 PSUs | 16 strata"
)

print(
    "Limited-support SHAP subgroups:",
    len(flags["limited_SHAP_subgroups"])
)

print(
    "Geographic gap contrasts crossing zero:",
    len(
        flags[
            "geographic_contrasts_with_CI_crossing_zero"
        ]
    )
)

print("\nMain and supplemental figure files:")

for path in sorted(
    FIGURES.glob("*.tiff")
):
    print(" ", path.name)

print(
    "\nTables created:",
    len(list(TABLES.glob("*.csv")))
)

print("OUTPUT:", OUT)


FINAL XAI REPORTING AUDIT: PASS
Model refits: 10 | Features: 15 | Targets: 4
Global SHAP CI: 60 | Subgroup CI: 84 | Targeted interactions: 48
PSU bootstrap: 2000 replicates | 672 PSUs | 16 strata
Limited-support SHAP subgroups: 4
Geographic gap contrasts crossing zero: 2

Main and supplemental figure files:
  Figure_XAI1_two_stage_SHAP_PSU_CI.tiff
  Figure_XAI2_three_class_SHAP_PSU_CI.tiff
  Figure_XAI3_wealth_gradient_observed_predicted.tiff
  Supplement_XAI_division_calibration_gap_PSU_CI.tiff
  Supplement_XAI_SHAP_background_seed_sensitivity.tiff
  Supplement_XAI_stage_importance_share_contrasts.tiff

Tables created: 10
OUTPUT: E:\Drive E\BDHS_ANC_Q1_FINAL_REBUILD_2026\10_explainability\XAI_MANUSCRIPT_AUDIT
